# Redes de entidades em notícias sobre inteligência artificial
## Auditoria, preparação, caracterização e consolidação NER

Autor: Pedro Henrique de Carvalho Godoy.

O notebook preserva as entradas, prepara o corpus, caracteriza sua cobertura e consolida menções de entidades. A fila de inferência contém os documentos cuja extração ainda precisa ser realizada. As seções de consolidação não acessam APIs.


### 1. Configurações e dependências
Defina os arquivos de entrada, a data de referência, o ano de comparação e os marcos temporais. Os caminhos são relativos à pasta do notebook.

As tabelas consolidadas são atualizadas ao repetir a auditoria. Cada execução preserva uma cópia de seus resultados e um manifesto com hashes das entradas, parâmetros e versões das bibliotecas.


In [ ]:
from pathlib import Path
from datetime import datetime, date
from zoneinfo import ZoneInfo
from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode
from collections import Counter
import ast, hashlib, json, re, platform, sys, shutil
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

PROJETO = Path.cwd()
ARQUIVO_NOTICIAS = PROJETO / 'dados/originais/NewsAI_BR_20261114_4termos_sem_notas_editoriais_textos_recuperados.csv'
ARQUIVO_CATEGORIAS = PROJETO / 'dados/originais/category_groups.py'
DATA_REFERENCIA = date(2026, 11, 14)  # Data fixa desta auditoria; ajuste em uma nova rodada.
ANO_COMPARACAO = 2025
FONTES_ESPERADAS = ['Folha de S.Paulo', 'G1', 'CNN Brasil']
MARCOS = list(range(1994, 2006)) + [2007, 2010, 2013, 2016, 2019, 2022, 2025]
TABELAS = PROJETO / 'resultados/tabelas'
FIGURAS = PROJETO / 'resultados/figuras'
for pasta in [TABELAS, FIGURAS]:
    pasta.mkdir(parents=True, exist_ok=True)

def sha_arquivo(caminho):
    # Leitura em blocos evita criar uma cópia de todo o arquivo em memória.
    h = hashlib.sha256()
    with caminho.open('rb') as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b''):
            h.update(bloco)
    return h.hexdigest()

def sha_texto(texto):
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()

assert ARQUIVO_NOTICIAS.is_file(), f'Arquivo ausente: {ARQUIVO_NOTICIAS}'
HASH_ORIGINAL = sha_arquivo(ARQUIVO_NOTICIAS)
HASH_CATEGORIAS = sha_arquivo(ARQUIVO_CATEGORIAS)
EXECUCAO_ID = datetime.now(ZoneInfo('America/Sao_Paulo')).strftime('%Y%m%d_%H%M%S')
PASTA_EXECUCAO = PROJETO / 'resultados/execucao' / f'auditoria_{EXECUCAO_ID}_{HASH_ORIGINAL[:12]}'
PASTA_EXECUCAO.mkdir(parents=True, exist_ok=True)
print('Dataset:', ARQUIVO_NOTICIAS.name)
print('SHA-256:', HASH_ORIGINAL)
print('Somente auditoria. Nenhuma exclusão será aplicada.')

### 2. Leitura e esquema
Preservamos os campos como texto e mantemos as células vazias como strings vazias. Assim, a leitura não transforma automaticamente valores como `NA` ou `NaN` em ausências. Espaços e marcadores textuais de ausência são relatados separadamente.
`linha_csv` identifica a posição do registro nesta versão: é um localizador, não a identidade persistente de uma notícia. Em CSV com quebras de linha dentro de campos, não corresponde à linha física do arquivo.

In [ ]:
COLUNAS_ESPERADAS = ['title','description','link','image_src','full_text','source','author',
    'publication_date','publication_update_date','scraping_date','text_length','category_raw']
df = pd.read_csv(ARQUIVO_NOTICIAS, encoding='utf-8-sig', dtype=str,
                 keep_default_na=False, on_bad_lines='error')
colunas_originais = df.columns.tolist()
ausentes = sorted(set(COLUNAS_ESPERADAS) - set(df.columns))
extras = sorted(set(df.columns) - set(COLUNAS_ESPERADAS))
assert not ausentes, f'Colunas obrigatórias ausentes: {ausentes}'
assert len(df) > 0, 'Dataset vazio.'
assert df.columns.is_unique, 'Colunas duplicadas.'
originais = df.copy()
df.insert(0, 'linha_csv', np.arange(1, len(df)+1))

def salvar_tabela(tabela, nome):
    tabela.to_csv(TABELAS / nome, index=False, encoding='utf-8-sig')
    tabela.to_csv(PASTA_EXECUCAO / nome, index=False, encoding='utf-8-sig')

esquema = pd.DataFrame({'coluna': colunas_originais,
    'vazios_ou_espacos': [int(df[c].str.strip().eq('').sum()) for c in colunas_originais],
    'marcadores_textuais_ausencia': [int(df[c].str.strip().str.casefold().isin(['nan','none','null','n/a']).sum()) for c in colunas_originais],
    'valores_distintos': [int(df[c].nunique()) for c in colunas_originais]})
salvar_tabela(esquema, '01_esquema.csv')
print('Dimensão original:', originais.shape)
print('Colunas extras:', extras)
print(esquema.to_string(index=False))

### 3. Datas e cobertura temporal
Conversões inválidas ficam ausentes apenas nas colunas auxiliares; o campo original é preservado. Uma data de publicação sem horário tem precisão diária. Datas com fuso são convertidas para São Paulo antes da extração do ano e mês.

As tabelas de cobertura distinguem quantidades observadas de lacunas. Uma célula zero significa que este arquivo não contém uma notícia naquela combinação; não significa que o veículo não publicou sobre IA.

In [ ]:
def data_calendario_local(valor):
    valor = valor.strip()
    if not valor:
        return pd.NaT
    try:
        # A leitura é restrita a datas ISO, evitando adivinhar dia/mês em formatos ambíguos.
        if not re.match(r'^\d{4}-\d{2}-\d{2}(?:$|[T ])', valor):
            return pd.NaT
        dt = pd.Timestamp(valor)
        if dt.tzinfo is not None:
            dt = dt.tz_convert('America/Sao_Paulo').tz_localize(None)
        return dt
    except (ValueError, TypeError, OverflowError):
        return pd.NaT

for campo in ['publication_date','publication_update_date','scraping_date']:
    df[campo+'_local'] = pd.to_datetime(df[campo].map(data_calendario_local), errors='coerce')
df['ano'] = df['publication_date_local'].dt.year.astype('Int64')
df['mes'] = df['publication_date_local'].dt.month.astype('Int64')
df['fonte_auditoria'] = df['source'].str.strip()
df['precisao_publicacao'] = np.where(df['publication_date'].str.fullmatch(r'\d{4}-\d{2}-\d{2}'), 'dia',
    np.where(df['publication_date_local'].notna(), 'horario_informado', 'ausente_ou_invalida'))

cobertura = df.groupby(['fonte_auditoria','ano'], dropna=False).agg(
    noticias=('linha_csv','size'), primeira_data=('publication_date_local','min'),
    ultima_data=('publication_date_local','max'), meses_observados=('mes','nunique')).reset_index()
salvar_tabela(cobertura, '02_cobertura_fonte_ano.csv')
mensal = df.groupby(['fonte_auditoria','ano','mes'], dropna=False).size().reset_index(name='noticias')
salvar_tabela(mensal, '03_cobertura_fonte_ano_mes.csv')
por_fonte = df.groupby('fonte_auditoria', dropna=False).agg(noticias=('linha_csv','size'),
    primeira_data=('publication_date_local','min'), ultima_data=('publication_date_local','max')).reset_index()
salvar_tabela(por_fonte, '04_resumo_fontes.csv')
marcos = pd.MultiIndex.from_product([FONTES_ESPERADAS,MARCOS],names=['fonte_auditoria','ano']).to_frame(index=False)
marcos = marcos.merge(cobertura, on=['fonte_auditoria','ano'],how='left')
marcos['noticias'] = marcos['noticias'].fillna(0).astype(int)
marcos['situacao'] = np.where(marcos['noticias'].eq(0),'sem_registros_no_arquivo','com_registros')
salvar_tabela(marcos, '05_disponibilidade_marcos.csv')
print(por_fonte.to_string(index=False))
print('Notícias de 2025 por fonte:')
print(df.loc[df['ano'].eq(ANO_COMPARACAO),'fonte_auditoria'].value_counts().to_string())

### 4. Textos, links e inconsistências
Calculamos separadamente caracteres (incluindo espaços e quebras de linha) e palavras (separação por espaços). Comparamos `text_length` com ambas as definições: não presumimos que o campo represente caracteres. Diferença para uma definição não determina erro nem exclusão.

A normalização de link é conservadora e usada somente para levantar candidatos: remove fragmentos e parâmetros de rastreamento conhecidos, mas preserva os demais parâmetros. Esta chave serve à detecção de candidatos e não substitui um identificador documental persistente.

In [ ]:
def normalizar_link(valor):
    valor = valor.strip()
    try:
        u = urlsplit(valor)
        if u.scheme.lower() not in ['http','https'] or not u.hostname:
            return ''
        params = [(k,v) for k,v in parse_qsl(u.query,keep_blank_values=True)
                  if not k.lower().startswith('utm_') and k.lower() not in ['fbclid','gclid']]
        return urlunsplit((u.scheme.lower(),u.netloc.lower(),u.path,urlencode(params),'')).strip()
    except ValueError:
        return ''

df['link_candidato'] = df['link'].map(normalizar_link)
df['tamanho_calculado'] = df['full_text'].str.len()
df['palavras_calculadas'] = df['full_text'].str.split().str.len()
tamanho_informado = pd.to_numeric(df['text_length'], errors='coerce')
comparacao_tamanho = pd.DataFrame({
    'comparacao':['coincide_com_palavras','coincide_com_caracteres','nao_coincide_com_nenhum'],
    'registros':[int(tamanho_informado.eq(df['palavras_calculadas']).sum()),
                 int(tamanho_informado.eq(df['tamanho_calculado']).sum()),
                 int((tamanho_informado.notna() & tamanho_informado.ne(df['palavras_calculadas']) & tamanho_informado.ne(df['tamanho_calculado'])).sum())]})
salvar_tabela(comparacao_tamanho,'08_definicao_text_length.csv')
df['hash_texto_exato'] = df['full_text'].map(lambda s: sha_texto(s) if s.strip() else '')
df['hash_texto_normalizado'] = df['full_text'].map(lambda s: sha_texto(re.sub(r'\s+',' ',s).strip().casefold()) if s.strip() else '')
limite = pd.Timestamp(DATA_REFERENCIA) + pd.Timedelta(days=1)
flags = pd.DataFrame(index=df.index)
flags['titulo_vazio'] = df['title'].str.strip().eq('')
flags['texto_vazio'] = df['full_text'].str.strip().eq('')
flags['link_ausente_ou_invalido'] = df['link_candidato'].eq('')
flags['fonte_fora_lista'] = ~df['fonte_auditoria'].isin(FONTES_ESPERADAS)
flags['publicacao_ausente_ou_invalida'] = df['publication_date_local'].isna()
flags['atualizacao_formato_nao_reconhecido'] = df['publication_update_date'].str.strip().ne('') & df['publication_update_date_local'].isna()
flags['atualizacao_formato_com_pontos'] = df['publication_update_date'].str.match(r'^\d{4}\.\d{2}\.\d{2}T')
flags['coleta_ausente_ou_invalida'] = df['scraping_date_local'].isna()
flags['publicacao_apos_referencia'] = df['publication_date_local'].ge(limite)
# Comparamos dias: uma publicação sem horário não permite concluir sobre minutos.
flags['publicacao_apos_coleta_dia'] = df['publication_date_local'].dt.normalize().gt(df['scraping_date_local'].dt.normalize())
flags['atualizacao_antes_publicacao_dia'] = df['publication_update_date_local'].dt.normalize().lt(df['publication_date_local'].dt.normalize())
flags['tamanho_informado_invalido'] = tamanho_informado.isna() | tamanho_informado.lt(0) | tamanho_informado.mod(1).ne(0)
flags['tamanho_divergente_das_duas_definicoes'] = tamanho_informado.notna() & tamanho_informado.ne(df['tamanho_calculado']) & tamanho_informado.ne(df['palavras_calculadas'])
flags['texto_menos_200_caracteres'] = df['tamanho_calculado'].lt(200)  # Apenas sinalização, nunca filtro.
indicadores = flags.sum().rename_axis('verificacao').reset_index(name='registros_sinalizados')
salvar_tabela(indicadores, '06_indicadores_integridade.csv')
problemas = df[['linha_csv','title','link','source','publication_date','publication_update_date','scraping_date','text_length','tamanho_calculado','palavras_calculadas']].join(flags)
salvar_tabela(problemas.loc[flags.any(axis=1)], '07_registros_para_revisao.csv')
textos = df.groupby('fonte_auditoria')['tamanho_calculado'].agg(['count','min','median','mean','max']).reset_index()
salvar_tabela(textos,'08_tamanho_textos.csv')
print(indicadores.to_string(index=False))

### 5. Duplicidades e identidade
Relatamos separadamente registros integralmente iguais, links repetidos, textos repetidos e títulos repetidos por fonte. Os totais podem se sobrepor e não devem ser somados.

`registros_participantes` conta todas as linhas em grupos repetidos; `repeticoes_alem_primeiro` conta o excedente. Mesmo o excedente não é automaticamente descartável: links repetidos podem conter versões distintas, e textos idênticos podem ser republicações legítimas.

In [ ]:
resumos_duplicidade = []
def auditar_duplicidade(nome, chaves, validos):
    candidatos = df.loc[validos].copy()
    tamanho_grupo = candidatos.groupby(chaves,dropna=False)['linha_csv'].transform('size')
    repetidos = candidatos.loc[tamanho_grupo.gt(1)].copy()
    repetidos['tamanho_grupo'] = tamanho_grupo.loc[repetidos.index]
    resumos_duplicidade.append({'criterio':nome,'registros_participantes':len(repetidos),
        'repeticoes_alem_primeiro':int(candidatos.duplicated(chaves).sum()),
        'grupos_repetidos':int(repetidos[chaves].drop_duplicates().shape[0])})
    campos = list(dict.fromkeys(['linha_csv']+chaves+['title','link','source','publication_date','tamanho_calculado','hash_texto_exato','tamanho_grupo']))
    salvar_tabela(repetidos[campos].sort_values(chaves+['linha_csv']),f'10_duplicidade_{nome}.csv')

auditar_duplicidade('link_exato',['link'],df['link'].str.strip().ne(''))
auditar_duplicidade('link_normalizado_candidato',['link_candidato'],df['link_candidato'].ne(''))
auditar_duplicidade('texto_exato',['hash_texto_exato'],df['hash_texto_exato'].ne(''))
auditar_duplicidade('texto_normalizado',['hash_texto_normalizado'],df['hash_texto_normalizado'].ne(''))
auditar_duplicidade('titulo_fonte',['fonte_auditoria','title'],df['title'].str.strip().ne(''))
mask_exato = originais.duplicated(keep=False)
resumos_duplicidade.append({'criterio':'registro_integral','registros_participantes':int(mask_exato.sum()),
    'repeticoes_alem_primeiro':int(originais.duplicated().sum()),
    'grupos_repetidos':int(originais.loc[mask_exato].drop_duplicates().shape[0])})
salvar_tabela(df.loc[mask_exato,['linha_csv','title','link','source','publication_date']], '10_duplicidade_registro_integral.csv')
duplicidades = pd.DataFrame(resumos_duplicidade)
salvar_tabela(duplicidades, '09_resumo_duplicidades.csv')
print(duplicidades.to_string(index=False))

### 6. Cobertura das categorias editoriais
Lemos o dicionário com `ast.literal_eval`, sem executar o código do arquivo de categorias. A correspondência é exata, após retirar espaços nas extremidades. Categoria sem correspondência fica sinalizada para revisão; nenhum macrotema é inferido automaticamente.
O literal `NaN` presente no dicionário não é equiparado silenciosamente a uma célula vazia.

In [ ]:
arvore = ast.parse(ARQUIVO_CATEGORIAS.read_text(encoding='utf-8'))
no = next(n for n in arvore.body if isinstance(n,ast.Assign) and any(isinstance(t,ast.Name) and t.id=='CATEGORY_GROUPS' for t in n.targets))
grupos = ast.literal_eval(no.value)
pares = [(categoria.strip(),grupo) for grupo,categorias in grupos.items() for categoria in categorias]
contagem_categorias = Counter(c for c,g in pares)
conflitos = [c for c,n in contagem_categorias.items() if n>1]
assert not conflitos, f'Mapeamento ambíguo: {conflitos}'
mapa = dict(pares)
df['categoria_consulta'] = df['category_raw'].str.strip()
df['categoria_mapeada'] = df['categoria_consulta'].map(mapa).fillna('SEM_MAPEAMENTO')
categorias = df.groupby(['fonte_auditoria','category_raw','categoria_mapeada'],dropna=False).size().reset_index(name='noticias')
salvar_tabela(categorias,'11_cobertura_categorias.csv')
salvar_tabela(categorias.loc[categorias['categoria_mapeada'].eq('SEM_MAPEAMENTO')], '12_categorias_sem_mapeamento.csv')
salvar_tabela(df.groupby(['fonte_auditoria','categoria_mapeada']).size().reset_index(name='noticias'), '13_macrocategorias_mapeadas.csv')
print('Grupos no dicionário:', len(grupos))
print('Notícias sem mapeamento:', int(df['categoria_mapeada'].eq('SEM_MAPEAMENTO').sum()))
print('Categorias vazias:', int(df['categoria_consulta'].eq('').sum()))

### 7. Figuras de cobertura
As figuras mostram a composição do arquivo auditado. Não demonstram a produção jornalística total dos veículos, nem a evolução da atenção pública. Anos e meses sem registros recebem zero apenas para visualizar a cobertura deste arquivo. O ano de 2026 é parcial: o último registro recebido é de 30/06/2026; sua queda não pode ser interpretada como redução anual da cobertura.

In [ ]:
cores = {'Folha de S.Paulo':'tab:orange','G1':'tab:green','CNN Brasil':'tab:blue'}
def salvar_figura(fig,nome):
    for extensao in ['svg','png','pdf']:
        fig.savefig(FIGURAS/f'{nome}.{extensao}',dpi=180,bbox_inches='tight')
        shutil.copy2(FIGURAS/f'{nome}.{extensao}', PASTA_EXECUCAO/f'{nome}.{extensao}')
    plt.close(fig)

anual = df.dropna(subset=['ano']).groupby(['ano','fonte_auditoria']).size().unstack(fill_value=0)
anual = anual.reindex(range(int(df['ano'].min()),int(df['ano'].max())+1),fill_value=0)
fig,ax=plt.subplots(figsize=(10,4.5))
for fonte in FONTES_ESPERADAS:
    ax.plot(anual.index,anual.get(fonte,pd.Series(0,index=anual.index)),label=fonte,color=cores[fonte],linewidth=1.8)
ax.set(xlabel='Ano de publicação',ylabel='Notícias no arquivo',title='Cobertura anual do dataset por fonte')
ax.axvspan(2025.5,2026.5,color='#999999',alpha=.15)
ax.text(.98,.94,'2026 parcial: até 30/06',transform=ax.transAxes,ha='right',va='top',fontsize=9)
ax.grid(axis='y',alpha=.2);ax.legend(frameon=False);fig.tight_layout()
salvar_figura(fig,'01_cobertura_anual')

mensal_2025 = df.loc[df['ano'].eq(ANO_COMPARACAO)].groupby(['mes','fonte_auditoria']).size().unstack(fill_value=0).reindex(range(1,13),fill_value=0)
fig,ax=plt.subplots(figsize=(10,4.5))
for fonte in FONTES_ESPERADAS:
    ax.plot(mensal_2025.index,mensal_2025.get(fonte,pd.Series(0,index=mensal_2025.index)),marker='o',label=fonte,color=cores[fonte])
ax.set(xlabel='Mês de publicação',ylabel='Notícias no arquivo',title=f'Cobertura mensal de {ANO_COMPARACAO}')
ax.set_xticks(range(1,13));ax.grid(axis='y',alpha=.2);ax.legend(frameon=False);fig.tight_layout()
salvar_figura(fig,'02_cobertura_mensal_2025')
print('Duas figuras salvas em SVG, PNG e PDF.')

### 8. Verificações e proveniência
As verificações conferem a reconciliação das contagens e a preservação dos campos e do arquivo de entrada. O manifesto registra as regras da auditoria, os hashes das entradas, o código do notebook e as versões das bibliotecas.

O relatório reúne os achados, suas limitações e as recomendações de tratamento. A auditoria não modifica a base original nem aplica exclusões automáticas.


In [ ]:
verificacoes = {
    'contagem_fontes_reconciliada':int(por_fonte['noticias'].sum())==len(df),
    'contagem_anos_reconciliada':int(cobertura['noticias'].sum())==len(df),
    'contagem_meses_reconciliada':int(mensal['noticias'].sum())==len(df),
    'contagem_categorias_reconciliada':int(categorias['noticias'].sum())==len(df),
    'campos_originais_preservados':df[colunas_originais].equals(originais),
    'arquivo_original_preservado':sha_arquivo(ARQUIVO_NOTICIAS)==HASH_ORIGINAL,
    'marcos_folha_completos_na_tabela':len(marcos.loc[marcos['fonte_auditoria'].eq('Folha de S.Paulo')])==19,
}
assert all(verificacoes.values()), verificacoes
salvar_tabela(pd.DataFrame(verificacoes.items(),columns=['verificacao','consistente']), '14_verificacoes.csv')
manifesto = {'analise':'auditoria dos dados','status':'concluida; sem alteracao dos registros originais',
    'data_referencia':str(DATA_REFERENCIA),'executado_em':datetime.now(ZoneInfo('America/Sao_Paulo')).isoformat(),
    'arquivo':str(ARQUIVO_NOTICIAS.relative_to(PROJETO)),'sha256':HASH_ORIGINAL,
    'bytes':ARQUIVO_NOTICIAS.stat().st_size,'noticias':len(df),'colunas':colunas_originais,
    'categorias_sha256':HASH_CATEGORIAS,'marcos_temporais':MARCOS,
    'dependencias':{'python':platform.python_version(),'pandas':pd.__version__,'numpy':np.__version__,'matplotlib':matplotlib.__version__},
    'criterios':{'linhas_excluidas':0,'campos_originais_alterados':False,
    'datas':'ISO; conversão de fuso explícito para America/Sao_Paulo; datas sem fuso mantêm calendário informado',
    'duplicidades':'candidatos; sem decisão automática','texto_curto':'menos de 200 caracteres; somente sinalização',
    'categorias':'correspondência exata após strip; vazios não convertidos para literal NaN'},
    'verificacoes':verificacoes}
nb_path=PROJETO/'Pipeline_Noticias_IA.ipynb'
if nb_path.exists():
    # Hash do código e Markdown, independente das saídas e contadores de execução.
    nb_json=json.loads(nb_path.read_text(encoding='utf-8'))
    fontes=[{'tipo':c['cell_type'],'fonte':''.join(c['source'])} for c in nb_json['cells']]
    manifesto['notebook_source_sha256']=sha_texto(json.dumps(fontes,ensure_ascii=False,sort_keys=True))
(PASTA_EXECUCAO/'manifesto.json').write_text(json.dumps(manifesto,ensure_ascii=False,indent=2),encoding='utf-8')

def tabela_md(t):
    cab='| '+' | '.join(map(str,t.columns))+' |'
    linhas=[cab,'| '+' | '.join(['---']*len(t.columns))+' |']
    for linha in t.fillna('').itertuples(index=False,name=None):
        linhas.append('| '+' | '.join(str(v).replace('|','/') for v in linha)+' |')
    return '\n'.join(linhas)

resumo_2025=df.loc[df['ano'].eq(ANO_COMPARACAO)].groupby('fonte_auditoria').size().reset_index(name='noticias_2025')
texto = f'''# Auditoria do corpus de notícias sobre inteligência artificial

Auditoria executada em {manifesto['executado_em']}. Referência temporal: {DATA_REFERENCIA}.

Foram lidos {len(df):,} registros e {len(colunas_originais)} colunas. Nenhum registro foi excluído ou corrigido. As contagens descrevem o arquivo de entrada integral, antes de qualquer seleção analítica.

## Identificação da entrada

Arquivo: `{ARQUIVO_NOTICIAS.name}`. SHA-256: `{HASH_ORIGINAL}`.

## Fontes e cobertura

{tabela_md(por_fonte)}

## Recorte de 2025

{tabela_md(resumo_2025)}

## Integridade

{tabela_md(indicadores)}

## Candidatos a duplicidade

{tabela_md(duplicidades)}

Os critérios se sobrepõem. Não somar seus totais. Título repetido não implica notícia repetida. Textos iguais podem corresponder a republicações ou versões distintas. Examinar as tabelas de detalhe antes de excluir.

## Definição do tamanho dos textos

{tabela_md(comparacao_tamanho)}

O campo `text_length` coincide predominantemente com palavras, mas também há registros coincidentes com caracteres. Por isso, uma comparação apenas com caracteres produziria um diagnóstico enganoso. Recomenda-se armazenar separadamente as contagens de caracteres e palavras e examinar os casos sem correspondência. O metadado original foi preservado.

## Cobertura parcial de 2026

Os registros terminam em 30/06/2026. A quantidade de 2026 não deve ser comparada diretamente com um ano completo como evidência de crescimento ou retração.

## Categorias

O dicionário contém {len(grupos)} grupos e {len(pares)} entradas. Há {int(df['categoria_mapeada'].eq('SEM_MAPEAMENTO').sum())} notícias sem correspondência e {int(df['categoria_consulta'].eq('').sum())} categorias vazias. O mapeamento foi calculado apenas em memória e em tabelas auxiliares, preservando as categorias originais.

## Interpretação e limites

A cobertura descreve o arquivo recebido, não toda a produção dos veículos. Ausência de um ano ou mês pode decorrer da coleta ou curadoria. Não houve revisão semântica de todas as notícias, classificação de pertinência à IA, busca de duplicidades aproximadas, NER ou RE. Tamanho curto é somente um sinal. Datas fora do formato reconhecido não são necessariamente irrecuperáveis: há datas de atualização escritas como `AAAA.MM.DD`, que admitem normalização de formato em um campo derivado. A tabela de detalhe preserva os valores originais.

## Recomendações de tratamento dos dados

1. Preservar o CSV como entrada e criar a base preparada separadamente.
2. Separar identidade da notícia e versão do texto; usar hash para invalidar checkpoints quando o texto mudar.
3. Quarentenar textos ausentes, links inválidos e datas não interpretáveis quando impedirem a análise, sem apagar a entrada.
4. Não excluir automaticamente por título, texto curto, tamanho divergente ou candidato de duplicidade.
5. Revisar categorias sem correspondência e documentar o tratamento dos vazios.
   Os casos sem correspondência encontrados nesta versão são categorias vazias; não criar automaticamente uma classificação temática.
6. Para a comparação entre fontes em 2025, considerar os desequilíbrios mensais e o volume de notícias de cada fonte.
7. Usar todos os anos elegíveis disponíveis nos acumulados, preservando os dezenove marcos.

## Verificações

{tabela_md(pd.DataFrame(verificacoes.items(),columns=['verificacao','consistente']))}

## Escopo da análise

Este relatório descreve a auditoria documental e suas verificações. Não inclui extração de entidades ou relações nem métricas de redes.
'''
(PROJETO/'Auditoria_Dados.md').write_text(texto,encoding='utf-8')
(PASTA_EXECUCAO/'Auditoria_Dados.md').write_text(texto,encoding='utf-8')
print('Verificações:', verificacoes)
print('Auditoria concluída. Relatório, tabelas e metadados salvos.')

## 9. Preparação da base

Construímos a base derivada a partir dos campos originais. Nenhuma linha é excluída. Os campos novos não substituem os originais.

`news_id` depende da fonte, URL e data de publicação normalizada. A regra mantém republicações com datas diferentes separadas e preserva os parâmetros das URLs. `news_version_id` combina essa identidade ao conteúdo original do título, descrição e texto. Os IDs são strings SHA-256, independentes da posição das linhas.

As datas normalizadas representam calendário local: fusos explícitos são convertidos para São Paulo, e horários sem fuso são preservados sem atribuição de fuso. A precisão diária é mantida. O formato AAAA.MM.DD é normalizado em campo auxiliar.

As categorias editoriais são mapeadas por dicionário. Células vazias recebem NAO_INFORMADA; categorias não vazias sem correspondência recebem SEM_MAPEAMENTO. Marcadores de ausência não são tratados como temas.


In [ ]:
# A base derivada preserva todos os campos da entrada. Não há exclusões automáticas.
PROCESSADOS = PROJETO / 'dados/processados'
PROCESSADOS.mkdir(parents=True, exist_ok=True)
PREPARATION_VERSION = '1.0.0'
ID_VERSION = 'news_identity_v1'

def hash_estrutura(valores):
    # Serialização sem ambiguidade entre campos, independente de delimitadores nos textos.
    return sha_texto(json.dumps(valores, ensure_ascii=False, separators=(',', ':')))

def normalizar_data_preparacao(valor):
    bruto = str(valor).strip()
    if not bruto:
        return '', 'AUSENTE', 'NAO_INFORMADA', 'NAO_INFORMADO'
    convertido = re.sub(r'^(\d{4})\.(\d{2})\.(\d{2})(?=$|[T ])', r'\1-\2-\3', bruto)
    dt = data_calendario_local(convertido)
    if pd.isna(dt):
        return '', 'NAO_INTERPRETAVEL', 'NAO_INFORMADA', 'NAO_INFORMADO'
    somente_dia = bool(re.fullmatch(r'\d{4}-\d{2}-\d{2}', convertido))
    tem_fuso = pd.Timestamp(convertido).tzinfo is not None
    normalizado = dt.strftime('%Y-%m-%d') if somente_dia else dt.isoformat()
    status = 'FORMATO_NORMALIZADO' if convertido != bruto else 'INTERPRETADA'
    return normalizado, status, 'DIA' if somente_dia else 'DATA_HORA', 'EXPLICITO' if tem_fuso else 'NAO_INFORMADO'

def identificar_noticia(fonte, link, publicacao_normalizada):
    # A data distingue republicações sob um mesmo link. Não se remove nenhum parâmetro de URL.
    return hash_estrutura([ID_VERSION, fonte.strip(), link.strip(), publicacao_normalizada])

def identificar_conteudo(titulo, descricao, texto):
    # Inclui todos os campos textuais potencialmente usados pela extração.
    return hash_estrutura(['news_content_v1', titulo, descricao, texto])

preparada = originais.copy()
assert not set(['news_id','news_version_id','record_id','category_grouped']).intersection(preparada.columns)
preparada['source_normalized'] = preparada['source'].str.strip()
for campo in ['publication_date','publication_update_date','scraping_date']:
    valores = preparada[campo].map(normalizar_data_preparacao)
    nomes = [campo+'_normalized', campo+'_status', campo+'_precision', campo+'_timezone_status']
    preparada[nomes] = pd.DataFrame(valores.tolist(), index=preparada.index, columns=nomes)

preparada['news_id'] = [identificar_noticia(f,l,d) for f,l,d in zip(
    preparada['source_normalized'],preparada['link'],preparada['publication_date_normalized'])]
preparada['content_hash'] = [identificar_conteudo(t,d,b) for t,d,b in zip(
    preparada['title'],preparada['description'],preparada['full_text'])]
preparada['text_sha256'] = preparada['full_text'].map(sha_texto)
preparada['news_version_id'] = [hash_estrutura(['news_version_v1',i,h]) for i,h in zip(preparada['news_id'],preparada['content_hash'])]
# record_id identifica o registro completo; news_id e news_version_id possuem funções distintas.
preparada['record_id'] = [hash_estrutura(['record_v1']+list(valores)) for valores in originais.itertuples(index=False,name=None)]
preparada['text_length_chars'] = preparada['full_text'].str.len().astype('int64')
preparada['text_length_words'] = preparada['full_text'].str.split().str.len().astype('int64')
categoria = preparada['category_raw'].str.strip()
preparada['category_grouped'] = categoria.map(mapa).fillna('SEM_MAPEAMENTO')
preparada.loc[categoria.eq(''),'category_grouped'] = 'NAO_INFORMADA'
preparada['category_mapping_status'] = np.select(
    [categoria.eq(''),preparada['category_grouped'].eq('SEM_MAPEAMENTO')],
    ['NAO_INFORMADA','SEM_MAPEAMENTO'],default='MAPEADA')
preparada['publication_year'] = df['ano']
preparada['publication_month'] = df['mes']
# As flags documentam situações para inspeção. Não são critérios automáticos de exclusão.
preparada['flag_update_format_normalized'] = preparada['publication_update_date_status'].eq('FORMATO_NORMALIZADO')
pub_dia = pd.to_datetime(preparada['publication_date_normalized'],format='mixed',errors='coerce').dt.normalize()
upd_dia = pd.to_datetime(preparada['publication_update_date_normalized'],format='mixed',errors='coerce').dt.normalize()
preparada['flag_update_before_publication'] = upd_dia.lt(pub_dia)
preparada['flag_text_short'] = preparada['text_length_chars'].lt(200)
preparada['flag_link_repeated'] = preparada['link'].str.strip().duplicated(keep=False) & preparada['link'].str.strip().ne('')
preparada['flag_text_repeated'] = preparada['text_sha256'].duplicated(keep=False) & preparada['full_text'].str.strip().ne('')
preparada['flag_title_source_repeated'] = preparada.duplicated(['source_normalized','title'],keep=False)
preparada['flag_record_repeated'] = preparada['record_id'].duplicated(keep=False)
preparada['flag_news_id_repeated'] = preparada['news_id'].duplicated(keep=False)
preparada['flag_news_version_repeated'] = preparada['news_version_id'].duplicated(keep=False)
preparada['flag_text_length_unmatched'] = flags['tamanho_divergente_das_duas_definicoes']
preparada['flag_publication_unparseable'] = preparada['publication_date_status'].isin(['AUSENTE','NAO_INTERPRETAVEL'])
preparada['flag_category_missing'] = preparada['category_mapping_status'].eq('NAO_INFORMADA')
preparada['flag_category_unmapped'] = preparada['category_mapping_status'].eq('SEM_MAPEAMENTO')

BASE_PREPARADA = PROCESSADOS / 'noticias_preparadas.csv'
preparada.to_csv(BASE_PREPARADA,index=False,encoding='utf-8-sig')
PREP_RUN = PROJETO / 'resultados/execucao' / f'preparacao_{EXECUCAO_ID}_{HASH_ORIGINAL[:12]}'
PREP_RUN.mkdir(parents=True,exist_ok=True)

def salvar_preparacao(tabela, nome):
    tabela.to_csv(TABELAS/nome,index=False,encoding='utf-8-sig')
    tabela.to_csv(PREP_RUN/nome,index=False,encoding='utf-8-sig')

contagem_macro = preparada.groupby(['source_normalized','category_grouped','category_mapping_status']).size().reset_index(name='noticias')
salvar_preparacao(contagem_macro,'20_macrocategorias_base_preparada.csv')
status_datas = pd.concat([preparada.groupby(c+'_status').size().rename_axis('status').reset_index(name='noticias').assign(campo=c)
                         for c in ['publication_date','publication_update_date','scraping_date']],ignore_index=True)
salvar_preparacao(status_datas,'21_status_datas_preparadas.csv')
flags_preparacao = [c for c in preparada if c.startswith('flag_')]
salvar_preparacao(preparada[flags_preparacao].sum().rename_axis('verificacao').reset_index(name='registros'), '22_sinalizacoes_base_preparada.csv')
ids = ['record_id','news_id','news_version_id']
salvar_preparacao(preparada[ids+['source','link','publication_date','publication_date_normalized','content_hash']], '23_identificadores_noticias.csv')
salvar_preparacao(preparada.loc[preparada['flag_update_format_normalized'] | preparada['flag_update_before_publication'],
    ids+['link','publication_date','publication_date_normalized','publication_update_date','publication_update_date_normalized',
         'publication_update_date_status','flag_update_before_publication']], '24_datas_normalizadas_e_sinalizadas.csv')
print('Base preparada:', BASE_PREPARADA.relative_to(PROJETO))
print('Notícias preservadas:',len(preparada))
print('Macrocategorias:',preparada['category_grouped'].nunique())
print(preparada['category_mapping_status'].value_counts().to_string())
print('Atualizações com formato normalizado:',int(preparada['flag_update_format_normalized'].sum()))


### 9.1 Verificações e documentação da base preparada

As verificações conferem preservação dos dados, serialização em CSV, reconciliação de categorias e estabilidade dos identificadores. Casos controlados verificam mudanças de conteúdo, datas distintas sob um mesmo link, conversão de fuso e datas inexistentes. As situações problemáticas permanecem sinalizadas, sem correção por suposição.


In [ ]:
# Verificações de integridade e da identidade documental.
cheques_preparacao = {
    'numero_registros_preservado':len(preparada)==len(originais),
    'todos_campos_originais_preservados':preparada[colunas_originais].equals(originais),
    'arquivo_original_preservado':sha_arquivo(ARQUIVO_NOTICIAS)==HASH_ORIGINAL,
    'record_id_unico_nesta_base':preparada['record_id'].is_unique,
    'news_id_unico_nesta_base':preparada['news_id'].is_unique,
    'news_version_id_unico_nesta_base':preparada['news_version_id'].is_unique,
    'noticias_macro_reconciliadas':int(contagem_macro['noticias'].sum())==len(preparada),
    'categorias_preenchidas':preparada['category_grouped'].str.strip().ne('').all(),
    'contagem_caracteres_reconciliada':preparada['text_length_chars'].eq(df['tamanho_calculado']).all(),
    'contagem_palavras_reconciliada':preparada['text_length_words'].eq(df['palavras_calculadas']).all(),
}
# Reordenar a base não pode modificar a identidade calculada.
amostra = preparada.sample(min(200,len(preparada)),random_state=42)
cheques_preparacao['identidade_independente_ordem'] = all(
    identificar_noticia(r.source_normalized,r.link,r.publication_date_normalized)==r.news_id
    for r in amostra.itertuples(index=False))
# Uma mudança textual cria outra versão, mantendo a identidade editorial.
r = preparada.iloc[0]
novo_hash = identificar_conteudo(r['title'],r['description'],r['full_text']+'\nTexto adicional de verificação.')
cheques_preparacao['mudanca_textual_invalida_versao'] = (
    novo_hash != r['content_hash'] and
    hash_estrutura(['news_version_v1',r['news_id'],novo_hash]) != r['news_version_id'])
cheques_preparacao['mesmo_link_datas_diferentes_nao_fundidas'] = (
    identificar_noticia('Fonte','https://exemplo.test/a','2024-01-01') !=
    identificar_noticia('Fonte','https://exemplo.test/a','2025-01-01'))
cheques_preparacao['normalizacao_pontos_equivalente'] = (
    normalizar_data_preparacao('2025.07.26T11:04:00')[0] == normalizar_data_preparacao('2025-07-26T11:04:00')[0])
cheques_preparacao['data_inexistente_nao_corrigida'] = normalizar_data_preparacao('2025.02.30T11:04:00')[1]=='NAO_INTERPRETAVEL'
cheques_preparacao['horario_fuso_convertido_calendario'] = normalizar_data_preparacao('2025-01-01T01:00:00Z')[0]=='2024-12-31T22:00:00'
cheques_preparacao['data_diaria_preserva_precisao'] = normalizar_data_preparacao('1994-02-02')[0:3] == ('1994-02-02','INTERPRETADA','DIA')

# Reabrir o CSV verifica que a serialização não perdeu registros, textos ou identificadores.
reaberta = pd.read_csv(BASE_PREPARADA,encoding='utf-8-sig',dtype=str,keep_default_na=False)
cheques_preparacao['csv_preserva_registros_e_campos_originais'] = (
    len(reaberta)==len(preparada) and reaberta[colunas_originais].equals(originais))
cheques_preparacao['csv_preserva_identificadores'] = reaberta[ids].equals(preparada[ids])
cheques_preparacao = {k:bool(v) for k,v in cheques_preparacao.items()}
assert all(cheques_preparacao.values()),cheques_preparacao
salvar_preparacao(pd.DataFrame(cheques_preparacao.items(),columns=['verificacao','consistente']), '25_verificacoes_preparacao.csv')
del reaberta
manifesto_preparacao = {
    'analise':'preparacao do corpus de noticias','preparation_version':PREPARATION_VERSION,
    'identity_version':ID_VERSION,'executado_em':datetime.now(ZoneInfo('America/Sao_Paulo')).isoformat(),
    'entrada':str(ARQUIVO_NOTICIAS.relative_to(PROJETO)),'entrada_sha256':HASH_ORIGINAL,
    'dicionario_sha256':HASH_CATEGORIAS,'saida':str(BASE_PREPARADA.relative_to(PROJETO)),
    'saida_sha256':sha_arquivo(BASE_PREPARADA),'registros_entrada':len(originais),
    'registros_saida':len(preparada),'registros_excluidos':0,
    'colunas_originais':colunas_originais,'colunas_derivadas':[c for c in preparada if c not in colunas_originais],
    'notebook_source_sha256':manifesto.get('notebook_source_sha256'),
    'dependencias':manifesto['dependencias'],'verificacoes':cheques_preparacao,
    'identidade':'SHA-256 da versao da regra, fonte sem espacos nas extremidades, URL sem espacos nas extremidades e data de publicacao normalizada; nenhuma equivalencia entre URLs e inferida',
    'versao_textual':'SHA-256 de news_id e content_hash; content_hash inclui titulo, descricao e full_text originais',
    'datas':'ISO em calendario local; fuso explicito convertido para America/Sao_Paulo; sem fuso preservado; precisao e presenca de fuso registradas; prefixo AAAA.MM.DD convertido em campo auxiliar',
    'categorias':'dicionario por correspondencia exata apos strip; vazias NAO_INFORMADA; nao vazias desconhecidas SEM_MAPEAMENTO',
    'limitacoes':['identidade documental operacional, nao resolucao semantica de republicacoes',
                  'uma mudanca da fonte, URL ou data pode exigir reconciliacao de identidade',
                  'flags nao aplicam exclusoes nem atestam validade semantica',
                  'checkpoints de extracao deverao incluir modelo, prompt e demais parametros alem da versao textual']}
(PREP_RUN/'manifesto_preparacao.json').write_text(json.dumps(manifesto_preparacao,ensure_ascii=False,indent=2),encoding='utf-8')
(PROCESSADOS/'metadados_preparacao.json').write_text(json.dumps(manifesto_preparacao,ensure_ascii=False,indent=2),encoding='utf-8')

relatorio_preparacao = f'''# Preparação do corpus de notícias sobre inteligência artificial

## Resultado

Base derivada: `dados/processados/noticias_preparadas.csv`.

Foram preservados {len(preparada):,} registros e os {len(colunas_originais)} campos originais. Não houve exclusões. As informações normalizadas são campos adicionais.

## Identificação documental

- `news_id`: identidade operacional baseada em fonte, URL e publicação normalizada. Datas distintas em uma URL permanecem distintas.
- `content_hash`: hash do título, descrição e texto integral originais.
- `text_sha256`: hash somente do texto integral original.
- `news_version_id`: identidade da notícia combinada ao hash do conteúdo; permite detectar mudança textual.
- `record_id`: hash de todos os campos originais, para rastrear o registro de entrada.

São strings SHA-256, não números de linha. A ordem dos registros não define os identificadores. A regra é conservadora: não resolve automaticamente equivalências entre URLs ou republicações. Alterar um metadado que integra a identidade pode exigir reconciliação explícita. Os checkpoints de extração também deverão considerar modelo, prompt e parâmetros.

## Campos derivados

| Campos | Definição |
| --- | --- |
| `source_normalized` | Fonte sem espaços nas extremidades |
| `*_date_normalized` | Data interpretada no calendário local, sem alterar a entrada |
| `*_date_status` | INTERPRETADA, FORMATO_NORMALIZADO, AUSENTE ou NAO_INTERPRETAVEL |
| `*_date_precision` | DIA, DATA_HORA ou NAO_INFORMADA |
| `*_date_timezone_status` | EXPLICITO ou NAO_INFORMADO no valor original |
| `text_length_chars` | Número de caracteres do texto original, incluindo espaços |
| `text_length_words` | Número de itens separados por espaços no texto original |
| `category_grouped` | Macrocategoria editorial ou marcador de ausência/mapeamento |
| `category_mapping_status` | MAPEADA, NAO_INFORMADA ou SEM_MAPEAMENTO |
| `publication_year`, `publication_month` | Ano e mês no calendário de publicação adotado |
| `flag_*` | Situações para inspeção; não implicam exclusão |

Nos campos normalizados, datas com fuso explícito foram convertidas para São Paulo. Datas sem fuso mantêm o horário informado; não é atribuído um fuso a elas. Datas com precisão diária continuam sem horário. Essas colunas servem à análise de calendário, não representam uma série universal de instantes UTC.

## Datas

{tabela_md(status_datas)}

Foram normalizados {int(preparada['flag_update_format_normalized'].sum())} formatos de atualização. Permanecem sinalizados {int(preparada['flag_update_before_publication'].sum())} registros com dia de atualização anterior ao da publicação. Nenhuma data foi corrigida por suposição.

## Macrocategorias

{tabela_md(preparada['category_mapping_status'].value_counts().rename_axis('situacao').reset_index(name='noticias'))}

O agrupamento utiliza o dicionário de categorias e não classifica o conteúdo das notícias por modelo de linguagem. Há {preparada['category_grouped'].nunique()} valores observados em `category_grouped`, incluindo os marcadores de ausência. A composição mistura temas e editorias de formato/origem; deve ser interpretada como organização editorial.

## Verificações

{tabela_md(pd.DataFrame(cheques_preparacao.items(),columns=['verificacao','consistente']))}

## Proveniência e reprodução

Entrada SHA-256: `{HASH_ORIGINAL}`.

Saída SHA-256: `{manifesto_preparacao['saida_sha256']}`.

Versão da preparação: `{PREPARATION_VERSION}`. Versão da identidade: `{ID_VERSION}`.

O manifesto em `resultados/execucao/` registra regras, ambiente, código e hashes. O notebook pode recriar a base a partir dos arquivos originais. Os duplicados e textos curtos permanecem na base; não foram aplicadas seleções científicas ou executadas extrações de entidades e relações.
'''
(PROJETO/'Preparacao_Dados.md').write_text(relatorio_preparacao,encoding='utf-8')
(PREP_RUN/'Preparacao_Dados.md').write_text(relatorio_preparacao,encoding='utf-8')
print('Verificações consistentes:',sum(cheques_preparacao.values()),'de',len(cheques_preparacao))
print('CSV reaberto e comparado com a entrada; nenhum registro excluído.')
print('Relatório de preparação e manifesto salvos.')


## 10. Carregar a base preparada

Esta célula pode ser executada isoladamente em um kernel novo, sem repetir a auditoria. O diretório de trabalho deve ser a pasta do notebook. Ela disponibiliza `df_news`, restaura os tipos dos campos derivados e confere o CSV, a entrada e o dicionário contra os hashes registrados nos metadados, quando esses arquivos estão disponíveis.

O carregamento restaura os dados salvos, não a memória de uma execução anterior. Campos originais continuam como strings; datas normalizadas mantêm a precisão registrada. Modelos, prompts e parâmetros compõem as chaves dos checkpoints de extração.


In [ ]:
# Esta célula é independente: pode ser executada em um kernel novo, na pasta do notebook.
from pathlib import Path
import json, hashlib
import pandas as pd

def carregar_base_preparada(pasta_projeto=None, conferir_hash=True):
    raiz = Path.cwd() if pasta_projeto is None else Path(pasta_projeto)
    csv = raiz/'dados/processados/noticias_preparadas.csv'
    meta = raiz/'dados/processados/metadados_preparacao.json'
    if not csv.is_file() or not meta.is_file():
        raise FileNotFoundError('Base preparada ou metadados ausentes. Execute a preparação ou confira a pasta do projeto.')
    m = json.loads(meta.read_text(encoding='utf-8'))
    def hash_local(arquivo):
        h=hashlib.sha256()
        with arquivo.open('rb') as f:
            for bloco in iter(lambda:f.read(1024*1024),b''):
                h.update(bloco)
        return h.hexdigest()
    if conferir_hash:
        if hash_local(csv)!=m['saida_sha256']:
            raise ValueError('O CSV difere da versão registrada nos metadados. Refaça a preparação ou investigue a alteração.')
        entrada=raiz/m['entrada']
        if entrada.exists() and hash_local(entrada)!=m['entrada_sha256']:
            raise ValueError('A entrada foi alterada após a preparação. Refaça a preparação antes de continuar.')
        dicionario=raiz/'dados/originais/category_groups.py'
        if dicionario.exists() and hash_local(dicionario)!=m['dicionario_sha256']:
            raise ValueError('O dicionário de categorias foi alterado. Refaça a preparação antes de continuar.')
    base = pd.read_csv(csv,encoding='utf-8-sig',dtype=str,keep_default_na=False)
    esperadas=m['colunas_originais']+m['colunas_derivadas']
    if base.columns.tolist()!=esperadas or len(base)!=m['registros_saida']:
        raise ValueError('Esquema ou contagem incompatível com os metadados.')
    for campo in ['text_length_chars','text_length_words','publication_year','publication_month']:
        base[campo]=pd.to_numeric(base[campo],errors='raise').astype('Int64')
    for campo in [c for c in base if c.startswith('flag_')]:
        if not base[campo].isin(['True','False']).all():
            raise ValueError(f'Flag com valor inesperado: {campo}')
        base[campo]=base[campo].map({'True':True,'False':False}).astype(bool)
    return base

df_news = carregar_base_preparada()
print(f'Base carregada do disco: {len(df_news):,} notícias. Campos originais preservados como texto.')


## 11. Caracterização do corpus

Estas análises dependem apenas de `df_news`, carregado na seção 10. Descrevem o arquivo recebido, sem excluir registros ou regravar a base preparada. Não medem toda a produção dos veículos, a opinião pública ou efeitos causais.

### 11.1 Volume de publicações

Calculamos contagens e proporções por fonte, ano e mês. Os anos com menos de doze meses observados e 2026 parcial são identificados. A comparação de janeiro a junho de 2025 e 2026 usa a mesma duração de calendário, sem pressupor igual completude da coleta.


In [ ]:
# Esta análise depende somente de df_news, carregado na seção anterior.
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib, json, re, unicodedata, platform, textwrap
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

CORPUS_ROOT = Path.cwd()
CORPUS_TABLES = CORPUS_ROOT/'resultados/tabelas'
CORPUS_FIGURES = CORPUS_ROOT/'resultados/figuras'
CORPUS_INPUT = CORPUS_ROOT/'dados/processados/noticias_preparadas.csv'
CORPUS_YEAR_COMPARE = 2025
CORPUS_SOURCES = ['Folha de S.Paulo','G1','CNN Brasil']
CORPUS_COLORS = {'Folha de S.Paulo':'tab:orange','G1':'tab:green','CNN Brasil':'tab:blue'}
CORPUS_MISSING_CATEGORIES = ['NAO_INFORMADA','SEM_MAPEAMENTO']
CORPUS_MISSING_AUTHORS = {'sem autor','sem autoria','autor nao informado','nao informado',
    'nao identificada','nao identificado','sem identificacao','autor desconhecido','nan','none','null','n/a'}
CORPUS_ANALYSIS_VERSION = '1.0.0'

def corpus_file_hash(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

assert 'df_news' in globals(), 'Execute a seção 10 para carregar df_news.'
CORPUS_INPUT_HASH = corpus_file_hash(CORPUS_INPUT)
CORPUS_EXECUTION_ID = datetime.now(ZoneInfo('America/Sao_Paulo')).strftime('%Y%m%d_%H%M%S')
CORPUS_RUN = CORPUS_ROOT/'resultados/execucao'/f'corpus_{CORPUS_EXECUTION_ID}_{CORPUS_INPUT_HASH[:12]}'
for directory in [CORPUS_TABLES,CORPUS_FIGURES,CORPUS_RUN]:directory.mkdir(parents=True,exist_ok=True)
corpus_outputs = []

def corpus_save_table(table,name):
    destination=CORPUS_TABLES/name
    table.to_csv(destination,index=False,encoding='utf-8-sig')
    table.to_csv(CORPUS_RUN/name,index=False,encoding='utf-8-sig')
    corpus_outputs.append(destination)

# Usa apenas os campos necessários às estatísticas, sem copiar os textos integrais.
corpus = df_news[['news_id','record_id','source_normalized','publication_date_normalized',
    'publication_year','publication_month','category_grouped','category_raw','author',
    'text_length_words','text_length_chars']].copy()
corpus = corpus.rename(columns={'source_normalized':'source','publication_year':'year','publication_month':'month'})
corpus['publication_calendar'] = pd.to_datetime(corpus['publication_date_normalized'],format='mixed',errors='coerce')
corpus['year_month'] = corpus['publication_calendar'].dt.to_period('M').astype(str)
corpus['category_known'] = ~corpus['category_grouped'].isin(CORPUS_MISSING_CATEGORIES)
valid_calendar = corpus['publication_calendar'].notna()
assert valid_calendar.all(), 'Há datas não interpretáveis: definir o tratamento antes das séries temporais.'
assert corpus['record_id'].is_unique and corpus['news_id'].is_unique, 'Definir unidade de contagem antes de contar versões repetidas.'

summary_sources = corpus.groupby('source').agg(news=('news_id','size'),
    first_publication=('publication_calendar','min'),last_publication=('publication_calendar','max'),
    known_category_news=('category_known','sum')).reset_index()
summary_sources['share_pct']=100*summary_sources['news']/len(corpus)
annual = corpus.groupby(['source','year']).agg(news=('news_id','size'),
    observed_months=('month','nunique'),first_publication=('publication_calendar','min'),
    last_publication=('publication_calendar','max')).reset_index()
annual['share_within_year_pct']=100*annual['news']/annual.groupby('year')['news'].transform('sum')
annual['relative_change_previous_observed_year_pct']=annual.groupby('source')['news'].pct_change(fill_method=None)*100
annual['previous_observed_year']=annual.groupby('source')['year'].shift()
annual['consecutive_to_previous']=annual['year'].sub(annual['previous_observed_year']).eq(1)
annual['observed_months_lt_12']=annual['observed_months'].lt(12)
annual['partial_2026']=annual['year'].eq(2026)
monthly = corpus.groupby(['source','year','month','year_month']).size().reset_index(name='news')
monthly['share_within_source_year_pct']=100*monthly['news']/monthly.groupby(['source','year'])['news'].transform('sum')
corpus_save_table(summary_sources,'30_corpus_fontes.csv')
corpus_save_table(annual,'31_corpus_publicacoes_ano_fonte.csv')
corpus_save_table(monthly,'32_corpus_publicacoes_mes_fonte.csv')



### 11.2 Distribuição e diversidade editorial

As participações incluem categorias não informadas no denominador de notícias de cada fonte e período. A diversidade usa H = −Σ pᵢ ln(pᵢ) entre categorias conhecidas, e exp(H) expressa o número efetivo de categorias. Editorias de formato e origem, como opinião e publicidade, permanecem na taxonomia. As medidas não representam automaticamente diversidade temática e dependem do volume e da classificação editorial disponível.


In [ ]:
category_source = corpus.groupby(['source','category_grouped']).size().reset_index(name='news')
category_source['share_all_news_pct']=100*category_source['news']/category_source.groupby('source')['news'].transform('sum')
category_year = corpus.groupby(['source','year','category_grouped']).size().reset_index(name='news')
category_year['share_all_news_pct']=100*category_year['news']/category_year.groupby(['source','year'])['news'].transform('sum')
category_month = corpus.groupby(['source','year','month','category_grouped']).size().reset_index(name='news')
category_month['share_all_news_pct']=100*category_month['news']/category_month.groupby(['source','year','month'])['news'].transform('sum')
category_raw = corpus.groupby(['source','category_raw','category_grouped']).size().reset_index(name='news')
corpus_save_table(category_source,'33_corpus_macrocategorias_fonte.csv')
corpus_save_table(category_year,'34_corpus_macrocategorias_ano_fonte.csv')
corpus_save_table(category_month,'35_corpus_macrocategorias_mes_fonte.csv')
corpus_save_table(category_raw,'36_corpus_editorias_originais.csv')

# H = -sum(p log p). exp(H) é o número efetivo de categorias, calculado entre categorias conhecidas.
diversity_rows=[]
for (source,year),g in corpus.groupby(['source','year']):
    counts=g.loc[g['category_known'],'category_grouped'].value_counts()
    probabilities=counts/counts.sum() if len(counts) else pd.Series(dtype=float)
    entropy=float(-(probabilities*np.log(probabilities)).sum()) if len(counts) else np.nan
    diversity_rows.append({'source':source,'year':int(year),'news':len(g),'known_category_news':int(counts.sum()),
        'category_missing_pct':100*(1-g['category_known'].mean()),'observed_macro_categories':len(counts),
        'shannon_nats':entropy,'effective_categories':float(np.exp(entropy)) if len(counts) else np.nan,
        'dominant_category_share_known_pct':100*counts.iloc[0]/counts.sum() if len(counts) else np.nan})
diversity=pd.DataFrame(diversity_rows)
corpus_save_table(diversity,'37_corpus_diversidade_editorial.csv')



### 11.3 Autoria registrada

Assinaturas podem designar pessoas, coautorias, agências ou redações. Normalizamos apenas espaços, prefixo inicial Por e capitalização para comparação dentro de cada fonte. Não dividimos por vírgula nem resolvemos identidades pessoais. Os rankings, HHI e continuidade são de unidades de assinatura, e não de jornalistas individualizados.

Campos vazios e marcadores como Sem autor são separados e excluídos dos denominadores de assinaturas presentes. O mapeamento de valores originais e normalizados permite inspeção. Concentração: HHI = Σ pᵢ². A continuidade usa anos observados desde a primeira ocorrência da assinatura, incluindo 2026 parcial.


In [ ]:
def corpus_author_signature(value):
    normalized=unicodedata.normalize('NFC',str(value))
    normalized=re.sub(r'\s+',' ',normalized).strip()
    return re.sub(r'^por\s+','',normalized,flags=re.I).strip()

def corpus_author_status(signature):
    if not signature:return 'CAMPO_VAZIO'
    folded=''.join(c for c in unicodedata.normalize('NFKD',signature.casefold()) if not unicodedata.combining(c))
    return 'NAO_IDENTIFICADA' if folded in CORPUS_MISSING_AUTHORS else 'ASSINATURA_PRESENTE'

corpus['author_signature']=corpus['author'].map(corpus_author_signature)
corpus['author_status']=corpus['author_signature'].map(corpus_author_status)
corpus['author_key']=corpus['author_signature'].str.casefold()
author_coverage=corpus.groupby(['source','author_status']).size().reset_index(name='news')
author_coverage['share_pct']=100*author_coverage['news']/author_coverage.groupby('source')['news'].transform('sum')
author_year=corpus.groupby(['source','year','author_status']).size().reset_index(name='news')
author_year['share_pct']=100*author_year['news']/author_year.groupby(['source','year'])['news'].transform('sum')
author_map=corpus.groupby(['source','author','author_signature','author_key','author_status']).size().reset_index(name='news')
corpus_save_table(author_coverage,'38_corpus_cobertura_autoria.csv')
corpus_save_table(author_year,'39_corpus_cobertura_autoria_anual.csv')
corpus_save_table(author_map,'40_corpus_assinaturas_mapeamento.csv')
signed=corpus.loc[corpus['author_status'].eq('ASSINATURA_PRESENTE')].copy()
# Um rótulo determinístico por chave, conservando todas as variantes no mapeamento.
author_labels=signed.groupby(['source','author_key'])['author_signature'].agg(lambda x:sorted(set(x))[0]).reset_index()
author_by_year=signed.groupby(['source','year','author_key']).size().reset_index(name='news')
author_by_year=author_by_year.merge(author_labels,on=['source','author_key'],validate='many_to_one')
author_by_year['share_signed_news_pct']=100*author_by_year['news']/author_by_year.groupby(['source','year'])['news'].transform('sum')
rank_parts=[]
for label,part in [('CORPUS',signed),('2025',signed.loc[signed['year'].eq(CORPUS_YEAR_COMPARE)])]:
    table=part.groupby(['source','author_key']).size().reset_index(name='news')
    table=table.merge(author_labels,on=['source','author_key'],validate='one_to_one')
    table['period']=label
    table['share_signed_news_pct']=100*table['news']/table.groupby('source')['news'].transform('sum')
    table['rank']=table.groupby('source')['news'].rank(method='min',ascending=False).astype(int)
    rank_parts.append(table)
author_rank=pd.concat(rank_parts,ignore_index=True).sort_values(['period','source','rank','author_key'])
concentration_rows=[]
for (source,year),g in author_by_year.groupby(['source','year']):
    n=int(g['news'].sum());weights=g['news'].to_numpy()/n
    concentration_rows.append({'source':source,'year':int(year),'signed_news':n,'signature_units':len(g),
        'hhi_signatures':float(np.square(weights).sum()),'effective_signature_units':float(1/np.square(weights).sum()),
        'top_5_share_signed_pct':100*g.nlargest(5,'news')['news'].sum()/n,
        'top_10_share_signed_pct':100*g.nlargest(10,'news')['news'].sum()/n})
author_concentration=pd.DataFrame(concentration_rows)
continuity=author_by_year.groupby(['source','author_key']).agg(news=('news','sum'),
    first_year=('year','min'),last_year=('year','max'),active_observed_years=('year','nunique')).reset_index()
source_years={s:set(g['year'].astype(int)) for s,g in annual.groupby('source')}
continuity['available_years_since_first']=[sum(y>=int(first) for y in source_years[s]) for s,first in zip(continuity['source'],continuity['first_year'])]
continuity['active_share_available_years_pct']=100*continuity['active_observed_years']/continuity['available_years_since_first']
continuity=continuity.merge(author_labels,on=['source','author_key'],validate='one_to_one')
corpus_save_table(author_rank,'41_corpus_rankings_assinaturas.csv')
corpus_save_table(author_by_year,'42_corpus_assinaturas_ano.csv')
corpus_save_table(author_concentration,'43_corpus_concentracao_assinaturas.csv')
corpus_save_table(continuity,'44_corpus_continuidade_assinaturas.csv')



### 11.4 Tamanho dos textos e janelas comparáveis

Caracteres incluem espaços e quebras de linha; palavras são itens separados por espaços. Exportamos média, mediana, quartis, extremos e percentil 90, sem truncar textos. Comparações do corpus completo e de 2025 são identificadas separadamente. As janelas de janeiro a junho evitam comparar 2026 parcial com um ano completo.


In [ ]:
text_rows=[]
for period,part in [('CORPUS',corpus),('2025',corpus.loc[corpus['year'].eq(CORPUS_YEAR_COMPARE)])]:
    for source,g in part.groupby('source'):
        for metric in ['text_length_words','text_length_chars']:
            v=g[metric].astype(float)
            text_rows.append({'period':period,'source':source,'metric':metric,'news':len(v),
                'min':v.min(),'q25':v.quantile(.25),'median':v.median(),'mean':v.mean(),
                'q75':v.quantile(.75),'q90':v.quantile(.9),'max':v.max()})
text_summary=pd.DataFrame(text_rows)
text_year=corpus.groupby(['source','year']).agg(news=('news_id','size'),
    median_words=('text_length_words','median'),mean_words=('text_length_words','mean'),
    median_chars=('text_length_chars','median')).reset_index()
text_category=corpus.groupby(['source','category_grouped']).agg(news=('news_id','size'),
    median_words=('text_length_words','median'),mean_words=('text_length_words','mean')).reset_index()
corpus_save_table(text_summary,'45_corpus_tamanho_textos.csv')
corpus_save_table(text_year,'46_corpus_tamanho_textos_anual.csv')
corpus_save_table(text_category,'47_corpus_tamanho_textos_categorias.csv')

# Janela com mesma duração de calendário: 1 de janeiro a 30 de junho de 2025 e de 2026.
matched=corpus.loc[corpus['year'].isin([2025,2026]) & corpus['month'].between(1,6)]
matched_counts=matched.groupby(['source','year']).size().unstack(fill_value=0).reindex(columns=[2025,2026],fill_value=0)
matched_counts.columns=['news_2025_jan_jun','news_2026_jan_jun']
matched_counts=matched_counts.reset_index()
matched_counts['relative_change_pct']=100*(matched_counts['news_2026_jan_jun']/matched_counts['news_2025_jan_jun']-1)
matched_categories=matched.groupby(['source','year','category_grouped']).size().reset_index(name='news')
matched_categories['share_pct']=100*matched_categories['news']/matched_categories.groupby(['source','year'])['news'].transform('sum')
corpus_save_table(matched_counts,'48_corpus_janelas_jan_jun.csv')
corpus_save_table(matched_categories,'49_corpus_categorias_janelas_jan_jun.csv')
print('Caracterização calculada para',len(corpus),'notícias. Nenhum registro foi excluído.')
print('Situação de autoria:')
print(corpus['author_status'].value_counts().to_string())
print('Comparação com mesma duração de calendário:')
print(matched_counts.to_string(index=False))


## 12. Visualizações científicas

Cada figura possui pergunta analítica e legenda registradas no catálogo de figuras. Os arquivos são exportados em SVG, PNG e PDF. As imagens podem ser consultadas na seção 14 e no relatório de caracterização. Os gráficos não removem registros dos cálculos.


In [ ]:
from matplotlib.ticker import MaxNLocator

CAT_LABELS = {
    'ECONOMIA':'Economia','TECNOLOGIA_INOVACAO':'Tecnologia e inovação','REGIONAL_LOCAL':'Regional e local',
    'OPINIAO_COLUNAS_BLOGS':'Opinião, colunas e blogs','CULTURA_ENTRETENIMENTO':'Cultura e entretenimento',
    'INTERNACIONAL':'Internacional','POLITICA_PODER':'Política e poder','NEGOCIOS_EMPREENDEDORISMO':'Negócios e empreendedorismo',
    'NAO_INFORMADA':'Não informada','SAUDE':'Saúde','COTIDIANO_SOCIEDADE':'Cotidiano e sociedade',
    'EDUCACAO':'Educação','CHECAGEM_DESINFORMACAO':'Checagem e desinformação','CIENCIA':'Ciência','ESPORTES':'Esportes',
    'EMPRESAS_PUBLICIDADE':'Empresas e publicidade','MEIO_AMBIENTE_AGRO':'Meio ambiente e agro',
    'TRABALHO_CARREIRAS':'Trabalho e carreiras','EVENTOS_ESPECIAIS':'Eventos especiais',
    'AUTO_MOBILIDADE_IMOVEIS':'Auto, mobilidade e imóveis','REDES_SOCIAIS':'Redes sociais','TURISMO_VIAGEM':'Turismo e viagem',
    'SEM_MAPEAMENTO':'Sem mapeamento'}
corpus_figure_specs=[]

def corpus_save_figure(fig,name,question,caption):
    # Mesma figura em formatos vetoriais e raster, com legenda descritiva separada.
    for ext in ['svg','png','pdf']:
        target=CORPUS_FIGURES/f'{name}.{ext}'
        fig.savefig(target,dpi=180,bbox_inches='tight')
        import shutil
        shutil.copy2(target,CORPUS_RUN/target.name)
        corpus_outputs.append(target)
    plt.close(fig)
    corpus_figure_specs.append({'figure':name,'analytical_question':question,'caption':caption})

def clean_axis(ax):
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y',alpha=.18)

# 1. Volumes anuais e mensais; anos anteriores à primeira observação não recebem zeros artificiais.
fig,axes=plt.subplots(2,1,figsize=(11,8),layout='constrained')
for source in CORPUS_SOURCES:
    g=annual.loc[annual['source'].eq(source)]
    axes[0].plot(g['year'],g['news'],marker='.',label=source,color=CORPUS_COLORS[source],linewidth=1.6)
    q=monthly.loc[monthly['source'].eq(source) & monthly['year'].ge(2023)].copy()
    q['date']=pd.to_datetime(q['year_month']+'-01')
    axes[1].plot(q['date'],q['news'],label=source,color=CORPUS_COLORS[source],linewidth=1.5)
axes[0].set(title='Notícias no corpus por ano e fonte',xlabel='Ano de publicação',ylabel='Notícias')
axes[0].axvspan(2025.5,2026.5,color='#999999',alpha=.14)
axes[0].text(.98,.94,'2026: janeiro a junho',transform=axes[0].transAxes,ha='right',va='top',fontsize=9,
    bbox={'facecolor':'white','edgecolor':'none','alpha':.9})
axes[0].legend(frameon=False,loc='upper left')
axes[1].set(title='Distribuição mensal desde 2023',xlabel='Mês de publicação',ylabel='Notícias')
for ax in axes:clean_axis(ax)
corpus_save_figure(fig,'30_corpus_volume_publicacoes','Como o volume observado varia por período e fonte?',
    'Contagens do corpus recebido. O ano de 2026 inclui somente janeiro a junho. Primeira observação não equivale à criação do veículo; volumes não representam toda sua produção editorial.')

# 2. Composição editorial de cada fonte: denominador inclui categorias não informadas.
order=corpus['category_grouped'].value_counts().index.tolist()
matrix=category_source.pivot(index='category_grouped',columns='source',values='share_all_news_pct').reindex(index=order,columns=CORPUS_SOURCES).fillna(0)
fig,ax=plt.subplots(figsize=(8.5,9),layout='constrained')
im=ax.imshow(matrix.to_numpy(),aspect='auto',cmap='Blues',vmin=0)
ax.set_xticks(range(3),CORPUS_SOURCES)
ax.set_yticks(range(len(order)),[CAT_LABELS.get(c,c) for c in order])
ax.set_title('Composição editorial por fonte no corpus completo',pad=15)
for i in range(len(order)):
    for j in range(3):
        val=matrix.iloc[i,j]
        ax.text(j,i,f'{val:.1f}%',ha='center',va='center',fontsize=9,color='white' if val>matrix.max().max()*.55 else '#202020')
fig.colorbar(im,ax=ax,label='Participação nas notícias da fonte (%)',shrink=.65)
corpus_save_figure(fig,'31_corpus_composicao_editorial','Quais editorias estão mais representadas em cada fonte?',
    'Percentuais dentro de cada fonte, incluindo categorias não informadas. A comparação do corpus completo também reflete coberturas temporais distintas. As categorias agrupam temas, formatos e origens editoriais.')

# 3. Fonte fixa para a trajetória longa, sem misturar mudanças da composição das fontes.
folha=category_year.loc[category_year['source'].eq('Folha de S.Paulo') & category_year['year'].le(2025)]
years=sorted(folha['year'].unique())
cat_order=corpus.loc[corpus['source'].eq('Folha de S.Paulo'),'category_grouped'].value_counts().index.tolist()
mat=folha.pivot(index='category_grouped',columns='year',values='share_all_news_pct').reindex(index=cat_order,columns=years).fillna(0)
fig,ax=plt.subplots(figsize=(13,8.5),layout='constrained')
im=ax.imshow(mat.to_numpy(),aspect='auto',cmap='YlGnBu',vmin=0)
ax.set_yticks(range(len(cat_order)),[CAT_LABELS.get(c,c) for c in cat_order])
positions=[i for i,y in enumerate(years) if i%3==0 or i==len(years)-1]
ax.set_xticks(positions,[str(years[i]) for i in positions],rotation=45,ha='right')
ax.set(xlabel='Ano de publicação',title='Composição editorial anual da Folha de S.Paulo, 1994–2025')
fig.colorbar(im,ax=ax,label='Participação nas notícias do ano (%)',shrink=.7)
corpus_save_figure(fig,'32_corpus_editorias_folha_temporal','Como a composição editorial muda mantendo a fonte fixa?',
    'Percentuais por ano na Folha. Células sem ocorrências recebem zero somente em anos observados. Volumes históricos reduzidos e mudanças nas editorias limitam a comparação; a figura não mede diretamente enquadramentos temáticos.')

# 4. Completude de autoria: marcadores textuais de ausência são separados de campos vazios.
statuses=['ASSINATURA_PRESENTE','NAO_IDENTIFICADA','CAMPO_VAZIO']
status_labels={'ASSINATURA_PRESENTE':'Assinatura presente','NAO_IDENTIFICADA':'Marcador de autoria não identificada','CAMPO_VAZIO':'Campo vazio'}
status_colors={'ASSINATURA_PRESENTE':'#777777','NAO_IDENTIFICADA':'#BBBBBB','CAMPO_VAZIO':'#E6E6E6'}
fig,axes=plt.subplots(2,1,figsize=(11,7.5),layout='constrained')
left=np.zeros(3)
for status in statuses:
    vals=author_coverage.loc[author_coverage['author_status'].eq(status)].set_index('source')['share_pct'].reindex(CORPUS_SOURCES,fill_value=0).to_numpy()
    axes[0].barh(CORPUS_SOURCES,vals,left=left,color=status_colors[status],label=status_labels[status])
    for k,v in enumerate(vals):
        if v>=4:axes[0].text(left[k]+v/2,k,f'{v:.1f}%',ha='center',va='center',fontsize=9)
    left+=vals
axes[0].set(xlabel='Notícias da fonte (%)',xlim=(0,100),title='Preenchimento do campo de autoria no corpus')
axes[0].legend(frameon=False,loc='upper center',bbox_to_anchor=(.5,-.18),ncol=1,fontsize=9)
missing=author_year.loc[~author_year['author_status'].eq('ASSINATURA_PRESENTE')].groupby(['source','year'])['share_pct'].sum()
for source in CORPUS_SOURCES:
    yy=annual.loc[annual['source'].eq(source),'year'].astype(int).to_list()
    vv=[missing.get((source,y),0) for y in yy]
    axes[1].plot(yy,vv,label=source,color=CORPUS_COLORS[source])
axes[1].set(xlabel='Ano de publicação',ylabel='Autoria ausente ou não identificada (%)',ylim=(0,105),title='Limitação da cobertura de autoria ao longo do tempo')
axes[1].legend(frameon=False)
for ax in axes:clean_axis(ax)
corpus_save_figure(fig,'33_corpus_cobertura_autoria','Quanto do corpus permite examinar assinaturas registradas?',
    'Assinatura presente pode designar pessoas, coautorias, agências ou redações. Campos vazios e marcadores como Sem autor são excluídos dos rankings e das métricas de concentração.')

# 5. Rankings de assinaturas, com recorte comum e empates preservados no arquivo de dados.
fig,axes=plt.subplots(3,1,figsize=(12,13.5),layout='constrained')
for ax,source in zip(axes,CORPUS_SOURCES):
    top=author_rank.loc[author_rank['source'].eq(source) & author_rank['period'].eq('2025')].sort_values(['news','author_key'],ascending=[False,True]).head(8).iloc[::-1]
    labels=['\n'.join(textwrap.wrap(v,width=64)) for v in top['author_signature']]
    ax.barh(range(len(top)),top['news'],color=CORPUS_COLORS[source])
    ax.set_yticks(range(len(top)),labels,fontsize=9)
    for y,v in enumerate(top['news']):ax.text(v+max(top['news'])*.015,y,str(v),va='center',fontsize=9)
    ax.set(xlabel='Notícias com a assinatura registrada',title=source,xlim=(0,max(top['news'])*1.18))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    clean_axis(ax)
fig.suptitle('Assinaturas mais frequentes em 2025, por fonte',fontsize=14)
corpus_save_figure(fig,'34_corpus_rankings_assinaturas_2025','Quais assinaturas registradas são mais recorrentes em 2025?',
    'Oito unidades de assinatura por fonte. Prefixo Por, espaços e capitalização são normalizados; vírgulas e sufixos de afiliação são preservados. Unidades não correspondem necessariamente a indivíduos distintos.')

# 6. Todos os valores da distribuição entram na ECDF; a caixa resume o recorte de 2025.
fig,axes=plt.subplots(1,2,figsize=(12,4.5),layout='constrained')
for source in CORPUS_SOURCES:
    v=np.sort(corpus.loc[corpus['source'].eq(source),'text_length_words'].to_numpy(dtype=float))
    axes[0].step(v,np.arange(1,len(v)+1)/len(v),where='post',label=source,color=CORPUS_COLORS[source])
axes[0].set(xscale='log',xlabel='Palavras por texto (escala logarítmica)',ylabel='Fração acumulada',title='Distribuição dos textos no corpus completo',ylim=(0,1.02))
axes[0].legend(frameon=False,fontsize=9)
arrays=[corpus.loc[corpus['source'].eq(s) & corpus['year'].eq(2025),'text_length_words'].to_numpy(dtype=float) for s in CORPUS_SOURCES]
box=axes[1].boxplot(arrays,tick_labels=['Folha','G1','CNN'],patch_artist=True,showfliers=False)
for patch,source in zip(box['boxes'],CORPUS_SOURCES):patch.set_facecolor(CORPUS_COLORS[source]);patch.set_alpha(.65)
axes[1].set(yscale='log',ylabel='Palavras por texto (escala logarítmica)',title='Comparação em 2025')
for ax in axes:clean_axis(ax)
corpus_save_figure(fig,'35_corpus_tamanho_textos','Como o tamanho dos textos difere entre fontes?',
    'ECDF do corpus completo e caixas de 2025. Caixas mostram quartis e mediana, com limites de 1,5 IQR; pontos externos não são exibidos nas caixas, mas permanecem nos cálculos e na ECDF. Os textos não foram truncados.')

# 7. A diversidade é apresentada junto ao número de notícias que sustenta cada estimativa.
fig,axes=plt.subplots(2,1,figsize=(11,7),sharex=True,layout='constrained')
for source in CORPUS_SOURCES:
    g=diversity.loc[diversity['source'].eq(source) & diversity['year'].le(2025)]
    axes[0].plot(g['year'],g['effective_categories'],label=source,color=CORPUS_COLORS[source])
    axes[1].plot(g['year'],g['known_category_news'],label=source,color=CORPUS_COLORS[source])
axes[0].set(ylabel='Número efetivo de macrocategorias',title='Distribuição editorial e volume disponível, até 2025')
axes[0].legend(frameon=False)
axes[1].set(xlabel='Ano de publicação',ylabel='Notícias com categoria conhecida',yscale='log')
for ax in axes:clean_axis(ax)
corpus_save_figure(fig,'36_corpus_diversidade_editorial','Como a distribuição editorial varia diante da cobertura disponível?',
    'Número efetivo de categorias exp(H), com H de Shannon calculado somente entre categorias conhecidas. O volume de notícias é exibido para contextualizar a estimativa. Diferenças de amostragem e disponibilidade editorial impedem atribuir a mudança automaticamente a maior diversidade temática.')

# 8. A comparação de 2026 usa uma janela de mesma duração, em vez do total de 2025.
match=matched_counts.set_index('source').reindex(CORPUS_SOURCES)
fig,ax=plt.subplots(figsize=(9,4.5),layout='constrained')
x=np.arange(3)
for shift,col,label,alpha,hatch in [(-.2,'news_2025_jan_jun','Jan–jun de 2025',.45,'///'),(.2,'news_2026_jan_jun','Jan–jun de 2026',1.0,'')]:
    bars=ax.bar(x+shift,match[col],width=.36,color=[CORPUS_COLORS[source] for source in CORPUS_SOURCES],
        alpha=alpha,hatch=hatch,edgecolor='#555555',linewidth=.4)
    ax.bar_label(bars,padding=3,fontsize=9)
from matplotlib.patches import Patch
period_legend=[Patch(facecolor='#A0A0A0',alpha=.45,hatch='///',edgecolor='#555555',label='Jan–jun de 2025'),
               Patch(facecolor='#A0A0A0',edgecolor='#555555',label='Jan–jun de 2026')]
ax.set_xticks(x,CORPUS_SOURCES)
ax.set(ylabel='Notícias no corpus',title='Volumes em janelas equivalentes de calendário')
ax.set_ylim(0,match[['news_2025_jan_jun','news_2026_jan_jun']].max().max()*1.18)
ax.legend(handles=period_legend,frameon=False);clean_axis(ax)
corpus_save_figure(fig,'37_corpus_janelas_equivalentes','Como os volumes se comparam em janeiro–junho de 2025 e 2026?',
    'Contagens entre 1 de janeiro e 30 de junho dos dois anos. Igual duração de calendário não garante mesma completude de coleta nem equivalência editorial. As diferenças são descritivas do arquivo. As cores identificam fontes; transparência e hachura distinguem os períodos.')

corpus_save_table(pd.DataFrame(corpus_figure_specs),'50_corpus_catalogo_figuras.csv')
print('Oito figuras exportadas em SVG, PNG e PDF, com perguntas analíticas e legendas.')


## 13. Verificações e relatório da caracterização

Conferimos contagens, denominadores, limites das métricas, tratamento de autoria e preservação da base preparada. O manifesto registra regras, versão da análise, dependências, hash da entrada, hash do código e hashes das tabelas e figuras. O relatório apresenta os resultados com limites de interpretação.


In [ ]:
corpus_checks={
    'contagem_fontes_reconciliada':int(summary_sources['news'].sum())==len(corpus),
    'contagem_anual_reconciliada':int(annual['news'].sum())==len(corpus),
    'contagem_mensal_reconciliada':int(monthly['news'].sum())==len(corpus),
    'categorias_fonte_reconciliadas':int(category_source['news'].sum())==len(corpus),
    'categorias_ano_reconciliadas':int(category_year['news'].sum())==len(corpus),
    'categorias_mes_reconciliadas':int(category_month['news'].sum())==len(corpus),
    'participacoes_categorias_somam_100':np.allclose(category_year.groupby(['source','year'])['share_all_news_pct'].sum(),100),
    'autoria_particiona_corpus':int(author_coverage['news'].sum())==len(corpus),
    'autoria_anual_particiona_corpus':int(author_year['news'].sum())==len(corpus),
    'rankings_excluem_autoria_ausente':not author_rank['author_key'].isin(CORPUS_MISSING_AUTHORS | {''}).any(),
    'assinaturas_anualmente_reconciliadas':int(author_by_year['news'].sum())==len(signed),
    'prefixo_por_normalizado':corpus_author_signature('Por Redação g1')=='Redação g1',
    'virgula_na_assinatura_preservada':corpus_author_signature('Por João Silva, da CNN')=='João Silva, da CNN',
    'marcador_sem_autor_nao_e_assinatura':corpus_author_status('Sem autor')=='NAO_IDENTIFICADA',
    'vazio_separado_de_marcador':corpus_author_status('  '.strip())=='CAMPO_VAZIO',
    'hhi_dentro_limites':author_concentration['hhi_signatures'].between(0,1+1e-12).all(),
    'hhi_limite_uniforme':(author_concentration['hhi_signatures']+1e-12>=1/author_concentration['signature_units']).all(),
    'concentracao_top10_maior_igual_top5':(author_concentration['top_10_share_signed_pct']+1e-10>=author_concentration['top_5_share_signed_pct']).all(),
    'concentracao_top10_ate100':author_concentration['top_10_share_signed_pct'].le(100+1e-10).all(),
    'continuidade_anos_limitada_exposicao':continuity['active_observed_years'].le(continuity['available_years_since_first']).all(),
    'categorias_efetivas_limite_superior':(diversity.loc[diversity['known_category_news'].gt(0),'effective_categories']<=diversity.loc[diversity['known_category_news'].gt(0),'observed_macro_categories']+1e-10).all(),
    'categorias_efetivas_minimo_um':diversity.loc[diversity['known_category_news'].gt(0),'effective_categories'].ge(1-1e-10).all(),
    'anos_sem_categoria_diversidade_indefinida':diversity.loc[diversity['known_category_news'].eq(0),['shannon_nats','effective_categories']].isna().all().all(),
    'janelas_mesmo_calendario':all(matched['month'].between(1,6)) and set(matched['year'])=={2025,2026},
    'janelas_contagens_reconciliadas':int(matched_counts[['news_2025_jan_jun','news_2026_jan_jun']].sum().sum())==len(matched),
    'corpus_nao_altera_base_preparada':corpus_file_hash(CORPUS_INPUT)==CORPUS_INPUT_HASH,
}
corpus_checks={k:bool(v) for k,v in corpus_checks.items()}
assert all(corpus_checks.values()),corpus_checks
corpus_save_table(pd.DataFrame(corpus_checks.items(),columns=['verification','consistent']),'51_corpus_verificacoes.csv')
notebook_json=json.loads((CORPUS_ROOT/'Pipeline_Noticias_IA.ipynb').read_text(encoding='utf-8'))
source_payload=[{'tipo':c['cell_type'],'fonte':''.join(c['source'])} for c in notebook_json['cells']]
source_hash=hashlib.sha256(json.dumps(source_payload,ensure_ascii=False,sort_keys=True).encode('utf-8')).hexdigest()
corpus_metadata={'analysis':'caracterizacao descritiva do corpus','analysis_version':CORPUS_ANALYSIS_VERSION,
    'executed_at':datetime.now(ZoneInfo('America/Sao_Paulo')).isoformat(),'input':str(CORPUS_INPUT.relative_to(CORPUS_ROOT)),
    'input_sha256':CORPUS_INPUT_HASH,'notebook_source_sha256':source_hash,'records':len(corpus),'records_excluded':0,
    'dependencies':{'python':platform.python_version(),'pandas':pd.__version__,'numpy':np.__version__,'matplotlib':matplotlib.__version__},
    'configuration':{'year_compare':CORPUS_YEAR_COMPARE,'matched_window':'January 1 to June 30, 2025 and 2026',
        'sources':CORPUS_SOURCES,'missing_categories':CORPUS_MISSING_CATEGORIES,'missing_author_markers':sorted(CORPUS_MISSING_AUTHORS),
        'author_normalization':'NFC; collapse whitespace; remove initial Por; casefold key within source; no comma splitting or person resolution',
        'diversity':'Shannon in nats and exp(H) among known categories; missingness separately reported',
        'concentration':'HHI and top shares among present signature units within each source and year'},
    'checks':corpus_checks,'figure_catalog':corpus_figure_specs,
    'outputs':[str(f.relative_to(CORPUS_ROOT)) for f in corpus_outputs],
    'output_sha256':{str(f.relative_to(CORPUS_ROOT)):corpus_file_hash(f) for f in corpus_outputs}}
(CORPUS_RUN/'manifesto_corpus.json').write_text(json.dumps(corpus_metadata,ensure_ascii=False,indent=2),encoding='utf-8')

def corpus_md_table(table):
    def fmt(v):
        if pd.isna(v):return ''
        if isinstance(v,(float,np.floating)):return f'{v:.2f}'
        return str(v).replace('|','/').replace('\n',' ')
    return '\n'.join(['| '+' | '.join(map(str,table.columns))+' |','| '+' | '.join(['---']*len(table.columns))+' |']+
        ['| '+' | '.join(fmt(v) for v in row)+' |' for row in table.itertuples(index=False,name=None)])

top_categories=corpus['category_grouped'].value_counts().rename_axis('category_grouped').reset_index(name='news')
top_categories['share_pct']=100*top_categories['news']/len(corpus)
by_author_status=corpus['author_status'].value_counts().rename_axis('author_status').reset_index(name='news')
by_author_status['share_pct']=100*by_author_status['news']/len(corpus)
summary2025=text_summary.loc[text_summary['period'].eq('2025') & text_summary['metric'].eq('text_length_words'),
    ['source','news','median','mean','q25','q75']]
top_by_source=author_rank.loc[author_rank['period'].eq('2025') & author_rank['rank'].le(5),
    ['source','author_signature','news','share_signed_news_pct','rank']]
yearly2025=annual.loc[annual['year'].eq(2025),['source','news','observed_months']]
effective2025=diversity.loc[diversity['year'].eq(2025),['source','news','category_missing_pct','observed_macro_categories','effective_categories']]
report=f'''# Caracterização do corpus de notícias sobre inteligência artificial

## Escopo e unidade de análise

A análise descreve os {len(corpus):,} registros da base preparada. Todos foram mantidos, inclusive os candidatos a duplicidade e os textos curtos sinalizados anteriormente. Os identificadores de notícia são únicos nesta versão. As estatísticas descrevem o corpus coletado, não toda a produção dos veículos, a atenção pública ou a importância social da IA.

Base SHA-256: `{CORPUS_INPUT_HASH}`. Versão da análise: `{CORPUS_ANALYSIS_VERSION}`.

## Fontes e períodos

{corpus_md_table(summary_sources)}

Em 2025:

{corpus_md_table(yearly2025)}

Há notícias nos doze meses de 2025 para as três fontes. Isso permite uma comparação em ano comum, mas não prova igual completude da coleta. As fontes têm início de cobertura observado diferente. A série de 2026 termina em 30 de junho; sua contagem anual não deve ser interpretada como retração em relação a 2025. Crescimentos de volume podem refletir mudanças de coleta e acesso, além de publicação editorial.

### Janelas com a mesma duração

{corpus_md_table(matched_counts)}

A comparação considera janeiro a junho dos dois anos. A duração equivalente evita comparar diretamente um semestre com um ano inteiro. Persistem diferenças de disponibilidade, seleção e composição editorial. A tabela de variação anual inclui a identificação do ano anterior observado, lacunas e a sinalização de 2026 parcial; seus percentuais não são estimativas causais.

## Macrocategorias editoriais

{corpus_md_table(top_categories)}

As categorias foram agrupadas pelo dicionário e incluem temas, formatos e origens, como opinião e publicidade. O termo diversidade editorial refere-se a essa distribuição, não à variedade de argumentos ou enquadramentos semânticos. As participações por fonte, ano e mês usam todas as notícias da respectiva combinação no denominador, incluindo a ausência de categoria.

A trajetória longa da Folha é apresentada separadamente para reduzir a mistura entre mudanças editoriais e entrada de outras fontes. Anos históricos com poucos registros e mudanças nas editorias exigem cautela.

### Distribuição e diversidade em 2025

{corpus_md_table(effective2025)}

Calculamos H = −Σ pᵢ ln(pᵢ), em nats, entre categorias conhecidas. O número efetivo exp(H) é o número de categorias igualmente frequentes que produziria a mesma entropia. Essa medida combina quantidade e equilíbrio das categorias. Não corrige automaticamente o efeito de amostragem: com mais notícias há mais oportunidades de observar categorias. A proporção de categorias ausentes e o volume por ano são apresentados junto à medida. Em anos sem nenhuma categoria conhecida, H e exp(H) ficam indefinidos, nunca iguais a zero.

## Autoria registrada

{corpus_md_table(by_author_status)}

O campo não contém apenas nomes de pessoas: há redações, agências, coautorias e sufixos de veículo ou localidade. Por isso, a unidade analisada é a assinatura registrada. Retiramos apenas o prefixo inicial Por, normalizamos espaços e comparamos capitalização dentro de cada fonte. Não dividimos vírgulas nem resolvemos identidades pessoais. As variantes permanecem na tabela de mapeamento.

Campos vazios e marcadores como Sem autor são categorias distintas de ausência e não entram nos rankings ou na concentração. A quantidade de assinaturas não deve ser apresentada como número de jornalistas. Diferenças entre fontes também refletem seus padrões de preenchimento.

### Assinaturas recorrentes em 2025

{corpus_md_table(top_by_source)}

Os percentuais usam somente as notícias com assinatura presente na fonte. Os rankings preservam empates no arquivo CSV. O gráfico limita a oito unidades para legibilidade, com desempate alfabético operacional, sem atribuir superioridade a assinaturas empatadas.

### Concentração e continuidade

O índice HHI = Σ pᵢ² mede a concentração das notícias entre unidades de assinatura, usando apenas assinaturas presentes. Também são exportadas as participações das cinco e dez unidades mais frequentes e o número efetivo 1/HHI. Esses indicadores não medem concentração individual de trabalho ou especialização em IA.

A continuidade registra primeiro e último ano, número de anos com ocorrência e fração dos anos observados da fonte desde a primeira ocorrência da assinatura. Inclui 2026 parcial; intervalos não têm exposições equivalentes. Uma mesma pessoa com assinaturas diferentes pode ser contada em várias unidades, e uma unidade coletiva pode representar várias pessoas.

## Tamanho dos textos

Resumo de palavras por texto no recorte comum de 2025:

{corpus_md_table(summary2025)}

Palavras são itens separados por espaços, sem tokenização linguística. Caracteres incluem espaços e quebras de linha do texto original. As distribuições incluem todos os registros. Diferenças podem refletir gênero editorial, disponibilidade do corpo da notícia e seleção da coleta. Não equivalem automaticamente a maior profundidade jornalística.

## Figuras e interpretação
'''
for item in corpus_figure_specs:
    report+=f"\n### {item['analytical_question']}\n\n![{item['analytical_question']}](resultados/figuras/{item['figure']}.png)\n\n{item['caption']}\n"
report+=f'''
## Verificações e reprodução

{corpus_md_table(pd.DataFrame(corpus_checks.items(),columns=['verification','consistent']))}

Para reproduzir somente esta análise em uma nova sessão, execute a seção 10 do notebook e as seções 11 a 14. A base preparada não é regravada. Tabelas, figuras em SVG/PNG/PDF e manifesto da execução são exportados para as pastas correspondentes. A extração de entidades e relações e os cálculos de redes não fazem parte desta caracterização.
'''
(CORPUS_ROOT/'Caracterizacao_Corpus.md').write_text(report,encoding='utf-8')
(CORPUS_RUN/'Caracterizacao_Corpus.md').write_text(report,encoding='utf-8')
print('Verificações:',sum(corpus_checks.values()),'de',len(corpus_checks))
print('Medianas de palavras em 2025:')
print(summary2025.to_string(index=False))
print('Distribuição de autoria:')
print(by_author_status.to_string(index=False))
print('Relatório de caracterização salvo; base preparada preservada.')


## 14. Figuras da caracterização

As figuras abaixo são carregadas dos arquivos locais após a execução da seção 12. Consulte `Caracterizacao_Corpus.md` e `50_corpus_catalogo_figuras.csv` para as legendas e limitações completas.

### Volume de publicações

![Volume de publicações](resultados/figuras/30_corpus_volume_publicacoes.png)

### Composição editorial por fonte

![Composição editorial por fonte](resultados/figuras/31_corpus_composicao_editorial.png)

### Composição editorial anual da Folha

![Composição editorial anual da Folha](resultados/figuras/32_corpus_editorias_folha_temporal.png)

### Cobertura de autoria

![Cobertura de autoria](resultados/figuras/33_corpus_cobertura_autoria.png)

### Assinaturas em 2025

![Assinaturas em 2025](resultados/figuras/34_corpus_rankings_assinaturas_2025.png)

### Tamanho dos textos

![Tamanho dos textos](resultados/figuras/35_corpus_tamanho_textos.png)

### Diversidade e volume disponível

![Diversidade e volume disponível](resultados/figuras/36_corpus_diversidade_editorial.png)

### Janelas de janeiro a junho

![Janelas de janeiro a junho](resultados/figuras/37_corpus_janelas_equivalentes.png)


## 15. Dados NER consolidados

Um snapshot reúne menções e cobertura documental com proveniência. `AVAILABLE` indica resultado disponível; importações possuem completude desconhecida. `PARTIAL` preserva menções aproveitáveis e pendências. Nenhum desses estados mede precisão semântica.

Esta seção pode ser executada em um kernel novo, sem credenciais. Para examinar os dados fornecidos, execute apenas esta seção. As seções 16 a 20 implementam uma nova extração integral e não são necessárias para carregar o snapshot.


In [ ]:
from pathlib import Path
import pandas as pd
from ner_dados import carregar_consolidado

PASTA_NER=Path.cwd()/'dados/processados/ner/consolidado'
BASE_NER=Path.cwd()/'dados/processados/noticias_preparadas.csv'
if (PASTA_NER/'manifesto.json').is_file():
    registros_consolidados_ner,metadados_consolidados_ner=carregar_consolidado(
        PASTA_NER,BASE_NER if BASE_NER.is_file() else None)
    documentos_consolidados_ner=pd.DataFrame(registros_consolidados_ner)
    print('Disponibilidade:',metadados_consolidados_ner['availability'])
    print('Menções:',metadados_consolidados_ner['mentions_all'])
    print('Subconjunto PESSOA/ORGANIZACAO/LOCAL:',metadados_consolidados_ner['mentions_pol'])
else:
    print('Nenhum snapshot disponível. Execute a extração e a consolidação para produzi-lo.')

# Leitura em blocos para limitar o uso de memória:
# for bloco in pd.read_csv(PASTA_NER/'mencoes_todas.csv.gz',dtype=str,
#                          keep_default_na=False,chunksize=50000):
#     ...


## 16. Configuração da extração

O executor recebe a base preparada da seção 10 e monta uma fila integral, agrupando textos idênticos. O prompt literal é conferido por SHA-256. As dez categorias são extraídas juntas; o subconjunto PESSOA/ORGANIZACAO/LOCAL é derivado depois.

`EXECUTAR_NER=False` desativa chamadas. O checkpoint desta execução é independente dos snapshots consolidados. Habilite chamadas somente para uma extração que deseja realizar. O período, limite e paralelismo são configuráveis.


In [ ]:
# Esta seção prepara a execução; não abre conexão de rede.
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from contextlib import closing
import csv, gzip, hashlib, io, json, os, re, sqlite3, tempfile
from urllib.request import Request, urlopen
from urllib.error import HTTPError
import pandas as pd

# O executor parte de todas as notícias da base preparada.
MODO_ENTRADA_NER='BASE_COMPLETA'
if 'df_news' not in globals():
    raise RuntimeError('Execute a seção 10 para carregar a base preparada.')
EXEC_RAIZ=Path.cwd()
EXEC_PROMPT=(EXEC_RAIZ/'dados/originais/ner/prompt_ner.txt').read_bytes().decode('utf-8')
EXEC_CONTRATO={
    'provider':'google','model':'gemini-2.5-flash','temperature':0,'seed':42,
    'response_mime_type':'application/json',
    'prompt_sha256':'87a4d970256b1e1301d0f84f22055a554439175db2082ed3c0db27505df86a82'}
EXEC_PROMPT_SHA='87a4d970256b1e1301d0f84f22055a554439175db2082ed3c0db27505df86a82'
assert hashlib.sha256(EXEC_PROMPT.encode('utf-8')).hexdigest()==EXEC_PROMPT_SHA
EXEC_ROTULOS={'PESSOA','ORGANIZACAO','LOCAL','TEMPO','ACONTECIMENTO','OBRA','VALOR','COISA','ABSTRACCAO','OUTRO'}
EXEC_POL={'PESSOA','ORGANIZACAO','LOCAL'}
EXEC_ALIASES={'ORGANIZACION':'ORGANIZACAO'}
EXEC_DB=EXEC_RAIZ/'dados/processados/ner/execucao_integral/checkpoint.sqlite'
EXEC_SAIDA=EXEC_RAIZ/'resultados/execucao/ner_integral'
EXEC_TERMINAIS={'SUCCESS','SUCCESS_EMPTY'}
EXEC_REPETIVEIS={'API_ERROR','INVALID_RESPONSE','BLOCKED'}
EXEC_INCERTOS={'RUNNING','UNCERTAIN'}
EXEC_VERSOES_ACEITAS={EXEC_CONTRATO['model'],EXEC_CONTRATO['model']+'-001'}
ANO_MAXIMO_NER=2025  # None inclui todos os anos.

# A mudança desta chave é uma ação local explícita para habilitar chamadas pagas.
EXECUTAR_NER=False
MODO_NER='PILOTO'  # PILOTO ou FILA
LIMITE_CHAMADAS_NER=5
TIMEOUT_NER_SEGUNDOS=180
REPETIR_FALHAS_NER=False
REPETIR_INCERTAS_NER=False  # Pode repetir uma chamada já cobrada sem resposta recebida.
MAX_TENTATIVAS_POR_REQUISICAO=2
LIMITE_FALHAS_CONSECUTIVAS_NER=3
TRABALHADORES_NER=1  # Número de chamadas simultâneas, entre 1 e 5.

def exec_sha(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def exec_json(value):
    return json.dumps(value,ensure_ascii=False,sort_keys=True,separators=(',',':'))

def exec_agora():
    return datetime.now(timezone.utc).isoformat()

def exec_request_id(text,contract):
    return exec_sha(exec_json({**contract,'input_sha256':exec_sha(text)}))

def preparar_ner_integral(base,contract,prompt):
    documents=[];requests={}
    for row in base.itertuples(index=False):
        title=str(row.title).strip();body=str(row.full_text).strip()
        text=title+'\n\n'+body if title and body else title or body
        if not text:raise ValueError('Notícia sem título e sem corpo: '+row.news_id)
        rid=exec_request_id(text,contract);year=int(row.publication_year)
        if rid not in requests:
            requests[rid]={'request_id':rid,'input_sha256':exec_sha(text),
                'prompt_sha256':contract['prompt_sha256'],'input_text':text,
                'prompt_characters':len(prompt.replace('{text}',text)),
                'source':row.source_normalized,'year':year,'ids':[],'sources':set(),'years':set(),
                'provider':contract['provider'],'model':contract['model']}
        requests[rid]['ids'].append(row.news_id);requests[rid]['sources'].add(row.source_normalized)
        requests[rid]['years'].add(year)
        documents.append({'news_id':row.news_id,'news_version_id':row.news_version_id,
            'source':row.source_normalized,'link':row.link,'title':row.title,
            'publication_date':row.publication_date_normalized,'year':year,
            'ner_input_sha256':exec_sha(text),'request_id':rid,'status':'PENDING',
            'reason':'FULL_EXTRACTION','mentions_all':0,'mentions_pol':0})
    queue=[]
    for item in requests.values():
        queue.append({k:v for k,v in item.items() if k not in ['ids','sources','years']} | {
            'news_ids_json':json.dumps(item['ids']),'news_count':len(item['ids']),
            'sources_json':json.dumps(sorted(item['sources']),ensure_ascii=False),
            'years_json':json.dumps(sorted(item['years'])),
            'reasons_json':json.dumps(['FULL_EXTRACTION'])})
    return pd.DataFrame(queue),pd.DataFrame(documents)

def validar_fila_execucao(queue,docs,contract,prompt):
    """Reconfere o conteúdo após a leitura do CSV, antes de qualquer chamada."""
    if set(contract)!={'provider','model','temperature','seed','response_mime_type','prompt_sha256'}:
        raise ValueError('Contrato com parâmetros não implementados; nenhum parâmetro será ignorado silenciosamente.')
    if exec_sha(prompt)!=contract['prompt_sha256'] or exec_sha(prompt)!=EXEC_PROMPT_SHA:
        raise ValueError('O prompt difere do template literal de referência.')
    if contract.get('provider')!='google':
        raise ValueError('Este executor implementa somente a API Gemini Developer.')
    if not queue.request_id.is_unique or not docs.news_id.is_unique:
        raise ValueError('Identificadores repetidos na fila ou na tabela de notícias.')
    docmap=docs.set_index('news_id');linked=[]
    for row in queue.itertuples(index=False):
        if exec_sha(row.input_text)!=row.input_sha256 or exec_request_id(row.input_text,contract)!=row.request_id:
            raise ValueError('Texto ou contrato alterado na fila: '+row.request_id)
        if row.prompt_sha256!=contract['prompt_sha256']:
            raise ValueError('Hash do prompt divergente na fila.')
        ids=json.loads(row.news_ids_json)
        if not isinstance(ids,list) or not ids:
            raise ValueError('Requisição sem vínculos documentais válidos.')
        for nid in ids:
            if nid not in docmap.index or docmap.loc[nid,'request_id']!=row.request_id:
                raise ValueError('Vínculo documental incompatível com a fila.')
        linked.extend(ids)
    expected=set(docs.loc[docs.status.eq('PENDING'),'news_id'])
    if len(linked)!=len(set(linked)) or set(linked)!=expected:
        raise ValueError('A fila não corresponde exatamente às notícias pendentes.')
    return True

def selecionar_piloto_ner(queue):
    """Seleciona até cinco entradas, cobrindo fontes e quantis de tamanho."""
    columns=['request_id','source','year','input_chars','news_count','reasons_json']
    if queue.empty:return pd.DataFrame(columns=columns)
    work=queue.copy();work['_size']=work.input_text.str.len()
    selected=[];used=set()
    profiles=[(source,0.5) for source in sorted(work.source.unique())]
    profiles += [(None,0.1),(None,0.9)]
    for source,quantile in profiles:
        subset=work.loc[~work.request_id.isin(used)].copy()
        if source is not None:subset=subset.loc[subset.source.eq(source)].copy()
        if subset.empty:continue
        subset['_distance']=(subset._size-subset._size.quantile(quantile)).abs()
        chosen=subset.sort_values(['_distance','request_id']).iloc[0];used.add(chosen.request_id)
        selected.append({'request_id':chosen.request_id,'source':chosen.source,'year':chosen.year,
                         'input_chars':int(chosen._size),'news_count':int(chosen.news_count),
                         'reasons_json':chosen.reasons_json})
        if len(selected)==5:break
    return pd.DataFrame(selected,columns=columns)

def selecionar_periodo_ner(queue,docs,ano_maximo):
    if ano_maximo is None:return queue.copy()
    ids=set(docs.loc[pd.to_numeric(docs.year).le(ano_maximo),'request_id'])
    return queue.loc[queue.request_id.isin(ids)].copy()

fila_ner,documentos_ner=preparar_ner_integral(df_news,EXEC_CONTRATO,EXEC_PROMPT)
validar_fila_execucao(fila_ner,documentos_ner,EXEC_CONTRATO,EXEC_PROMPT)
fila_ner_escopo=selecionar_periodo_ner(fila_ner,documentos_ner,ANO_MAXIMO_NER)
piloto_ner=selecionar_piloto_ner(fila_ner_escopo)
(EXEC_RAIZ/'resultados/tabelas').mkdir(parents=True,exist_ok=True)
piloto_ner.to_csv(EXEC_RAIZ/'resultados/tabelas/68_ner_piloto_planejado.csv',index=False,encoding='utf-8-sig')
print('Prompt literal e fila conferidos. Chamadas habilitadas:',EXECUTAR_NER)
print('Requisições no período selecionado:',len(fila_ner_escopo),'— adiadas:',len(fila_ner)-len(fila_ner_escopo))
print(piloto_ner[['source','year','input_chars','news_count']].to_string(index=False))


## 17. Executor, validação e checkpoint

Cada tentativa registra identificador, estado, tempos, resposta e uso informado pela API. As respostas são persistidas antes da interpretação. Falhas isoladas permanecem registradas; erros de configuração, autenticação e quota suspendem novos envios. Respostas interrompidas conservam apenas objetos JSON completos, com estado parcial.

O alinhamento literal da superfície utiliza a primeira ocorrência livre. A compatibilidade de posição não demonstra correção semântica nem exaustividade. Transformações, duplicações e pendências são auditáveis.


In [ ]:
def recuperar_mencoes_literais(entities,text):
    """Alinha superfícies em ordem de resposta pela primeira ocorrência livre.

    A associação é heurística: índices literais não comprovam a identidade da
    ocorrência pretendida pelo modelo, a categoria ou a completude do NER.
    """
    validated=[];issues=[];used=[];seen=set();dups=0;adjusted=0
    for index,e in enumerate(entities):
        candidates=[]
        try:
            if not isinstance(e,dict) or not {'start','end','label','surface'}.issubset(e):raise ValueError('INVALID_ENTITY_SCHEMA')
            a,b=e['start'],e['end'];s=e['surface'];label=e['label']
            if type(a) is not int or type(b) is not int or not isinstance(s,str) or not s.strip():raise ValueError('INVALID_ENTITY_FIELDS')
            if not isinstance(label,str):raise ValueError('INVALID_LABEL_TYPE')
            normalized=EXEC_ALIASES.get(label,label)
            if normalized not in EXEC_ROTULOS:raise ValueError('UNEXPECTED_LABEL')
            clean=s.strip()
            matches=list(re.finditer(re.escape(clean),text))
            relaxed=False
            if not matches:
                matches=list(re.finditer(re.escape(clean.replace('\n',' ')),text,flags=re.IGNORECASE))
                relaxed=True
            candidates=[m.span() for m in matches]
            if not candidates:raise ValueError('SURFACE_ABSENT')
            free=[(x,y) for x,y in candidates if not any(not(y<=u or x>=v) for u,v in used)]
            if not free:
                # Registros excedentes não reutilizam uma posição já contabilizada.
                if all((normalized,x,y,text[x:y]) in seen for x,y in candidates):
                    dups+=1;continue
                raise ValueError('OVERLAPPING_OCCURRENCES')
            x,y=free[0];literal=text[x:y]
            seen.add((normalized,x,y,literal));used.append((x,y))
            action=('FIRST_UNUSED_RELAXED' if relaxed else 'FIRST_UNUSED_LITERAL')
            adjusted+=int((a,b,s,label)!=(x,y,literal,normalized))
            validated.append({'entity_index':index,'start':x,'end':y,'label':normalized,'surface':literal,
                'original_start':a,'original_end':b,'original_label':label,'original_surface':s,
                'alignment_action':action})
        except (ValueError,TypeError,KeyError) as error:
            issues.append({'entity_index':index,'reason':str(error),'original_entity':e,
                           'candidate_spans':[list(span) for span in candidates]})
    return validated,issues,dups,adjusted

def _json_sem_chaves_duplicadas(pairs):
    d={}
    for k,v in pairs:
        if k in d:raise ValueError('Chave JSON repetida: '+k)
        d[k]=v
    return d

def _ler_json_ner_basico(answer):
    """Lê uma única carga NER; não aceita uma sublista de um JSON quebrado."""
    errors=[]
    def failed(strategy,error):
        errors.append({'strategy':strategy,'error':str(error)[:250]})
    def accept(data,strategy):
        if isinstance(data,list):return {'entities':data,'strategy':strategy+'_list','errors':errors,'repairs':[]}
        if isinstance(data,dict) and isinstance(data.get('entities'),list):
            return {'entities':data['entities'],'strategy':strategy+'_dict_entities','errors':errors,'repairs':[]}
        failed(strategy,'JSON_NER_FORMAT: esperado lista ou objeto com entities do tipo lista.')
        return {'entities':None,'strategy':'failed','errors':errors,'repairs':[]}
    def failure():return {'entities':None,'strategy':'failed','errors':errors,'repairs':[]}
    text=answer.strip()
    if not text:
        failed('direct','EMPTY_RESPONSE');return failure()
    try:return accept(json.loads(text,object_pairs_hook=_json_sem_chaves_duplicadas),'direct')
    except json.JSONDecodeError as error:failed('direct',error)
    except ValueError as error:
        failed('direct',error);return failure()
    fences=list(re.finditer(r'```(?:json)?[ \t]*(?:\r?\n)?(.*?)```',text,re.DOTALL|re.IGNORECASE))
    if len(fences)>1:
        failed('fenced','MULTIPLE_FENCED_PAYLOADS');return failure()
    if fences:
        outside=text[:fences[0].start()]+text[fences[0].end():]
        if re.search(r'[\[{]',outside):
            failed('fenced','COMPETING_JSON_OUTSIDE_FENCE');return failure()
        candidate=fences[0].group(1).strip()
        try:return accept(json.loads(candidate,object_pairs_hook=_json_sem_chaves_duplicadas),'fenced')
        except json.JSONDecodeError as error:failed('fenced',error)
        except ValueError as error:
            failed('fenced',error);return failure()
        # Um bloco delimitado incompleto não é substituído por uma sublista interna.
        text=candidate
    start=re.search(r'[\[{]',text)
    if start is None:
        failed('embedded','NO_JSON_CONTAINER');return failure()
    try:
        data,end=json.JSONDecoder(object_pairs_hook=_json_sem_chaves_duplicadas).raw_decode(text,start.start())
    except (ValueError,json.JSONDecodeError) as error:
        failed('embedded',error);return failure()
    if re.search(r'[\[{]',text[end:]):
        failed('embedded','MULTIPLE_OR_AMBIGUOUS_JSON_CONTAINERS');return failure()
    return accept(data,'embedded')

def _fora_string_json(text,position):
    quoted=False;escaped=False
    for char in text[:position]:
        if quoted:
            if escaped:escaped=False
            elif char=='\\':escaped=True
            elif char=='"':quoted=False
        elif char=='"':quoted=True
    return not quoted

def interpretar_json_ner(answer,permitir_reparo=True):
    """Interpreta a resposta sem transformar falha em lista vazia válida.

    O único reparo de sintaxe admitido remove uma aspa excedente na chave
    surface, fora de strings e em posição de chave. Não altera seus valores.
    """
    result=_ler_json_ner_basico(answer)
    if result['entities'] is not None or not permitir_reparo:return result
    matches=list(re.finditer(r'(?m)^[ \t]*(?P<key>""surface")(?=[ \t]*:)',answer))
    if len(matches)!=1:return result
    position=matches[0].start('key')
    before=answer[:position].rstrip()
    if not before or before[-1] not in '{,' or not _fora_string_json(answer,position):return result
    # A correção só é aceita se a resposta completa passar novamente pelo parser.
    repaired=answer[:position]+answer[position+1:]
    recovered=_ler_json_ner_basico(repaired)
    if recovered['entities'] is None:
        result['errors'].append({'strategy':'repair_single_surface_key_quote','error':'REPAIR_NOT_ACCEPTED'})
        return result
    recovered['errors']=result['errors']+recovered['errors']
    recovered['repairs']=[{'rule':'single_extra_quote_surface_key','position':position,
                           'removed_character':'"','coordinate_system':'model_answer_zero_based_characters'}]
    recovered['strategy']='repair_surface_key_quote+'+recovered['strategy']
    return recovered

def ler_prefixo_interrompido_ner(answer):
    """Lê objetos fechados de uma lista interrompida, sem completar o JSON.

    Aceita lista direta, bloco delimitado ou objeto iniciado por entities.
    A coordenada final pertence à resposta do modelo, não ao texto da notícia.
    """
    start=re.match(r'\s*(?:```(?:json)?[ \t]*(?:\r?\n)?\s*)?(?:\{\s*"entities"\s*:\s*)?\[',answer,re.I)
    result={'entities':None,'strategy':'max_tokens_prefix','repairs':[],'errors':[],
            'last_complete_object_end':None,'suffix_chars':len(answer),'closed_array_seen':False}
    if start is None:
        result['errors'].append({'strategy':'max_tokens_prefix','error':'NO_SUPPORTED_ARRAY_PREFIX'})
        return result
    pos=start.end();last_end=pos;entities=[]
    decoder=json.JSONDecoder(object_pairs_hook=_json_sem_chaves_duplicadas)
    while pos<len(answer):
        while pos<len(answer) and answer[pos].isspace():pos+=1
        if pos>=len(answer):break
        if answer[pos]==']':result['closed_array_seen']=True;break
        if entities:
            if answer[pos]!=',':
                result['errors'].append({'strategy':'max_tokens_prefix','error':'EXPECTED_COMMA'});break
            pos+=1
            while pos<len(answer) and answer[pos].isspace():pos+=1
        try:item,end=decoder.raw_decode(answer,pos)
        except ValueError as error:
            result['errors'].append({'strategy':'max_tokens_prefix','error':str(error)[:250]});break
        if not isinstance(item,dict):
            result['errors'].append({'strategy':'max_tokens_prefix','error':'ENTITY_IS_NOT_OBJECT'});break
        entities.append(item);pos=end;last_end=end
    if entities:
        result.update(entities=entities,last_complete_object_end=last_end,suffix_chars=len(answer)-last_end)
    else:
        result['errors'].append({'strategy':'max_tokens_prefix','error':'NO_COMPLETE_ENTITY_OBJECT'})
    return result

def validar_resposta_ner(raw,http_status,text,accepted_models):
    """Valida o documento inteiro. Uma falha nunca se transforma em lista vazia."""
    result={'state':'INVALID_RESPONSE','entities':[],'model_version':'',
            'usage':{},'diagnostic':'','duplicates_removed':0,'adjusted':0,
            'unresolved':[],'validator_version':'ner-validation-5',
            'parser_version':'ner-json-2','alignment_version':'ner-first-unused-3',
            'parser_strategy':'not_run','parser_repairs':[],'parser_errors':[],
            'finish_reason':'','response_truncated':False,'truncation_review_required':False,
            'last_complete_object_end':None,'unprocessed_suffix_chars':None,'answer_chars':0}
    if http_status!=200:
        result.update(state='API_ERROR',diagnostic='HTTP '+str(http_status));return result
    try:
        outer=json.loads(raw.decode('utf-8'),object_pairs_hook=_json_sem_chaves_duplicadas)
        if not isinstance(outer,dict):raise ValueError('Envelope da API não é objeto.')
        version=outer.get('modelVersion','')
        result['model_version']=version if isinstance(version,str) else ''
        result['usage']=outer.get('usageMetadata',{}) or {}
        if not isinstance(result['usage'],dict):raise ValueError('usageMetadata inválido.')
        feedback=outer.get('promptFeedback',{}) or {}
        if feedback.get('blockReason') not in (None,'','BLOCK_REASON_UNSPECIFIED'):
            result.update(state='BLOCKED',diagnostic='Prompt bloqueado pelo provedor.');return result
        if result['model_version'] not in accepted_models:
            result.update(state='MODEL_MISMATCH',diagnostic='Modelo efetivo diferente do esperado ou ausente.');return result
        candidates=outer.get('candidates',[])
        if not isinstance(candidates,list) or len(candidates)!=1:
            raise ValueError('Esperado exatamente um candidato.')
        candidate=candidates[0];finish=candidate.get('finishReason','')
        result.update(finish_reason=finish,response_truncated=(finish=='MAX_TOKENS'),truncation_review_required=(finish=='MAX_TOKENS'))
        if finish not in {'STOP','MAX_TOKENS'}:
            if finish in {'SAFETY','BLOCKLIST','PROHIBITED_CONTENT','RECITATION','SPII'}:
                result.update(state='BLOCKED',diagnostic='finishReason='+finish);return result
            raise ValueError('Resposta não concluída: finishReason='+str(finish))
        parts=candidate.get('content',{}).get('parts',[])
        answer=''.join(p['text'] for p in parts if isinstance(p,dict) and
                       isinstance(p.get('text'),str) and not p.get('thought',False))
        result['answer_chars']=len(answer)
        parsed=ler_prefixo_interrompido_ner(answer) if finish=='MAX_TOKENS' else interpretar_json_ner(answer)
        if finish=='MAX_TOKENS':
            result.update(last_complete_object_end=parsed['last_complete_object_end'],unprocessed_suffix_chars=parsed['suffix_chars'])
        result.update(parser_strategy=parsed['strategy'],parser_repairs=parsed['repairs'],parser_errors=parsed['errors'])
        if parsed['entities'] is None:raise ValueError('Falha de parse: '+parsed['errors'][-1]['error'])
        entities=parsed['entities']
        validated,issues,dups,adjusted=recuperar_mencoes_literais(entities,text)
        result.update(state=('PARTIAL' if issues or finish=='MAX_TOKENS' else ('SUCCESS' if entities else 'SUCCESS_EMPTY')),
                      entities=validated,unresolved=issues,duplicates_removed=dups,adjusted=adjusted,
                      raw_entities_count=len(entities))
        result['diagnostic']=(f"{len(validated)} menções alinhadas; {len(issues)} pendências de alinhamento/esquema."
                              if entities else 'Lista vazia válida.')
        if finish=='MAX_TOKENS':result['diagnostic']='Resposta interrompida por MAX_TOKENS; aproveitamento parcial. '+result['diagnostic']

    except (ValueError,TypeError,KeyError,AttributeError,UnicodeDecodeError,RecursionError,OverflowError) as e:
        result.update(state='INVALID_RESPONSE',entities=[],diagnostic=str(e)[:500])
    return result

def abrir_checkpoint_ner(path):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    db=sqlite3.connect(path,timeout=30)
    db.row_factory=sqlite3.Row
    db.execute('PRAGMA foreign_keys=ON');db.execute('PRAGMA synchronous=FULL')
    db.executescript('''
    CREATE TABLE IF NOT EXISTS requests (
        request_id TEXT PRIMARY KEY,input_sha256 TEXT NOT NULL,prompt_sha256 TEXT NOT NULL,
        contract_json TEXT NOT NULL,state TEXT NOT NULL,attempts INTEGER NOT NULL DEFAULT 0,
        latest_attempt INTEGER,entities_json TEXT,model_version TEXT,diagnostic TEXT,
        validation_json TEXT,updated_at TEXT NOT NULL);
    CREATE TABLE IF NOT EXISTS attempts (
        attempt_id INTEGER PRIMARY KEY,request_id TEXT NOT NULL REFERENCES requests(request_id),
        started_at TEXT NOT NULL,finished_at TEXT,state TEXT NOT NULL,http_status INTEGER,
        raw_gzip BLOB,headers_json TEXT,usage_json TEXT,error_type TEXT);
    ''')
    return db

def registrar_fila_checkpoint(db,queue,contract):
    c=exec_json(contract)
    with db:
        for row in queue.itertuples(index=False):
            if exec_request_id(row.input_text,contract)!=row.request_id or exec_sha(row.input_text)!=row.input_sha256:
                raise ValueError('Fila alterada: '+row.request_id)
            old=db.execute('SELECT input_sha256,prompt_sha256,contract_json FROM requests WHERE request_id=?',(row.request_id,)).fetchone()
            if old and tuple(old)!=(row.input_sha256,contract['prompt_sha256'],c):
                raise ValueError('Contrato incompatível no checkpoint.')
            db.execute('INSERT OR IGNORE INTO requests(request_id,input_sha256,prompt_sha256,contract_json,state,updated_at) VALUES(?,?,?,?,?,?)',
                       (row.request_id,row.input_sha256,contract['prompt_sha256'],c,'PENDING',exec_agora()))

def reservar_requisicao(db,rid,retry_failed=False,retry_uncertain=False,max_attempts=2):
    db.execute('BEGIN IMMEDIATE')
    try:
        row=db.execute('SELECT * FROM requests WHERE request_id=?',(rid,)).fetchone()
        allowed={'PENDING'} | (EXEC_REPETIVEIS if retry_failed else set()) | (EXEC_INCERTOS if retry_uncertain else set())
        if not row or row['state'] not in allowed or row['attempts']>=max_attempts:
            db.rollback();return None
        attempt=db.execute('INSERT INTO attempts(request_id,started_at,state) VALUES(?,?,?)',(rid,exec_agora(),'RUNNING')).lastrowid
        db.execute('UPDATE requests SET state=?,attempts=attempts+1,latest_attempt=?,entities_json=NULL,updated_at=? WHERE request_id=?',
                   ('RUNNING',attempt,exec_agora(),rid))
        db.commit();return attempt
    except BaseException:
        db.rollback();raise

def salvar_resposta_bruta(db,rid,attempt,status,raw,headers):
    if not isinstance(raw,bytes):raise TypeError('A resposta HTTP precisa ser bytes.')
    safe={str(k):str(v) for k,v in headers.items() if str(k).lower() in
          {'date','retry-after','x-request-id','x-goog-request-id','content-type'}}
    with db:
        db.execute('UPDATE attempts SET finished_at=?,state=?,http_status=?,raw_gzip=?,headers_json=? WHERE attempt_id=?',
            (exec_agora(),'RAW_SAVED',int(status),gzip.compress(raw,mtime=0),exec_json(safe),attempt))
        db.execute('UPDATE requests SET state=?,updated_at=? WHERE request_id=? AND latest_attempt=?',
                   ('RAW_SAVED',exec_agora(),rid,attempt))

def finalizar_resposta_salva(db,rid,text,accepted_models):
    row=db.execute('SELECT r.latest_attempt,a.http_status,a.raw_gzip FROM requests r JOIN attempts a ON a.attempt_id=r.latest_attempt WHERE r.request_id=? AND r.state=?',
                   (rid,'RAW_SAVED')).fetchone()
    if row is None:return None
    result=validar_resposta_ner(gzip.decompress(row['raw_gzip']),row['http_status'],text,accepted_models)
    details={k:v for k,v in result.items() if k not in ['entities','usage']}
    with db:
        db.execute('UPDATE requests SET state=?,entities_json=?,model_version=?,diagnostic=?,validation_json=?,updated_at=? WHERE request_id=? AND latest_attempt=?',
            (result['state'],exec_json(result['entities']),result['model_version'],result['diagnostic'],exec_json(details),exec_agora(),rid,row['latest_attempt']))
        db.execute('UPDATE attempts SET state=?,usage_json=? WHERE attempt_id=?',
                   (result['state'],exec_json(result['usage']),row['latest_attempt']))
    return result

def marcar_transporte_incerto(db,rid,attempt,error_type):
    # Não registra cabeçalhos de autenticação, a chave nem a mensagem de um proxy.
    with db:
        db.execute('UPDATE attempts SET finished_at=?,state=?,error_type=? WHERE attempt_id=?',
                   (exec_agora(),'UNCERTAIN',error_type,attempt))
        db.execute('UPDATE requests SET state=?,diagnostic=?,updated_at=? WHERE request_id=? AND latest_attempt=?',
            ('UNCERTAIN','Transporte interrompido; não há resposta persistida.',exec_agora(),rid,attempt))

def transporte_gemini_rest(prompt,contract,api_key,timeout):
    if not re.fullmatch(r'[A-Za-z0-9._-]+',contract['model']):raise ValueError('Nome de modelo inválido.')
    payload={'contents':[{'role':'user','parts':[{'text':prompt}]}],
             'generationConfig':{'temperature':contract['temperature'],'seed':contract['seed'],
                                 'responseMimeType':contract['response_mime_type']}}
    request=Request('https://generativelanguage.googleapis.com/v1beta/models/'+contract['model']+':generateContent',
        data=json.dumps(payload,ensure_ascii=False).encode('utf-8'),
        headers={'Content-Type':'application/json','x-goog-api-key':api_key},method='POST')
    # urllib não faz retentativas de geração. Cada tentativa fica sob controle do executor.
    try:
        with urlopen(request,timeout=timeout) as response:
            return response.status,response.read(),dict(response.headers)
    except HTTPError as error:
        return error.code,error.read(),dict(error.headers)

def credenciais_ner_env(path):
    """Lê a chave e um apelido opcional; a chave nunca integra as mensagens."""
    from dotenv import dotenv_values
    path=Path(path)
    if not path.is_file():raise FileNotFoundError('Crie o .env na pasta do projeto, a partir de .env.example.')
    values=dotenv_values(path,interpolate=False,encoding='utf-8-sig')
    key=values.get('GEMINI_API_KEY')
    if not key or key.strip() in {'COLE_SUA_CHAVE_AQUI','sua_chave_aqui'}:
        raise ValueError('Defina GEMINI_API_KEY no .env local.')
    key=key.strip()
    name=values.get('KEY_NAME') or 'chave local'
    # Remove controles de terminal e limita o tamanho do apelido exibido.
    name=' '.join(re.sub(r'[^\w .@()/-]',' ',name).split())[:80] or 'chave local'
    if key in name or key in (values.get('KEY_NAME') or ''):name='chave local'
    return key,name

def chave_ner_env(path):
    return credenciais_ner_env(path)[0]

def politica_falha_ner(state,http_status=None):
    """Classifica interrupções sistêmicas e falhas individuais de uma tentativa."""
    if state=='MODEL_MISMATCH':return 'STOP','MODEL_MISMATCH'
    if state=='API_ERROR' and http_status in {400,401,402,403,404,405,415,429}:
        return 'STOP','HTTP_'+str(http_status)
    if state in {'API_ERROR','UNCERTAIN','INVALID_RESPONSE','BLOCKED'}:
        return 'COUNT_FAILURE',state
    if state in {'SUCCESS','SUCCESS_EMPTY','PARTIAL'}:return 'CONTINUE',''
    return 'STOP','UNEXPECTED_STATE'

def executar_fila_ner(queue,docs,contract,prompt,checkpoint,selected_ids=None,*,executar=False,
        max_calls=5,timeout=180,retry_failed=False,retry_uncertain=False,max_attempts=2,
        accepted_models=None,env_path=None,transport=None,workers=1,max_consecutive_failures=3):
    validar_fila_execucao(queue,docs,contract,prompt)
    if type(max_calls) is not int or max_calls<0 or type(max_attempts) is not int or max_attempts<1 or type(workers) is not int or not 1<=workers<=5:
        raise ValueError('Limites de chamadas/tentativas inválidos.')
    if type(max_consecutive_failures) is not int or max_consecutive_failures<1:
        raise ValueError('O limite de falhas consecutivas deve ser um inteiro positivo.')
    accepted_models=set(accepted_models or {contract['model'],contract['model']+'-001'})
    selected=set(queue.request_id if selected_ids is None else selected_ids)
    if not selected.issubset(set(queue.request_id)):raise ValueError('Seleção contém ID fora da fila.')
    if not executar:
        print(f'[NER] PRÉVIA — chamadas desativadas; {len(selected)} requisições selecionadas; limite do lote: {max_calls}. A chave não foi lida.',flush=True)
        return {'mode':'PREVIEW','selected_requests':len(selected),'max_calls':max_calls,'calls':0,
                'max_consecutive_failures':max_consecutive_failures}
    from time import perf_counter
    from threading import Lock
    console_lock=Lock()
    log_root=Path(checkpoint).parent/'logs'
    def log(message,rid=None,attempt=None):
        stamp=datetime.now(timezone.utc).isoformat(timespec='seconds')
        tag=f'T{attempt:06d} R{rid[:12]}' if rid is not None else 'NER'
        line=f'{stamp} [{tag}] {message}'
        with console_lock:
            print(line,flush=True)
            if rid is not None:
                folder=log_root/rid;folder.mkdir(parents=True,exist_ok=True)
                with (folder/f'tentativa_{attempt:06d}.log').open('a',encoding='utf-8') as handle:
                    handle.write(line+'\n')
    api_key=None;key_name='transporte de teste'
    if transport is None:
        api_key,key_name=credenciais_ner_env(env_path or Path.cwd()/'.env')
        transport=transporte_gemini_rest
    log(f'EXECUÇÃO — apelido: {key_name}; modelo: {contract["model"]}; limite: {max_calls}; trabalhadores: {workers}.')
    calls=0;recovered=0;states=Counter();consecutive_failures=0;halt_reason=None;halt_failure_streak=None
    with closing(abrir_checkpoint_ner(checkpoint)) as db:
        registrar_fila_checkpoint(db,queue,contract)
        # Uma resposta já recebida é revalidada localmente, sem pagar outra chamada.
        for row in queue.itertuples(index=False):
            existing=db.execute('SELECT state,model_version FROM requests WHERE request_id=?',(row.request_id,)).fetchone()
            if existing['state']=='MODEL_MISMATCH' and existing['model_version'] in accepted_models:
                with db:db.execute("UPDATE requests SET state='RAW_SAVED' WHERE request_id=?",(row.request_id,))
            result=finalizar_resposta_salva(db,row.request_id,row.input_text,accepted_models)
            if result is not None:recovered+=1
        mismatch=any(db.execute("SELECT state FROM requests WHERE request_id=?",(rid,)).fetchone()[0]=='MODEL_MISMATCH' for rid in queue.request_id)
        if mismatch:
            log('LOTE INTERROMPIDO — modelo efetivo incompatível; nenhuma nova chamada.')
            return {'mode':'EXECUTION','calls':0,'halt':'MODEL_MISMATCH','halt_reason':'MODEL_MISMATCH','recovered_without_call':recovered}
        from concurrent.futures import ThreadPoolExecutor,wait,FIRST_COMPLETED
        def perform(row,attempt):
            started=perf_counter()
            title=' '.join(re.sub(r'[\x00-\x1f\x7f-\x9f]',' ',row.input_text.split('\n',1)[0]).split())[:120]
            ids=json.loads(row.news_ids_json)
            log(f'NOTÍCIA — {row.source}; ano {row.year}; {title}',row.request_id,attempt)
            log(f'VÍNCULOS — request_id={row.request_id}; news_ids={exec_json(ids)}',row.request_id,attempt)
            log(f'ENVIO — aguardando resposta HTTP; limite de espera: {timeout}s.',row.request_id,attempt)
            with closing(abrir_checkpoint_ner(checkpoint)) as worker_db:
                try:
                    status,raw,headers=transport(prompt.replace('{text}',row.input_text),contract,api_key,timeout)
                    log(f'RETORNO — HTTP {status}; {perf_counter()-started:.1f}s.',row.request_id,attempt)
                    salvar_resposta_bruta(worker_db,row.request_id,attempt,status,raw,headers)
                except KeyboardInterrupt:
                    marcar_transporte_incerto(worker_db,row.request_id,attempt,'KeyboardInterrupt')
                    log('INTERROMPIDA — estado UNCERTAIN; conferir checkpoint antes de repetir.',row.request_id,attempt);raise
                except Exception as error:
                    marcar_transporte_incerto(worker_db,row.request_id,attempt,type(error).__name__)
                    log(f'SEM RESPOSTA PERSISTIDA — estado UNCERTAIN; {perf_counter()-started:.1f}s. Conferir antes de repetir.',row.request_id,attempt)
                    return {'state':'UNCERTAIN','entities':[],'completed_monotonic':perf_counter()}
                result=finalizar_resposta_salva(worker_db,row.request_id,row.input_text,accepted_models)
                log(f'PARSE — {result.get("parser_strategy","not_run")}; {len(result.get("parser_repairs",[]))} reparos; interrompida: {result.get("response_truncated",False)}.',row.request_id,attempt)
                log(f'VALIDAÇÃO — {result["state"]}; {len(result["entities"])} menções; {len(result.get("unresolved",[]))} pendências; resposta salva.',row.request_id,attempt)
                return {**result,'http_status':status,'completed_monotonic':perf_counter()}
        iterator=iter(row for row in queue.itertuples(index=False) if row.request_id in selected)
        halted=False;exhausted=False;active={}
        # No máximo workers chamadas em curso. Reservas são transações atômicas.
        with ThreadPoolExecutor(max_workers=workers) as pool:
            while active or (not exhausted and not halted and calls<max_calls):
                while not exhausted and not halted and calls<max_calls and len(active)<workers:
                    try:row=next(iterator)
                    except StopIteration:exhausted=True;break
                    attempt=reservar_requisicao(db,row.request_id,retry_failed,retry_uncertain,max_attempts)
                    if attempt is None:continue
                    calls+=1
                    active[pool.submit(perform,row,attempt)]=(row.request_id,attempt)
                if not active:break
                done,_=wait(active,return_when=FIRST_COMPLETED)
                finished=sorted(((future,future.result()) for future in done),key=lambda item:item[1]['completed_monotonic'])
                for future,result in finished:
                    rid,attempt=active.pop(future)
                    states[result['state']]+=1
                    completed=sum(states.values())
                    log(f'ENCERRADA — {result["state"]}; tentativas encerradas no lote: {completed}.',rid,attempt)
                    action,reason=politica_falha_ner(result['state'],result.get('http_status'))
                    if action=='COUNT_FAILURE':
                        consecutive_failures+=1
                        log(f'PENDÊNCIA PRESERVADA — {reason}; falhas consecutivas: {consecutive_failures}/{max_consecutive_failures}; sem reenvio automático.',rid,attempt)
                    elif action=='CONTINUE':
                        consecutive_failures=0
                    if not halted and (action=='STOP' or consecutive_failures>=max_consecutive_failures):
                        halted=True
                        halt_reason=reason if action=='STOP' else 'CONSECUTIVE_FAILURES'
                        halt_failure_streak=consecutive_failures
                        log(f'NOVOS ENVIOS INTERROMPIDOS — {halt_reason}; chamadas já iniciadas serão preservadas ao terminar.')
    if calls==0:log('NENHUMA NOVA CHAMADA — seleção sem requisições elegíveis ou limite igual a zero; confira o modo, o período e o progresso.')
    log(f'LOTE ENCERRADO — {calls} novas tentativas; {sum(states.values())} encerradas; {recovered} respostas recuperadas sem chamada.')
    return {'mode':'EXECUTION','calls':calls,'recovered_without_call':recovered,'states_this_run':dict(states),'halt_reason':halt_reason,
            'max_consecutive_failures':max_consecutive_failures,'failure_streak_at_halt':halt_failure_streak}

def consultar_progresso_ner(queue,checkpoint):
    status={}
    if Path(checkpoint).is_file():
        # Consulta somente leitura: não cria checkpoint ao executar o notebook desativado.
        uri=Path(checkpoint).resolve().as_uri()+'?mode=ro'
        with closing(sqlite3.connect(uri,uri=True)) as db:
            status=dict(db.execute('SELECT request_id,state FROM requests'))
    result=queue[['request_id','news_count']].copy()
    result['state']=result.request_id.map(status).fillna('PENDING')
    result['news_count']=pd.to_numeric(result.news_count)
    return result.groupby('state').agg(requests=('request_id','size'),news=('news_count','sum')).reset_index()

print('Executor e validador definidos. Nenhuma chamada efetuada.')


def revalidar_respostas_salvas_ner(queue,contract,prompt,checkpoint,accepted_models=None):
    """Somente disco: não acessa .env, transporte HTTP ou modelo."""
    path=Path(checkpoint)
    if not path.is_file():return {'revalidated':0,'api_calls':0,'backup':None}
    if exec_sha(prompt)!=contract['prompt_sha256']:raise ValueError('Prompt incompatível.')
    accepted_models=set(accepted_models or {contract['model'],contract['model']+'-001'})
    qmap=queue.set_index('request_id').to_dict('index');targets=[]
    with closing(abrir_checkpoint_ner(path)) as db:
        for row in db.execute('SELECT r.*,a.raw_gzip,a.http_status FROM requests r LEFT JOIN attempts a ON a.attempt_id=r.latest_attempt'):
            if row['request_id'] not in qmap or row['raw_gzip'] is None or row['http_status']!=200:continue
            info=json.loads(row['validation_json']) if row['validation_json'] else {}
            if row['state'] not in {'INVALID_RESPONSE','RAW_SAVED','PARTIAL','SUCCESS','SUCCESS_EMPTY'}:continue
            if info.get('validator_version')=='ner-validation-5' and row['state']!='RAW_SAVED':continue
            q=qmap[row['request_id']]
            if row['input_sha256']!=q['input_sha256'] or exec_sha(q['input_text'])!=row['input_sha256']:
                raise ValueError('Entrada diferente da resposta salva.')
            if row['contract_json']!=exec_json(contract):raise ValueError('Contrato diferente da resposta salva.')
            targets.append(row['request_id'])
        if not targets:return {'revalidated':0,'api_calls':0,'backup':None}
        backup=path.with_name('checkpoint.antes_revalidacao_'+datetime.now().strftime('%Y%m%d_%H%M%S_%f')+'.sqlite')
        with closing(sqlite3.connect(backup)) as destination:db.backup(destination)
        before=db.execute('SELECT count(*) FROM attempts').fetchone()[0]
        outcomes=Counter()
        for rid in targets:
            with db:db.execute("UPDATE requests SET state='RAW_SAVED' WHERE request_id=?",(rid,))
            result=finalizar_resposta_salva(db,rid,qmap[rid]['input_text'],accepted_models)
            outcomes[result['state']]+=1
        assert before==db.execute('SELECT count(*) FROM attempts').fetchone()[0]
    return {'revalidated':len(targets),'states':dict(outcomes),'api_calls':0,'backup':str(backup)}

def exportar_resultados_ner(queue,docs,contract,checkpoint,destino,base_mentions_path):
    """Exporta somente a fila corrente; nunca inclui .env ou credenciais."""
    import zipfile
    checkpoint=Path(checkpoint);destino=Path(destino)
    if not checkpoint.is_file():
        return None
    destino.mkdir(parents=True,exist_ok=True)
    qmap=queue.set_index('request_id').to_dict('index')
    dmap=docs.set_index('news_id').to_dict('index')
    columns=['mention_id','news_id','news_version_id','ner_input_sha256','source','link',
        'publication_date','start','end','ent_text','ent_label','model','origin',
        'source_mention_id','source_news_id','source_file','source_publication_date',
        'original_label','original_surface','original_start','original_end','alignment_action']
    if Path(base_mentions_path).is_file() and pd.read_csv(base_mentions_path,nrows=0).columns.tolist()!=columns:
        raise ValueError('Esquema de menções incompatível com a exportação.')
    truncation_rows=[];parse_rows=[];validation_rows=[];request_rows=[];usage_rows=[];new_mentions=[];partial_mentions=[];alignment_issues=[];new_doc_counts={};partial_doc_counts={}
    with closing(abrir_checkpoint_ner(checkpoint)) as db:
        # Snapshot consistente mesmo se outra instância estiver concluindo uma resposta.
        db.execute('BEGIN')
        for row in db.execute('SELECT * FROM requests ORDER BY request_id'):
            rid=row['request_id']
            if rid not in qmap:continue
            request_rows.append({k:row[k] for k in ['request_id','state','attempts','model_version','diagnostic','updated_at']})
            partial=row['state']=='PARTIAL'
            details=json.loads(row['validation_json']) if row['validation_json'] else {}
            if details.get('response_truncated'):
                truncation_rows.append({'request_id':rid,'news_ids_json':qmap[rid]['news_ids_json'],
                    'state':row['state'],'model_version':row['model_version'],'finish_reason':details.get('finish_reason',''),
                    'input_chars':len(qmap[rid]['input_text']),'answer_chars':details.get('answer_chars',0),
                    'complete_objects':details.get('raw_entities_count',0),
                    'aligned_mentions':len(json.loads(row['entities_json'] or '[]')),
                    'alignment_issues':len(details.get('unresolved',[])),
                    'last_complete_object_end':details.get('last_complete_object_end'),
                    'unprocessed_suffix_chars':details.get('unprocessed_suffix_chars'),
                    'requires_future_review':True,'automatic_retry':False})
            if row['attempts']>0:
                parse_rows.append({'request_id':rid,'state':row['state'],
                    'parser_version':details.get('parser_version',''),
                    'parser_strategy':details.get('parser_strategy','not_run'),
                    'repair_count':len(details.get('parser_repairs',[])),
                    'repairs_json':exec_json(details.get('parser_repairs',[])),
                    'parse_errors_json':exec_json(details.get('parser_errors',[]))})
            if row['state'] not in EXEC_TERMINAIS and not partial:continue
            validation_rows.append({'request_id':rid,'state':row['state'],'validator_version':details.get('validator_version',''),
                'raw_entities_count':details.get('raw_entities_count',0),'aligned_mentions':len(json.loads(row['entities_json'] or '[]')),
                'unresolved_records':len(details.get('unresolved',[])),'duplicate_records_removed':details.get('duplicates_removed',0),
                'alignment_policy':'first_unused_surface'})
            for problem in details.get('unresolved',[]):
                entity=problem.get('original_entity',{})
                alignment_issues.append({'request_id':rid,'news_ids_json':qmap[rid]['news_ids_json'],
                    'entity_index':problem.get('entity_index'),'reason':problem.get('reason'),
                    'original_entity_json':exec_json(entity),'candidate_spans_json':exec_json(problem.get('candidate_spans',[]))})
            entities=json.loads(row['entities_json']);q=qmap[rid]
            for nid in json.loads(q['news_ids_json']):
                doc=dmap[nid]
                if partial:
                    partial_doc_counts[nid]=(len(entities),len(details.get('unresolved',[])))
                else:
                    new_doc_counts[nid]=(len(entities),sum(e['label'] in EXEC_POL for e in entities),row['state'],row['model_version'])
                for index,e in enumerate(entities):
                    mid=exec_sha(json.dumps([nid,e['label'],e['start'],e['end'],e['surface']],ensure_ascii=False,separators=(',',':')))
                    (partial_mentions if partial else new_mentions).append({'mention_id':mid,'news_id':nid,'news_version_id':doc['news_version_id'],
                        'ner_input_sha256':q['input_sha256'],'source':doc['source'],'link':doc['link'],
                        'publication_date':doc['publication_date'],'start':e['start'],'end':e['end'],
                        'ent_text':e['surface'],'ent_label':e['label'],'model':row['model_version'],
                        'origin':'inference_partial' if partial else 'inference','source_mention_id':rid+':'+str(e.get('entity_index',index)),
                        'source_news_id':nid,'source_file':'checkpoint.sqlite#attempt_'+str(row['latest_attempt']),
                        'source_publication_date':doc['publication_date'],
                        'original_label':e['original_label'],'original_surface':e['original_surface'],
                        'original_start':e['original_start'],'original_end':e['original_end'],
                        'alignment_action':e['alignment_action']})
        with gzip.open(destino/'respostas_brutas.jsonl.gz','wt',encoding='utf-8',newline='') as raw_out:
            for row in db.execute('SELECT * FROM attempts ORDER BY attempt_id'):
                if row['request_id'] not in qmap:continue
                raw=gzip.decompress(row['raw_gzip']) if row['raw_gzip'] is not None else None
                try:envelope=json.loads(raw) if raw is not None else {}
                except (ValueError,UnicodeDecodeError):envelope={}
                usage=json.loads(row['usage_json']) if row['usage_json'] else {}
                usage_rows.append({'request_id':row['request_id'],'attempt_id':row['attempt_id'],
                    'state':row['state'],'http_status':row['http_status'],
                    'model_version':envelope.get('modelVersion','') if isinstance(envelope,dict) else '',
                    'prompt_tokens':usage.get('promptTokenCount'),
                    'candidate_tokens':usage.get('candidatesTokenCount'),
                    'thinking_tokens':usage.get('thoughtsTokenCount'),
                    'total_tokens':usage.get('totalTokenCount'),
                    'usage_reported':bool(usage),'started_at':row['started_at'],'finished_at':row['finished_at']})
                raw_out.write(exec_json({'request_id':row['request_id'],'attempt_id':row['attempt_id'],
                    'state':row['state'],'http_status':row['http_status'],
                    'started_at':row['started_at'],'finished_at':row['finished_at'],
                    'response_text':raw.decode('utf-8',errors='replace') if raw is not None else None})+'\n')
        db.rollback()
    pd.DataFrame(truncation_rows,columns=['request_id','news_ids_json','state','model_version','finish_reason','input_chars','answer_chars','complete_objects','aligned_mentions','alignment_issues','last_complete_object_end','unprocessed_suffix_chars','requires_future_review','automatic_retry']).to_csv(destino/'respostas_interrompidas.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(parse_rows,columns=['request_id','state','parser_version','parser_strategy','repair_count','repairs_json','parse_errors_json']).to_csv(destino/'auditoria_parse.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame(validation_rows,columns=['request_id','state','validator_version','raw_entities_count','aligned_mentions','unresolved_records','duplicate_records_removed','alignment_policy']).to_csv(destino/'auditoria_alinhamento.csv',index=False,encoding='utf-8-sig')
    requests=pd.DataFrame(request_rows,columns=['request_id','state','attempts','model_version','diagnostic','updated_at'])
    use=pd.DataFrame(usage_rows,columns=['request_id','attempt_id','state','http_status','model_version',
        'prompt_tokens','candidate_tokens','thinking_tokens','total_tokens','usage_reported','started_at','finished_at'])
    mentions=pd.DataFrame(new_mentions,columns=columns)
    partial_table=pd.DataFrame(partial_mentions,columns=columns)
    partial_table.to_csv(destino/'mencoes_parciais.csv.gz',index=False,encoding='utf-8-sig',compression='gzip')
    pd.DataFrame(alignment_issues,columns=['request_id','news_ids_json','entity_index','reason','original_entity_json','candidate_spans_json']).to_csv(destino/'pendencias_alinhamento.csv',index=False,encoding='utf-8-sig')
    if not mentions.empty:
        if not mentions.mention_id.is_unique:raise ValueError('Menções novas repetidas na exportação.')
        if set(mentions.news_id)&set(docs.loc[docs.status.eq('ARTIFACT_AVAILABLE'),'news_id']):
            raise ValueError('A exportação tentaria substituir um artefato já disponível.')
    mentions.to_csv(destino/'mencoes_todas_execucao.csv.gz',index=False,encoding='utf-8-sig',compression='gzip')
    mentions.loc[mentions.ent_label.isin(EXEC_POL)].to_csv(destino/'mencoes_pol_execucao.csv.gz',index=False,encoding='utf-8-sig',compression='gzip')
    requests.to_csv(destino/'status_requisicoes.csv',index=False,encoding='utf-8-sig')
    use.to_csv(destino/'uso_tokens.csv',index=False,encoding='utf-8-sig')
    tracking=queue[['request_id','news_ids_json','source','year','news_count']].copy()
    tracking['title']=queue.input_text.map(lambda text:' '.join(text.split('\n',1)[0].split()))
    tracking=tracking.merge(requests[['request_id','state','attempts']],on='request_id',how='left',validate='one_to_one')
    tracking['state']=tracking.state.fillna('PENDING')
    tracking['attempts']=pd.to_numeric(tracking.attempts).fillna(0).astype(int)
    tracking.to_csv(destino/'acompanhamento_requisicoes.csv',index=False,encoding='utf-8-sig')

    attempted=set(requests.loc[pd.to_numeric(requests.attempts).gt(0),'request_id'])
    queue.loc[queue.request_id.isin(attempted)].to_csv(destino/'entradas_processadas.csv.gz',
        index=False,encoding='utf-8-sig',compression='gzip')
    request_state_map=dict(zip(requests.request_id,requests.state))
    current=docs.copy();current['effective_status']=current.status
    has_request=current.request_id.isin(request_state_map) & current.status.eq('PENDING')
    current.loc[has_request,'effective_status']=current.loc[has_request,'request_id'].map(request_state_map)
    current['new_mentions_all']=0;current['new_mentions_pol']=0
    current['partial_mentions_aligned']=0;current['alignment_issues']=0
    truncated_ids={r['request_id'] for r in truncation_rows}
    current['response_truncated']=current.request_id.isin(truncated_ids)
    current['truncation_review_required']=current['response_truncated']
    for nid,(aligned,unresolved) in partial_doc_counts.items():
        mask=current.news_id.eq(nid);current.loc[mask,'effective_status']='PARTIAL'
        current.loc[mask,'partial_mentions_aligned']=aligned;current.loc[mask,'alignment_issues']=unresolved
    for nid,(nall,npol,state,version) in new_doc_counts.items():
        mask=current.news_id.eq(nid)
        current.loc[mask,'effective_status']=state
        current.loc[mask,'new_mentions_all']=nall;current.loc[mask,'new_mentions_pol']=npol
    attempted_ids=set(requests.loc[pd.to_numeric(requests.attempts).gt(0),'request_id'])
    current['inference_attempted']=current.request_id.isin(attempted_ids)
    current.to_csv(destino/'cobertura_atualizada.csv',index=False,encoding='utf-8-sig')
    state_map=dict(zip(requests.request_id,requests.state))
    successful=set(requests.loc[requests.state.isin(EXEC_TERMINAIS),'request_id'])
    scope_ids=set(selecionar_periodo_ner(queue,docs,ANO_MAXIMO_NER).request_id)
    scoped_docs=docs.loc[docs.status.eq('PENDING') & (
        pd.to_numeric(docs.year).le(ANO_MAXIMO_NER) if ANO_MAXIMO_NER is not None else True)]
    scoped_news=set(scoped_docs.news_id)
    summary={'created_at_utc':exec_agora(),'contract':contract,'prompt_sha256':EXEC_PROMPT_SHA,
        'requests_total':len(queue),'requests_successful':len(successful),
        'requests_not_successful':len(queue)-len(successful),
        'states':dict(Counter(state_map.get(r,'PENDING') for r in queue.request_id)),
        'attempts_recorded':len(use),'attempts_without_usage':int((~use.usage_reported).sum()),
        'news_completed_now':len(new_doc_counts),'new_mentions_all':len(mentions),
        'new_mentions_pol':int(mentions.ent_label.isin(EXEC_POL).sum()),
        'news_with_available_artifact':int(docs.status.eq('ARTIFACT_AVAILABLE').sum()),
        'news_still_pending':int(docs.status.eq('PENDING').sum())-len(new_doc_counts),
        'news_not_attempted':int((current.status.eq('PENDING') & ~current.inference_attempted).sum()),
        'prompt_tokens_reported':int(pd.to_numeric(use.prompt_tokens,errors='coerce').fillna(0).sum()),
        'candidate_tokens_reported':int(pd.to_numeric(use.candidate_tokens,errors='coerce').fillna(0).sum()),
        'thinking_tokens_reported':int(pd.to_numeric(use.thinking_tokens,errors='coerce').fillna(0).sum()),
        'executor_version':'ner-executor-8','validator_version':'ner-validation-5',
        'parser_version':'ner-json-2','alignment_version':'ner-first-unused-3',
        'requests_partial':int(requests.state.eq('PARTIAL').sum()),
        'requests_truncated':len(truncation_rows),
        'truncated_aligned_mentions':sum(r['aligned_mentions'] for r in truncation_rows),
        'text_quality_policy':'as_collected_no_automatic_exclusion',
        'news_partial':len(partial_doc_counts),
        'partial_mentions_aligned':len(partial_table),'alignment_issues':len(alignment_issues),
        'duplicate_records_removed':sum(r['duplicate_records_removed'] for r in validation_rows),
        'responses_with_parser_repair':sum(bool(r['repair_count']) for r in parse_rows),
        'parser_strategies':dict(Counter(r['parser_strategy'] for r in parse_rows)),
        'alignment_policy':'first_unused_surface',
        'year_maximum_selected':ANO_MAXIMO_NER,
        'selected_period_requests':len(scope_ids),'implementation':'Gemini REST generateContent v1beta',
        'failure_policy':{'individual_failures':'continue_without_automatic_retry',
                          'max_consecutive_failures_configured':globals().get('LIMITE_FALHAS_CONSECUTIVAS_NER',3),
                          'immediate_http_stop':[400,401,402,403,404,405,415,429],
                          'failure_order':'local_completion'},
        'selected_period_states':dict(Counter(state_map.get(r,'PENDING') for r in scope_ids)),
        'selected_period_requests_successful':len(successful & scope_ids),
        'selected_period_requests_not_successful':len(scope_ids-successful),
        'selected_period_news_still_pending':len(scoped_news)-sum(nid in scoped_news for nid in new_doc_counts),
        'selected_period_news_not_attempted':int((current.news_id.isin(scoped_news) & ~current.inference_attempted).sum()),
        'deferred_requests':len(queue)-len(scope_ids),
        'input_mode':MODO_ENTRADA_NER,'accepted_model_versions':sorted(EXEC_VERSOES_ACEITAS),
        'notebook_sources_sha256':exec_sha(exec_json([(c['cell_type'],''.join(c['source'])) for c in
            json.loads((EXEC_RAIZ/'Pipeline_Noticias_IA.ipynb').read_text(encoding='utf-8'))['cells']])),
        'key_stored':False,'base_artifacts_modified':False}
    (destino/'resumo_execucao_ner.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
    names=['respostas_interrompidas.csv','auditoria_parse.csv','acompanhamento_requisicoes.csv','auditoria_alinhamento.csv','resumo_execucao_ner.json','status_requisicoes.csv','uso_tokens.csv','cobertura_atualizada.csv',
           'entradas_processadas.csv.gz','mencoes_parciais.csv.gz','pendencias_alinhamento.csv',
           'mencoes_todas_execucao.csv.gz','mencoes_pol_execucao.csv.gz','respostas_brutas.jsonl.gz']
    package=destino/'Resultados_NER.zip'
    with zipfile.ZipFile(package,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for name in names:z.write(destino/name,name)
    return package


## 18. Testes locais do executor

Os testes utilizam respostas e transportes simulados, sem rede e sem credenciais. Verificam contrato, persistência, retomada, parsing, alinhamento, truncamento e concorrência. Não avaliam a qualidade linguística do modelo.


In [ ]:
def testar_executor_ner():
    """Respostas sintéticas, banco temporário e nenhum acesso à rede ou ao .env real."""
    import concurrent.futures
    checks=[]
    def check(name,condition):
        if not condition:raise AssertionError(name)
        checks.append(name)
    model=EXEC_CONTRATO['model']
    def response(entities,finish='STOP',version=None):
        answer=entities if isinstance(entities,str) else json.dumps(entities,ensure_ascii=False)
        return json.dumps({'modelVersion':version or model,
            'candidates':[{'finishReason':finish,'content':{'parts':[{'text':answer}]}}],
            'usageMetadata':{'promptTokenCount':10,'candidatesTokenCount':3,'thoughtsTokenCount':2}}).encode()
    entity={'start':0,'end':3,'label':'PESSOA','surface':'Ana'}
    validate=lambda raw,text='Ana':validar_resposta_ner(raw,200,text,{model})
    check('valid_entity',validate(response([entity]))['state']=='SUCCESS')
    check('valid_empty',validate(response([]))['state']=='SUCCESS_EMPTY')
    check('invalid_is_not_empty',validate(response('{'))['state']=='INVALID_RESPONSE')
    check('truncated_is_not_success',validate(response([],finish='MAX_TOKENS'))['state']=='INVALID_RESPONSE')
    check('blocked_is_not_empty',validate(response([],finish='SAFETY'))['state']=='BLOCKED')
    check('effective_model_verified',validate(response([],version='another-model'))['state']=='MODEL_MISMATCH')
    check('duplicate_json_keys_rejected',validate(response('[{"start":0,"start":1,"end":3,"label":"PESSOA","surface":"Ana"}]'))['state']=='INVALID_RESPONSE')
    check('bool_is_not_offset',validate(response([{**entity,'start':True}]))['state']=='PARTIAL')
    check('unexpected_label_flagged',validate(response([{**entity,'label':'PERSONAGEM'}]))['state']=='PARTIAL')
    fixed=validate(response([{**entity,'start':90,'end':93}]))
    check('unique_literal_repair',fixed['state']=='SUCCESS' and fixed['entities'][0]['start']==0)
    check('repeated_surface_first_unused',validate(response([{**entity,'start':90,'end':93}]),'Ana Ana')['entities'][0]['start']==0)
    check('repeated_occurrence_by_offset',validate(response([entity,{**entity,'start':4,'end':7}]),'Ana Ana')['state']=='SUCCESS')
    check('excess_duplicate_removed',len(validate(response([entity,entity]))['entities'])==1)
    check('identical_predictions_use_distinct_occurrences',len(validate(response([entity,entity]),'Ana Ana')['entities'])==2)
    check('literal_surface_repair',validate(response([{**entity,'surface':'ANA'}]))['entities'][0]['surface']=='Ana')
    check('alias_label_audited',validate(response([{**entity,'label':'ORGANIZACION'}]))['entities'][0]['original_label']=='ORGANIZACION')
    check('http_error_distinct',validar_resposta_ner(b'{"error":{}}',429,'Ana',{model})['state']=='API_ERROR')
    check('envelope_model_type',validate(json.dumps({'modelVersion':{},'candidates':[]}).encode())['state']=='MODEL_MISMATCH')
    # Valida a montagem real do pedido HTTP, substituindo a rede por um objeto local.
    original_urlopen=globals()['urlopen'];captured=[]
    class MockHTTP:
        status=200;headers={'Content-Type':'application/json'}
        def __enter__(self):return self
        def __exit__(self,*args):return False
        def read(self):return response([])
    def mock_open(request,timeout):
        captured.append((request,timeout));return MockHTTP()
    try:
        globals()['urlopen']=mock_open
        transporte_gemini_rest(EXEC_PROMPT.replace('{text}','Ana'),EXEC_CONTRATO,'unit-test-key',180)
    finally:globals()['urlopen']=original_urlopen
    payload=json.loads(captured[0][0].data)
    check('rest_exact_prompt',payload['contents'][0]['parts'][0]['text']==EXEC_PROMPT.replace('{text}','Ana'))
    check('rest_same_generation_parameters',payload['generationConfig']=={'temperature':0,'seed':42,'responseMimeType':'application/json'})
    check('rest_key_not_in_url','unit-test-key' not in captured[0][0].full_url and '?' not in captured[0][0].full_url)
    count_fix=validate(response([{**entity,'start':90,'end':93},{**entity,'start':95,'end':98}]),'Ana Ana')
    check('count_match_recovers_all_occurrences',count_fix['state']=='SUCCESS' and len(count_fix['entities'])==2)
    check('count_match_is_audited',all(x['alignment_action']=='FIRST_UNUSED_LITERAL' for x in count_fix['entities']))
    boundary=validate(response([{**entity,'start':0,'end':4}]),'Ana Ana')
    check('one_boundary_repairs_end',boundary['state']=='SUCCESS' and boundary['entities'][0]['end']==3)
    partial=validate(response([entity,{**entity,'surface':'Bia','start':50,'end':53}]),'Ana Bia Bia')
    check('repeated_surface_assigns_first_unused',partial['state']=='SUCCESS' and len(partial['entities'])==2)
    overlap=validate(response([entity,{**entity,'surface':'Ana Bia','end':7}]),'Ana Bia')
    check('overlap_candidates_preserved',overlap['state']=='PARTIAL' and overlap['unresolved'][0]['candidate_spans']==[[0,7]])
    check('partial_is_not_empty_success',validate(response([{**entity,'surface':'Bia'}]))['state']=='PARTIAL')
    # Estratégias de leitura e recuperação do conteúdo NER.
    payload=json.dumps([entity],ensure_ascii=False)
    wrappers=[('direct_dict_entities',json.dumps({'entities':[entity]})),
        ('fenced_list','```json\n'+payload+'\n```'),
        ('fenced_dict_entities','```json\n'+json.dumps({'entities':[entity]})+'\n```'),
        ('embedded_list','Resultado:\n'+payload+'\nFim.'),
        ('embedded_dict_entities','Resultado:\n'+json.dumps({'entities':[entity]})+'\nFim.')]
    for strategy,answer in wrappers:
        parsed=interpretar_json_ner(answer)
        check('parse_'+strategy,parsed['entities']==[entity] and parsed['strategy']==strategy)
    check('bare_fence_supported',interpretar_json_ner('```\n'+payload+'\n```')['entities']==[entity])
    check('valid_empty_wrapper',validate(response(json.dumps({'entities':[]})))['state']=='SUCCESS_EMPTY')
    check('parse_failure_is_not_empty',interpretar_json_ner('sem resposta')['entities'] is None)
    check('wrong_wrapper_type_rejected',interpretar_json_ner('{"entities":{}}')['entities'] is None)
    check('unrelated_object_not_salvaged',interpretar_json_ner('{"other":[]}')['entities'] is None)
    check('multiple_payloads_rejected',interpretar_json_ner(payload+'\n'+payload)['entities'] is None)
    check('multiple_fences_rejected',interpretar_json_ner('```json\n[]\n```\n```json\n[]\n```')['entities'] is None)
    check('competing_payload_outside_fence_rejected',interpretar_json_ner('```json\n[]\n```\n[]')['entities'] is None)
    check('truncated_outer_not_salvaged',interpretar_json_ner('{"entities":[]')['entities'] is None)
    brackets={**entity,'surface':'Ana [teste]'}
    check('brackets_in_surface_parsed',interpretar_json_ner('Resposta: '+json.dumps([brackets]))['entities']==[brackets])
    malformed='[\n {"start":0,"end":3,"label":"PESSOA",\n  ""surface":"Ana"}\n]'
    repaired=interpretar_json_ner(malformed)
    check('single_key_quote_recovered',repaired['entities']==[entity])
    check('key_quote_repair_audited',len(repaired['repairs'])==1 and repaired['repairs'][0]['rule']=='single_extra_quote_surface_key')
    check('key_quote_can_disable_repair',interpretar_json_ner(malformed,permitir_reparo=False)['entities'] is None)
    fixed=validate(response(malformed))
    check('repaired_response_aligned',fixed['state']=='SUCCESS' and fixed['entities'][0]['surface']=='Ana')
    check('repaired_response_keeps_parser_trace',len(fixed['parser_repairs'])==1)
    check('truncated_response_not_repaired',validate(response(malformed,finish='MAX_TOKENS'))['state']=='INVALID_RESPONSE')
    check('multiple_quote_errors_not_repaired',interpretar_json_ner(malformed[:-1]+',\n {"start":4,"end":7,"label":"PESSOA",\n ""surface":"Bia"}]')['entities'] is None)
    quoted=json.dumps([{**entity,'surface':'texto \"\"surface\": valor'}])
    check('surface_content_not_repaired',interpretar_json_ner(quoted)['repairs']==[])
    check('duplicate_keys_not_repaired',interpretar_json_ner('[{"start":0,"start":1,"end":3,"label":"PESSOA","surface":"Ana"}]')['entities'] is None)

    # Aproveitamento parcial de respostas encerradas no limite de geração.
    incomplete='[\n'+json.dumps(entity)+',\n {"start":4,"end":'
    recovered=validate(response(incomplete,finish='MAX_TOKENS'))
    check('max_tokens_preserves_complete_object',recovered['state']=='PARTIAL' and len(recovered['entities'])==1)
    check('max_tokens_requires_future_review',recovered['response_truncated'] and recovered['truncation_review_required'])
    check('max_tokens_suffix_audited',recovered['unprocessed_suffix_chars']>0 and recovered['last_complete_object_end'] is not None)
    full_at_limit=validate(response([entity],finish='MAX_TOKENS'))
    check('closed_list_at_limit_stays_partial',full_at_limit['state']=='PARTIAL' and full_at_limit['response_truncated'])
    check('empty_limit_is_not_valid_empty',validate(response([],finish='MAX_TOKENS'))['state']=='INVALID_RESPONSE')
    check('unfinished_first_object_not_invented',validate(response('[{"surface":"Ana"',finish='MAX_TOKENS'))['state']=='INVALID_RESPONSE')
    check('unknown_finish_not_recovered',validate(response([entity],finish='OTHER'))['state']=='INVALID_RESPONSE')
    check('safety_not_recovered',validate(response([entity],finish='SAFETY'))['state']=='BLOCKED')
    wrapped='{"entities":['+json.dumps(entity)+', {"start":'
    check('truncated_entities_wrapper_supported',validate(response(wrapped,finish='MAX_TOKENS'))['state']=='PARTIAL')
    fenced='```json\n['+json.dumps(entity)+', {"start":'
    check('truncated_fence_supported',validate(response(fenced,finish='MAX_TOKENS'))['state']=='PARTIAL')
    ambiguous_prefix='['+json.dumps({**entity,'surface':'Ausente'})+', {"start":'
    pp=validate(response(ambiguous_prefix,finish='MAX_TOKENS'))
    check('truncated_alignment_issues_preserved',pp['state']=='PARTIAL' and len(pp['unresolved'])==1)
    duplicate_prefix='['+json.dumps(entity)+',{ "start":0,"start":1,"end":'
    check('truncated_duplicate_key_not_overwritten',len(validate(response(duplicate_prefix,finish='MAX_TOKENS'))['entities'])==1)
    invalid_object='[true,'+json.dumps(entity)
    check('nonobject_prefix_not_skipped',validate(response(invalid_object,finish='MAX_TOKENS'))['state']=='INVALID_RESPONSE')

    def fixture(texts):
        qs=[];ds=[]
        for i,text in enumerate(texts):
            rid=exec_request_id(text,EXEC_CONTRATO);nid='test-news-'+str(i)
            qs.append({'request_id':rid,'input_sha256':exec_sha(text),'prompt_sha256':EXEC_PROMPT_SHA,
                'input_text':text,'news_ids_json':json.dumps([nid]),'news_count':'1','source':'Folha de S.Paulo','year':'2012'})
            ds.append({'news_id':nid,'request_id':rid,'status':'PENDING'})
        return pd.DataFrame(qs),pd.DataFrame(ds)
    q,d=fixture(['Ana']);calls=[]
    def fake(prompt,contract,key,timeout):
        calls.append(prompt)
        return 200,response([entity]),{'Content-Type':'application/json','x-goog-api-key':'NEVER_SAVE_THIS'}
    with tempfile.TemporaryDirectory(prefix='ner_executor_test_') as temp:
        temp=Path(temp);dbpath=temp/'main.sqlite'
        preview=executar_fila_ner(q,d,EXEC_CONTRATO,EXEC_PROMPT,dbpath,executar=False,transport=fake)
        check('preview_no_state_no_call',preview['calls']==0 and not dbpath.exists() and not calls)
        kwargs=dict(executar=True,transport=fake,max_calls=1)
        executar_fila_ner(q,d,EXEC_CONTRATO,EXEC_PROMPT,dbpath,**kwargs)
        executar_fila_ner(q,d,EXEC_CONTRATO,EXEC_PROMPT,dbpath,**kwargs)
        check('resume_does_not_repeat_success',len(calls)==1)
        check('exact_prompt_sent',calls[0]==EXEC_PROMPT.replace('{text}','Ana'))
        with closing(abrir_checkpoint_ner(dbpath)) as db:
            row=db.execute('SELECT * FROM attempts').fetchone()
            check('raw_response_preserved',gzip.decompress(row['raw_gzip'])==response([entity]))
            check('credential_headers_not_stored','NEVER_SAVE_THIS' not in row['headers_json'])
        # Resposta persistida e processo interrompido antes da validação.
        recover=temp/'recover.sqlite'
        with closing(abrir_checkpoint_ner(recover)) as db:
            registrar_fila_checkpoint(db,q,EXEC_CONTRATO)
            aid=reservar_requisicao(db,q.iloc[0].request_id)
            salvar_resposta_bruta(db,q.iloc[0].request_id,aid,200,response([]),{})
        before=len(calls)
        result=executar_fila_ner(q,d,EXEC_CONTRATO,EXEC_PROMPT,recover,**kwargs)
        check('recover_raw_without_paid_repeat',len(calls)==before and result['recovered_without_call']==1)
        old_db=temp/'legacy.sqlite'
        with closing(abrir_checkpoint_ner(old_db)) as db:
            registrar_fila_checkpoint(db,q,EXEC_CONTRATO)
            aid=reservar_requisicao(db,q.iloc[0].request_id)
            salvar_resposta_bruta(db,q.iloc[0].request_id,aid,200,response([entity]),{})
            with db:
                db.execute("UPDATE requests SET state='INVALID_RESPONSE',validation_json='{}' WHERE request_id=?",(q.iloc[0].request_id,))
                db.execute("UPDATE attempts SET state='INVALID_RESPONSE' WHERE attempt_id=?",(aid,))
        local=revalidar_respostas_salvas_ner(q,EXEC_CONTRATO,EXEC_PROMPT,old_db)
        check('legacy_response_recovers_without_api',local['revalidated']==1 and local['api_calls']==0)
        check('legacy_backup_created',Path(local['backup']).is_file())
        with closing(abrir_checkpoint_ner(old_db)) as db:
            check('revalidation_does_not_create_attempt',db.execute('SELECT count(*) FROM attempts').fetchone()[0]==1)
        again=revalidar_respostas_salvas_ner(q,EXEC_CONTRATO,EXEC_PROMPT,old_db)
        check('revalidation_is_idempotent',again['revalidated']==0)
        # Falha de transporte não pode virar sucesso vazio ou repetição automática.
        failed=[]
        def timeout(*args):failed.append(1);raise TimeoutError('simulated')
        uncertain=temp/'uncertain.sqlite'
        for _ in range(2):executar_fila_ner(q,d,EXEC_CONTRATO,EXEC_PROMPT,uncertain,executar=True,max_calls=1,transport=timeout)
        check('uncertain_not_retried_by_default',len(failed)==1)
        check('uncertain_state_reported',consultar_progresso_ner(q,uncertain).iloc[0].state=='UNCERTAIN')
        rate_calls=[]
        def rate(*args):rate_calls.append(1);return 429,b'{"error":{"code":429}}',{'Retry-After':'30'}
        rate_db=temp/'rate.sqlite'
        for _ in range(2):executar_fila_ner(q,d,EXEC_CONTRATO,EXEC_PROMPT,rate_db,executar=True,max_calls=1,transport=rate)
        check('http_error_not_retried_by_default',len(rate_calls)==1)
        executar_fila_ner(q,d,EXEC_CONTRATO,EXEC_PROMPT,rate_db,executar=True,max_calls=1,transport=rate,retry_failed=True)
        check('explicit_retry_is_recorded',len(rate_calls)==2)
        executar_fila_ner(q,d,EXEC_CONTRATO,EXEC_PROMPT,rate_db,executar=True,max_calls=1,transport=rate,retry_failed=True)
        check('attempt_cap_prevents_unbounded_retries',len(rate_calls)==2)
        wrong_calls=[]
        def wrong(*args):wrong_calls.append(1);return 200,response([],version='another-model'),{}
        wrong_db=temp/'wrong.sqlite'
        for _ in range(2):executar_fila_ner(q,d,EXEC_CONTRATO,EXEC_PROMPT,wrong_db,executar=True,max_calls=1,transport=wrong)
        check('model_mismatch_halts_next_calls',len(wrong_calls)==1)
        accepted=executar_fila_ner(q,d,EXEC_CONTRATO,EXEC_PROMPT,wrong_db,executar=True,max_calls=1,transport=wrong,accepted_models={model,'another-model'})
        check('review_model_reuses_saved_response',len(wrong_calls)==1 and accepted['recovered_without_call']==1)
        bad=q.copy();bad.loc[0,'input_text']='Outro texto'
        try:executar_fila_ner(bad,d,EXEC_CONTRATO,EXEC_PROMPT,temp/'bad.sqlite',**kwargs)
        except ValueError:check('modified_queue_rejected_before_call',len(calls)==before)
        else:raise AssertionError('modified_queue_rejected_before_call')
        # Duas instâncias tentam reservar a mesma requisição.
        concurrent_db=temp/'concurrent.sqlite'
        with closing(abrir_checkpoint_ner(concurrent_db)) as db:registrar_fila_checkpoint(db,q,EXEC_CONTRATO)
        def reserve(_):
            with closing(abrir_checkpoint_ner(concurrent_db)) as db:return reservar_requisicao(db,q.iloc[0].request_id)
        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:claims=list(pool.map(reserve,range(2)))
        check('atomic_claim_prevents_double_call',sum(x is not None for x in claims)==1)
        # Limite global de tentativas, inclusive com mais de um trabalhador.
        q3,d3=fixture(['Ana','Bia','Eva']);parallel=[]
        def empty(*args):parallel.append(1);return 200,response([]),{}
        executar_fila_ner(q3,d3,EXEC_CONTRATO,EXEC_PROMPT,temp/'parallel.sqlite',executar=True,max_calls=2,workers=2,transport=empty)
        check('parallel_global_call_limit',len(parallel)==2)
        # A chave só é lida do arquivo escolhido; não é copiada para os resultados.
        env=temp/'.env';env.write_text('GEMINI_API_KEY="unit-test-key"\n',encoding='utf-8-sig')
        check('dotenv_local_key',chave_ner_env(env)=='unit-test-key')
        # O apelido identifica a configuração local sem revelar credenciais.
        env.write_text('GEMINI_API_KEY="unit-test-key"\nKEY_NAME="Pesquisa IA"\n',encoding='utf-8-sig')
        check('key_alias_from_local_env',credenciais_ner_env(env)==('unit-test-key','Pesquisa IA'))
        env.write_text('GEMINI_API_KEY="unit-test-key"\nKEY_NAME="unit-test-key"\n',encoding='utf-8')
        check('alias_cannot_expose_key',credenciais_ner_env(env)[1]=='chave local')
        env.write_text('GEMINI_API_KEY="unit-test-key"\nKEY_NAME=""\n',encoding='utf-8')
        check('empty_alias_uses_default',credenciais_ner_env(env)[1]=='chave local')
        env.write_text('GEMINI_API_KEY="unit-test-key"\nKEY_NAME="Pesquisa IA"\n',encoding='utf-8')
        from contextlib import redirect_stdout
        output=io.StringIO()
        saved_transport=globals()['transporte_gemini_rest']
        globals()['transporte_gemini_rest']=fake
        try:
            with redirect_stdout(output):
                executar_fila_ner(q,d,EXEC_CONTRATO,EXEC_PROMPT,temp/'console.sqlite',executar=True,max_calls=1,env_path=env)
        finally:globals()['transporte_gemini_rest']=saved_transport
        console=output.getvalue()
        check('console_displays_alias','apelido: Pesquisa IA' in console)
        check('console_never_displays_key','unit-test-key' not in console and 'NEVER_SAVE_THIS' not in console)
        check('console_reports_send','ENVIO' in console and 'aguardando resposta HTTP' in console)
        check('console_reports_http_and_duration','HTTP 200' in console and 'RETORNO' in console)
        check('console_reports_validation','VALIDAÇÃO' in console and 'SUCCESS' in console and 'resposta salva' in console)
        console_path=temp/'logs'/q.iloc[0].request_id
        attempt_logs=list(console_path.glob('tentativa_*.log'))
        check('per_attempt_log_created',(console_path/'tentativa_000001.log').is_file())
        request_log=(console_path/'tentativa_000001.log').read_text(encoding='utf-8')
        check('log_identifies_attempt_and_request',f'T000001 R{q.iloc[0].request_id[:12]}' in request_log)
        check('log_identifies_news','NOTÍCIA' in request_log and 'Ana' in request_log and 'test-news-0' in request_log)
        check('log_keeps_request_lifecycle',all(stage in request_log for stage in ['ENVIO','RETORNO','VALIDAÇÃO','ENCERRADA']))
        check('log_excludes_key_and_alias','unit-test-key' not in request_log and 'Pesquisa IA' not in request_log)
        check('log_timestamp_utc',all('+00:00 [' in line for line in request_log.splitlines()))

        output=io.StringIO()
        with redirect_stdout(output):
            executar_fila_ner(q,d,EXEC_CONTRATO,EXEC_PROMPT,temp/'console.sqlite',executar=True,max_calls=1,transport=fake)
        check('console_explains_zero_calls','NENHUMA NOVA CHAMADA' in output.getvalue())
        output=io.StringIO()
        with redirect_stdout(output):
            executar_fila_ner(q,d,EXEC_CONTRATO,EXEC_PROMPT,temp/'preview_missing.sqlite',executar=False,env_path=temp/'nonexistent.env')
        check('preview_explains_disabled_calls','PRÉVIA' in output.getvalue() and 'A chave não foi lida' in output.getvalue())
        # Uma única resposta pode servir a dois registros com entrada idêntica.
        q2=q.copy();q2.loc[0,'news_ids_json']=json.dumps(['test-news-0','test-news-alias']);q2.loc[0,'news_count']='2'
        d2=pd.concat([d,d.assign(news_id='test-news-alias')],ignore_index=True)
        d2['news_version_id']='fixture-version';d2['source']='Folha de S.Paulo'
        d2['link']='https://example.test/news';d2['publication_date']='2012-01-01';d2['year']='2012'
        exports=temp/'exports'
        package=exportar_resultados_ner(q2,d2,EXEC_CONTRATO,dbpath,exports,
            EXEC_RAIZ/'dados/processados/ner/mencoes_todas.csv.gz')
        exported=pd.read_csv(exports/'mencoes_todas_execucao.csv.gz')
        check('one_response_multiple_news',len(exported)==2 and exported.mention_id.is_unique)
        summary=json.loads((exports/'resumo_execucao_ner.json').read_text())
        check('coverage_export_reconciled',summary['news_completed_now']==2 and summary['news_still_pending']==0)
        import zipfile
        with zipfile.ZipFile(package) as z:
            check('share_package_excludes_secrets',not any(n.endswith('.env') or n.endswith('.sqlite') for n in z.namelist()))
        exportar_resultados_ner(q2,d2,EXEC_CONTRATO,dbpath,exports,
            EXEC_RAIZ/'dados/processados/ner/mencoes_todas.csv.gz')
        check('repeat_export_no_appended_duplicates',len(pd.read_csv(exports/'mencoes_todas_execucao.csv.gz'))==2)
        # Cobertura parcial permanece pendente e não entra nas menções completas.
        pq,pd_docs=fixture(['Ana Bia'])
        pd_docs['news_version_id']='fixture-version';pd_docs['source']='Folha de S.Paulo'
        pd_docs['link']='https://example.test/partial';pd_docs['publication_date']='2012-01-01';pd_docs['year']='2012'
        partial_db=temp/'partial.sqlite'
        def partial_transport(*args):return 200,response([entity,{**entity,'surface':'Ausente'}]),{}
        executar_fila_ner(pq,pd_docs,EXEC_CONTRATO,EXEC_PROMPT,partial_db,executar=True,max_calls=1,transport=partial_transport)
        partial_exports=temp/'partial_exports'
        exportar_resultados_ner(pq,pd_docs,EXEC_CONTRATO,partial_db,partial_exports,temp/'missing.csv.gz')
        ps=json.loads((partial_exports/'resumo_execucao_ner.json').read_text())
        pc=pd.read_csv(partial_exports/'cobertura_atualizada.csv')
        check('partial_news_not_completed',ps['news_completed_now']==0 and ps['news_partial']==1)
        check('partial_remains_pending_completion',ps['news_still_pending']==1 and ps['selected_period_news_still_pending']==1)
        check('partial_is_not_unattempted',ps['news_not_attempted']==0 and ps['selected_period_news_not_attempted']==0)
        check('partial_coverage_fields',pc.iloc[0].partial_mentions_aligned==1 and pc.iloc[0].alignment_issues==1 and pc.iloc[0].new_mentions_all==0)
        check('partial_not_in_complete_mentions',pd.read_csv(partial_exports/'mencoes_todas_execucao.csv.gz').empty)
        check('partial_audit_exported',pd.read_csv(partial_exports/'auditoria_alinhamento.csv').iloc[0].unresolved_records==1)
        check('tracking_identifies_request',pd.read_csv(partial_exports/'acompanhamento_requisicoes.csv').iloc[0].request_id==pq.iloc[0].request_id)
        uq,ud=fixture(['Bia'])
        ud['news_version_id']='fixture-version';ud['source']='Folha de S.Paulo'
        ud['link']='https://example.test/uncertain';ud['publication_date']='2012-01-01';ud['year']='2012'
        uncertain_export_db=temp/'uncertain_export.sqlite'
        executar_fila_ner(uq,ud,EXEC_CONTRATO,EXEC_PROMPT,uncertain_export_db,executar=True,max_calls=1,transport=timeout)
        uncertain_exports=temp/'uncertain_exports'
        exportar_resultados_ner(uq,ud,EXEC_CONTRATO,uncertain_export_db,uncertain_exports,temp/'missing.csv.gz')
        us=json.loads((uncertain_exports/'resumo_execucao_ner.json').read_text())
        uc=pd.read_csv(uncertain_exports/'cobertura_atualizada.csv')
        check('uncertain_not_counted_unattempted',us['news_not_attempted']==0 and us['selected_period_news_not_attempted']==0)
        check('uncertain_state_in_news_coverage',uc.iloc[0].effective_status=='UNCERTAIN' and bool(uc.iloc[0].inference_attempted))
        check('uncertain_remains_pending_completion',us['news_still_pending']==1)
        # Identificadores e arquivos de log permanecem separados em concorrência.
        log_files=list((temp/'logs').glob('*/tentativa_*.log'))
        check('multiple_attempt_logs_isolated',len(log_files)>1)
        check('parallel_logs_keep_request_identity',all(all('R'+rid[:12] in line for path in (temp/'logs'/rid).glob('tentativa_*.log') for line in path.read_text(encoding='utf-8').splitlines()) for rid in q3.request_id.iloc[:2]))

        # Uma mudança de política também revalida respostas previamente completas.
        with closing(abrir_checkpoint_ner(dbpath)) as db:
            with db:db.execute("UPDATE requests SET validation_json=?",(json.dumps({'validator_version':'ner-literal-2'}),))
        migrated=revalidar_respostas_salvas_ner(q,EXEC_CONTRATO,EXEC_PROMPT,dbpath)
        check('successful_response_policy_migration',migrated['revalidated']==1 and migrated['api_calls']==0)
        # Falhas isoladas não impedem o tratamento de outros documentos.
        many_q,many_d=fixture(['Ana '+str(i) for i in range(8)])
        def simulate(name,outcomes,workers=1,limit=3):
            sequence=iter(outcomes);observed=[]
            def mixed(*args):
                value=next(sequence);observed.append(value)
                if value=='timeout':raise TimeoutError('simulated')
                if value=='invalid':return 200,response('{'),{}
                if value=='partial':return 200,response([entity,{**entity,'surface':'Ausente'}]),{}
                if isinstance(value,int):return value,b'{}',{}
                return 200,response([]),{}
            result=executar_fila_ner(many_q,many_d,EXEC_CONTRATO,EXEC_PROMPT,temp/(name+'.sqlite'),
                executar=True,max_calls=len(outcomes),workers=workers,transport=mixed,max_consecutive_failures=limit)
            return result,observed
        r,o=simulate('isolated_invalid',['invalid','ok','ok'])
        check('isolated_invalid_does_not_halt',r['calls']==3 and r['halt_reason'] is None)
        r,o=simulate('isolated_timeout',['timeout','ok'])
        check('isolated_timeout_does_not_halt',r['calls']==2 and r['states_this_run'].get('UNCERTAIN')==1)
        r,o=simulate('isolated_server_error',[503,'ok'])
        check('isolated_server_error_does_not_halt',r['calls']==2 and r['halt_reason'] is None)
        r,o=simulate('consecutive_invalid',['invalid']*5)
        check('consecutive_invalid_stops_new_calls',r['calls']==3 and r['halt_reason']=='CONSECUTIVE_FAILURES')
        r,o=simulate('consecutive_timeouts',['timeout']*5)
        check('consecutive_timeouts_stops_new_calls',r['calls']==3 and r['failure_streak_at_halt']==3)
        r,o=simulate('reset_streak',['invalid','invalid','ok','invalid','invalid','ok'])
        check('success_resets_failure_streak',r['calls']==6 and r['halt_reason'] is None)
        r,o=simulate('partial_resets_streak',['invalid','invalid','partial','invalid'])
        check('partial_resets_failure_streak',r['calls']==4 and r['halt_reason'] is None)
        for code in [400,401,402,403,404,405,415,429]:
            r,o=simulate('stop_http_'+str(code),[code,'ok','ok'])
            check('http_'+str(code)+'_stops_new_calls',r['calls']==1 and r['halt_reason']=='HTTP_'+str(code))
        r,o=simulate('custom_failure_limit',['invalid']*4,limit=2)
        check('failure_threshold_configurable',r['calls']==2 and r['failure_streak_at_halt']==2)
        try:
            executar_fila_ner(q,d,EXEC_CONTRATO,EXEC_PROMPT,temp/'bad_threshold.sqlite',max_consecutive_failures=0)
        except ValueError:check('invalid_failure_threshold_rejected',True)
        else:check('invalid_failure_threshold_rejected',False)
        # Um bloqueio sistêmico impede novos envios, sem descartar chamadas em curso.
        from threading import Barrier,Event
        barrier=Barrier(4);release_inflight=Event();parallel_seen=[]
        def auth_parallel(prompt,*args):
            parallel_seen.append(prompt);barrier.wait(timeout=10)
            if prompt==EXEC_PROMPT.replace('{text}',many_q.iloc[0].input_text):return 401,b'{}',{}
            if not release_inflight.wait(timeout=10):raise TimeoutError('test synchronization')
            return 200,response([]),{}
        stop_db=temp/'stop_parallel.sqlite'
        original_policy=globals()['politica_falha_ner']
        def release_on_fatal(state,http_status=None):
            action=original_policy(state,http_status)
            if action[0]=='STOP':release_inflight.set()
            return action
        globals()['politica_falha_ner']=release_on_fatal
        try:
            stopped=executar_fila_ner(many_q,many_d,EXEC_CONTRATO,EXEC_PROMPT,stop_db,
                executar=True,max_calls=8,workers=4,transport=auth_parallel)
        finally:
            release_inflight.set();globals()['politica_falha_ner']=original_policy
        check('fatal_error_drains_inflight_calls',stopped['calls']==4 and len(parallel_seen)==4)
        with closing(abrir_checkpoint_ner(stop_db)) as db:
            check('inflight_responses_persisted',db.execute('SELECT count(*) FROM attempts WHERE raw_gzip IS NOT NULL').fetchone()[0]==4)
            check('never_started_remain_pending',db.execute("SELECT count(*) FROM requests WHERE state='PENDING'").fetchone()[0]==4)
        check('unknown_state_stops',politica_falha_ner('UNKNOWN')[0]=='STOP')
        tq,tdoc=fixture(['Ana Bia'])
        tdoc['news_version_id']='fixture-version';tdoc['source']='Folha de S.Paulo'
        tdoc['link']='https://example.test/truncated';tdoc['publication_date']='2012-01-01';tdoc['year']='2012'
        truncated_db=temp/'truncated.sqlite'
        def truncated_transport(*args):return 200,response(incomplete,finish='MAX_TOKENS'),{}
        executar_fila_ner(tq,tdoc,EXEC_CONTRATO,EXEC_PROMPT,truncated_db,executar=True,max_calls=1,transport=truncated_transport)
        truncated_exports=temp/'truncated_exports'
        exportar_resultados_ner(tq,tdoc,EXEC_CONTRATO,truncated_db,truncated_exports,temp/'absent.csv.gz')
        ts=json.loads((truncated_exports/'resumo_execucao_ner.json').read_text())
        tc=pd.read_csv(truncated_exports/'cobertura_atualizada.csv')
        check('truncation_exported_separately',len(pd.read_csv(truncated_exports/'respostas_interrompidas.csv'))==1)
        check('truncation_summary_reconciles',ts['requests_truncated']==1 and ts['truncated_aligned_mentions']==1)
        check('truncation_not_marked_complete',ts['news_completed_now']==0 and ts['news_partial']==1)
        check('truncation_coverage_flag',bool(tc.iloc[0].response_truncated) and bool(tc.iloc[0].truncation_review_required))
        check('truncated_mentions_in_partial_file',len(pd.read_csv(truncated_exports/'mencoes_parciais.csv.gz'))==1 and pd.read_csv(truncated_exports/'mencoes_todas_execucao.csv.gz').empty)
        with closing(abrir_checkpoint_ner(truncated_db)) as db:
            with db:db.execute("UPDATE requests SET state='INVALID_RESPONSE',validation_json=?",(json.dumps({'validator_version':'ner-validation-4'}),))
        migrated=revalidar_respostas_salvas_ner(tq,EXEC_CONTRATO,EXEC_PROMPT,truncated_db)
        check('saved_truncation_revalidated_without_api',migrated['revalidated']==1 and migrated['states']=={'PARTIAL':1} and migrated['api_calls']==0)
        check('truncation_revalidation_idempotent',revalidar_respostas_salvas_ner(tq,EXEC_CONTRATO,EXEC_PROMPT,truncated_db)['revalidated']==0)

        # Uma execução integral não depende de artefatos NER ou IDs de notícias antigos.
        fresh_base=pd.DataFrame([{'news_id':nid,'news_version_id':'v','title':'Ana','full_text':'',
            'source_normalized':'Folha de S.Paulo','publication_year':2012,
            'publication_date_normalized':'2012-01-01','link':'https://example.test/'+nid}
            for nid in ['fresh-1','fresh-2']])
        fq,fd=preparar_ner_integral(fresh_base,EXEC_CONTRATO,EXEC_PROMPT)
        check('fresh_mode_has_all_news',len(fd)==2 and fd.status.eq('PENDING').all())
        check('fresh_mode_deduplicates_equal_input',len(fq)==1 and fq.iloc[0].news_count==2)
        check('fresh_mode_valid_contract',validar_fila_execucao(fq,fd,EXEC_CONTRATO,EXEC_PROMPT))
        period_docs=fd.copy();period_docs['year']=[2025,2026]
        check('period_includes_shared_request_for_old_news',len(selecionar_periodo_ner(fq,period_docs,2025))==1)
        period_docs['year']=2026
        check('period_excludes_2026_only_requests',selecionar_periodo_ner(fq,period_docs,2025).empty)
        check('period_none_includes_everything',len(selecionar_periodo_ner(fq,period_docs,None))==1)

    return checks

testes_executor_ner=testar_executor_ner()
print(f'{len(testes_executor_ner)} testes offline aprovados. Chamadas reais: 0.')


## 19. Execução em lotes

Confira a seção 16 antes de executar. A chave é lida do `.env` local apenas quando as chamadas estão habilitadas. O apelido `KEY_NAME` identifica a execução no console. Preserve o checkpoint para continuar lotes em outro kernel.


In [ ]:
# Não há chamadas enquanto EXECUTAR_NER permanecer False.
if MODO_NER not in {'PILOTO','FILA'}:
    raise ValueError('MODO_NER precisa ser PILOTO ou FILA.')
recuperacao_cache_ner=revalidar_respostas_salvas_ner(
    fila_ner,EXEC_CONTRATO,EXEC_PROMPT,EXEC_DB,EXEC_VERSOES_ACEITAS)
print('Revalidação local, sem API:',recuperacao_cache_ner)
selecao_ner=piloto_ner.request_id.tolist() if MODO_NER=='PILOTO' else fila_ner_escopo.request_id.tolist()
resultado_executor_ner=executar_fila_ner(
    fila_ner,documentos_ner,EXEC_CONTRATO,EXEC_PROMPT,EXEC_DB,selecao_ner,
    executar=EXECUTAR_NER,max_calls=LIMITE_CHAMADAS_NER,workers=TRABALHADORES_NER,
    timeout=TIMEOUT_NER_SEGUNDOS,retry_failed=REPETIR_FALHAS_NER,
    retry_uncertain=REPETIR_INCERTAS_NER,max_attempts=MAX_TENTATIVAS_POR_REQUISICAO,
    accepted_models=EXEC_VERSOES_ACEITAS,env_path=EXEC_RAIZ/'.env',
    max_consecutive_failures=LIMITE_FALHAS_CONSECUTIVAS_NER)
print(resultado_executor_ner)
print('Progresso no período selecionado:')
print(consultar_progresso_ner(fila_ner_escopo,EXEC_DB).to_string(index=False))


## 20. Exportação da execução

A exportação contém respostas, menções completas e parciais, estados e auditorias. Não contém credenciais. O checkpoint local é preservado para retomada.


In [ ]:
pacote_resultados_ner=exportar_resultados_ner(
    fila_ner,documentos_ner,EXEC_CONTRATO,EXEC_DB,EXEC_SAIDA,
    EXEC_RAIZ/'dados/processados/ner/consolidado/mencoes_todas.csv.gz')
if pacote_resultados_ner is None:
    print('Ainda não há execução real. Nenhum pacote de resultados foi criado.')
else:
    print('Pacote para análise:',pacote_resultados_ner)
    resumo_exportado_ner=json.loads((EXEC_SAIDA/"resumo_execucao_ner.json").read_text(encoding="utf-8"))
    print("Exportação concluída (UTC):",resumo_exportado_ner["created_at_utc"])
    print("Tentativas registradas no pacote:",resumo_exportado_ner["attempts_recorded"])


## 21. Consolidação de uma execução

A consolidação valida a ligação com a base, os rótulos, os intervalos literais e a unicidade das menções. O destino deve ser novo: snapshots existentes são preservados. Documentos não tentados continuam identificados. Este fluxo trata a exportação da execução corrente; fontes tabulares adicionais podem ser fornecidas à função `consolidar` de `ner_dados.py`.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from ner_dados import normalizar_exportacao,consolidar

CONSOLIDAR_EXECUCAO_NER=False
EXPORTACAO_NER=Path.cwd()/'resultados/execucao/ner_integral/Resultados_NER.zip'
DESTINO_CONSOLIDADO_NER=Path.cwd()/'dados/processados/ner/consolidado_execucao'
if CONSOLIDAR_EXECUCAO_NER:
    with TemporaryDirectory() as temporario:
        fonte_ner=normalizar_exportacao(EXPORTACAO_NER,Path(temporario)/'fonte')
        resumo_consolidado_ner=consolidar(
            Path.cwd()/'dados/processados/noticias_preparadas.csv',[fonte_ner],DESTINO_CONSOLIDADO_NER)
    print('Consolidação concluída:',resumo_consolidado_ner['availability'])
else:
    print('Consolidação desativada. Nenhum snapshot foi alterado.')


## Limitações metodológicas da resolução de entidades

A resolução por formas textuais e representações contextuais pode reunir homônimos, pessoas com nomes semelhantes ou instituições de jurisdições diferentes. Similaridade vetorial elevada não comprova identidade, e o contexto agregado de uma forma não garante a desambiguação de cada ocorrência. Foram observados exemplos pontuais de associações inadequadas nos resultados de entrada; sua frequência total e seu impacto nas análises de redes não foram quantificados.

A implementação de referência também apresenta limitações na recuperação de candidatos e na aplicação de restrições de separação. A deduplicação pode omitir pares de vizinhança assimétrica, e decisões manuais KEEP_SEPARATE não são garantidas como proibições às uniões diretas ou transitivas. O efeito desses pontos sobre a execução examinada não foi demonstrado quantitativamente.

Essas ressalvas devem acompanhar a interpretação das redes derivadas das associações. A continuidade técnica dos registros não equivale a uma validação semântica das identidades. As melhorias correspondentes são trabalhos futuros; não estão implementadas nesta etapa. Consulte [Limitacoes_Resolucao_Entidades.md](Limitacoes_Resolucao_Entidades.md) para a descrição completa.


## 22. Extração de relações: entradas e contrato

A extração recebe o texto e as menções originais de cada notícia. Os identificadores de entidades resolvidas serão aplicados posteriormente. A fila contém somente publicações anteriores a 2026 selecionadas no planejamento. A ordem é ano, data, fonte e identificador.

O prompt `TEST_D12_T13_L04_R15` tem suas instruções fixas verificadas por SHA-256. O conjunto de menções, o texto, o provedor e os parâmetros integram a identidade de cada requisição.

Selecione explicitamente o provedor: `google` utiliza `GEMINI_API_KEY`; `deepinfra` utiliza `DEEPINFRA_API_KEY`. Ambos são lidos exclusivamente do `.env` local durante a execução habilitada. Não há troca automática de provedor. `KEY_NAME` é apenas o apelido exibido.


In [10]:
from pathlib import Path
from re_execucao import carregar_entradas, executar, exportar

PROJETO_RE = Path.cwd()  # Pasta que contém este notebook.
PROVEDOR_RE = 'deepinfra' # 'google'

EXECUTAR_RE = True # 'False' primeiro, apenas conferir sem chamar a API.
LIMITE_RE = 1288
TRABALHADORES_RE = 4
TIMEOUT_RE = 360
ANOS_RE = list(range(1994, 2025))  # De 1994 até 2024. Quando chegar a vez de 2025, basta alterar: ANOS_RE = [2025]

# Autoriza retomar somente as quatro recusas por chave inválida.
# RETENTAR_REQUISICOES = [
#     '123e8cc350293a211d319454a699c129211b8bae7356012c869be86447bd0c7d',
#     'aa25bd8c6c85521a4cc31de18965be71894cdd0b34f62c8efa83b93620116c2b',
#     'b24c29782ed133f0ad32c650f99a1dcf488ea754408aefcdab325a162fb61449',
#     'f6c0601911aa664e02960e127bf3758dab24a745f23a6cd917b6647512f6bc82',
# ]
RETENTAR_REQUISICOES = []

entradas_re = carregar_entradas(PROJETO_RE)

print(f"Entradas conferidas: {len(entradas_re['queue']):,} notícias anteriores a 2026.")
print(entradas_re['queue'].groupby('re_processing_type').size().to_string())


Entradas conferidas: 9,126 notícias anteriores a 2026.
re_processing_type
PROCESSAR_DO_ZERO    8362
REPROCESSAR           764


## 23. Execução em lotes e retomada

Com `EXECUTAR_RE=False`, a célula abaixo apenas monta uma prévia local e não lê a chave. Com `True`, o limite conta chamadas novas, incluindo tentativas encerradas com erro. Após a validação do primeiro envio, ajuste gradualmente o tamanho do lote e a concorrência.

Repita a execução para avançar pelas notícias ainda não tentadas. `SUCCESS_ZERO` é uma resposta válida sem relações. `PARTIAL`, `INVALID_RESPONSE`, `HTTP_ERROR` e `UNCERTAIN` ficam registrados para inspeção, sem reenvio automático. Respostas recebidas e salvas antes de uma interrupção são recuperadas sem nova chamada. Um timeout não confirma se houve cobrança no provedor.

Respostas inválidas isoladas não encerram o lote. Erros de configuração, autenticação, pagamento e quota pausam novos envios; três falhas consecutivas de transporte/HTTP também pausam. Requisições já em curso ainda podem terminar. O checkpoint só admite uma execução ativa por vez.


In [11]:
resultado_lote_re = executar(
    entradas_re,
    executar_api=EXECUTAR_RE,
    limite=LIMITE_RE,
    trabalhadores=TRABALHADORES_RE,
    provider=PROVEDOR_RE,
    timeout=TIMEOUT_RE,
    anos=ANOS_RE,
    retentar_requisicoes=RETENTAR_REQUISICOES,
)
resultado_lote_re

[RE] EXECUÇÃO — apelido: Deepinfra Ruben; provedor: deepinfra; modelo: google/gemini-2.5-flash; limite: 1288; trabalhadores: 4.
[RE] RECORTE — 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024; 1288 chamadas novas selecionadas.
[RE 5f17e1ad6f30] ENVIO — deepinfra; tentativa c35f4698; 1996; Folha de S.Paulo; 9 menções; espera máxima 360s.
[RE e9b832674f25] ENVIO — deepinfra; tentativa ed90d78d; 1996; Folha de S.Paulo; 23 menções; espera máxima 360s.
[RE b5cdc40fd5f5] ENVIO — deepinfra; tentativa dcadbb68; 1996; Folha de S.Paulo; 31 menções; espera máxima 360s.
[RE 8da86687d384] ENVIO — deepinfra; tentativa db5413a2; 1997; Folha de S.Paulo; 4 menções; espera máxima 360s.
[RE 8da86687d384] RETORNO — HTTP 200; 12.6s; resposta persistida.
[RE 8da86687d384] VALIDAÇÃO — SUCCESS; 1 relações; 0 rejeições.
[RE cb0cf424f84a] ENVIO — deepinfra; tentativa 68eb58c4; 

{'mode': 'EXECUTION',
 'api_calls': 1169,
 'recovered_without_call': 0,
 'states_this_run': {'SUCCESS': 1019,
  'PARTIAL': 105,
  'SUCCESS_ZERO': 44,
  'HTTP_ERROR': 1},
 'provider': 'deepinfra',
 'model': 'google/gemini-2.5-flash',
 'paused': True,
 'selected_years': [1994,
  1995,
  1996,
  1997,
  1998,
  1999,
  2000,
  2001,
  2002,
  2003,
  2004,
  2005,
  2006,
  2007,
  2008,
  2009,
  2010,
  2011,
  2012,
  2013,
  2014,
  2015,
  2016,
  2017,
  2018,
  2019,
  2020,
  2021,
  2022,
  2023,
  2024],
 'selected_news': 8528,
 'finished_at': '2026-11-18T20:44:58.140628+00:00'}

## 24. Cobertura, exportação e verificações

As tabelas distinguem o estado da extração e a disponibilidade do NER. `PARTIAL` no NER significa cobertura de menções incompleta, mesmo quando a chamada de RE termina com sucesso. Relações de respostas de RE parciais são marcadas como provisórias. Nenhuma expansão transitiva, resolução de entidades ou agregação de pesos ocorre nesta etapa.

O pacote de verificação contém estados, validações e respostas da API. O `.env`, o banco local, os prompts e os textos integrais de entrada não são incluídos. As próprias respostas do modelo podem conter trechos da notícia.



In [12]:
pacote_re = exportar(entradas_re)
print(pacote_re)


Progresso na fila anterior a 2026:
state
HTTP_ERROR         1
PARTIAL          109
PENDING         7892
SUCCESS         1078
SUCCESS_ZERO      46
Pacote de verificação: C:\Users\pedro\Documents\MO438\PROJECT\resultados\re\Resumo_RE_20261118_210837_b7d8f3.zip
C:\Users\pedro\Documents\MO438\PROJECT\resultados\re\Resumo_RE_20261118_210837_b7d8f3.zip


## 25.

In [1]:
from pathlib import Path
from re_posprocessamento import processar_re

resumo_pos_re = processar_re(Path.cwd())
resumo_pos_re

[PÓS-RE] {"documents": 1234, "original_relations": 34574, "derived_relations": 34579, "recovered_relations_net": 5, "approved_id_mappings_used": 3, "repaired_endpoint_occurrences": 5, "original_states": {"SUCCESS_ZERO": 46, "PARTIAL": 109, "SUCCESS": 1078, "HTTP_ERROR": 1}, "revalidation_states": {"SUCCESS_ZERO": 46, "PARTIAL": 106, "SUCCESS": 1081, "HTTP_ERROR": 1}, "flagged_relations": 1090, "candidate_rows_for_review": 110, "remaining_rejections": 220, "semantically_auto_approved": 0, "api_calls": 0}
[PÓS-RE] Saídas: C:\Users\pedro\Documents\MO438\PROJECT\dados\processados\re\posprocessado\862c6eeabcc9de314562
[PÓS-RE] Pacote de verificação: C:\Users\pedro\Documents\MO438\PROJECT\resultados\re\Resumo_Pos_RE_20261119_134629_6884b7.zip


{'documents': 1234,
 'original_relations': 34574,
 'derived_relations': 34579,
 'recovered_relations_net': 5,
 'approved_id_mappings_used': 3,
 'repaired_endpoint_occurrences': 5,
 'original_states': {'SUCCESS_ZERO': 46,
  'PARTIAL': 109,
  'SUCCESS': 1078,
  'HTTP_ERROR': 1},
 'revalidation_states': {'SUCCESS_ZERO': 46,
  'PARTIAL': 106,
  'SUCCESS': 1081,
  'HTTP_ERROR': 1},
 'flagged_relations': 1090,
 'candidate_rows_for_review': 110,
 'remaining_rejections': 220,
 'semantically_auto_approved': 0,
 'api_calls': 0,
 'output_directory': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/dados/processados/re/posprocessado/862c6eeabcc9de314562'),
 'package': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/re/Resumo_Pos_RE_20261119_134629_6884b7.zip')}

## 26. Consolidação versionada das relações

Integra relações importadas compatíveis e respostas revalidadas, preservando a origem, a cobertura documental e as pendências. Identidades e demais relações são exportadas separadamente. Esta etapa não realiza chamadas à API nem agrega entidades ou pesos.

In [2]:
from pathlib import Path
from re_consolidacao import consolidar_re

resumo_consolidacao_re = consolidar_re(
    Path.cwd(),
    pasta_posprocessada=resumo_pos_re['output_directory'],
)
resumo_consolidacao_re

[CONSOLIDAÇÃO RE] {"news": 23657, "news_with_re_result": 9906, "news_by_state": {"SUCCESS": 9657, "PENDING": 7892, "NOT_READY_NER": 5686, "TOO_FEW_MENTIONS": 172, "SUCCESS_ZERO": 143, "PARTIAL": 106, "HTTP_ERROR": 1}, "imported_news_selected": 8673, "executed_news_revalidated": 1234, "relations": 418824, "identity_relations": 223877, "semantic_relations": 194947, "relations_by_origin": {"IMPORT": 384245, "EXECUTION": 34579}, "technical_provisional_relations": 9374, "semantic_diagnostic_flags": 15946, "api_calls": 0}
[CONSOLIDAÇÃO RE] Snapshot: C:\Users\pedro\Documents\MO438\PROJECT\dados\processados\re\consolidado\ca93bcb5a6d6a8b46d89
[CONSOLIDAÇÃO RE] Pacote: C:\Users\pedro\Documents\MO438\PROJECT\resultados\re\Resumo_Consolidacao_RE_ca93bcb5a6d6_8ab929.zip


{'news': 23657,
 'news_with_re_result': 9906,
 'news_by_state': {'SUCCESS': 9657,
  'PENDING': 7892,
  'NOT_READY_NER': 5686,
  'TOO_FEW_MENTIONS': 172,
  'SUCCESS_ZERO': 143,
  'PARTIAL': 106,
  'HTTP_ERROR': 1},
 'imported_news_selected': 8673,
 'executed_news_revalidated': 1234,
 'relations': 418824,
 'identity_relations': 223877,
 'semantic_relations': 194947,
 'relations_by_origin': {'IMPORT': 384245, 'EXECUTION': 34579},
 'technical_provisional_relations': 9374,
 'semantic_diagnostic_flags': 15946,
 'api_calls': 0,
 'output_directory': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/dados/processados/re/consolidado/ca93bcb5a6d6a8b46d89'),
 'package': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/re/Resumo_Consolidacao_RE_ca93bcb5a6d6_8ab929.zip'),
 'snapshot_sha256': 'ca93bcb5a6d6a8b46d89ab6392ec76d00b52dba36ba77638e6a58aea23413069'}

## 27. Integração da resolução de entidades

Associa os grupos documentados às menções e aos endpoints das relações consolidadas. Preserva os IDs e as marcações de revisão, registrando diagnósticos sem executar fusões ou agregar pesos.

In [2]:
from pathlib import Path
from entidades_integracao import aplicar_resolucao

resumo_entidades = aplicar_resolucao(Path.cwd())
resumo_entidades

[ENTIDADES] {"mentions": 579016, "groups": 95925, "review_mentions": 968, "relations_preserved": 418824, "relation_roles": {"SEMANTIC_BETWEEN_GROUPS": 194720, "IDENTITY_WITHIN_GROUP": 165188, "IDENTITY_BETWEEN_GROUPS": 58689, "SEMANTIC_WITHIN_GROUP": 227}, "groups_in_re": 49526, "groups_in_semantic_re": 40955, "relations_with_resolution_flags": 59306, "relations_touching_review_mentions": 660, "groups_merged": 0, "relations_removed": 0, "weights_aggregated": false, "api_calls": 0, "embeddings_generated": 0}
[ENTIDADES] Saídas: C:\Users\pedro\Documents\MO438\PROJECT\dados\processados\entidades\resolucao\338d4698428ef9f95162
[ENTIDADES] Pacote: C:\Users\pedro\Documents\MO438\PROJECT\resultados\entidades\Resumo_Integracao_Entidades_338d4698428e_b7438a.zip


{'mentions': 579016,
 'groups': 95925,
 'review_mentions': 968,
 'relations_preserved': 418824,
 'relation_roles': {'SEMANTIC_BETWEEN_GROUPS': 194720,
  'IDENTITY_WITHIN_GROUP': 165188,
  'IDENTITY_BETWEEN_GROUPS': 58689,
  'SEMANTIC_WITHIN_GROUP': 227},
 'groups_in_re': 49526,
 'groups_in_semantic_re': 40955,
 'relations_with_resolution_flags': 59306,
 'relations_touching_review_mentions': 660,
 'groups_merged': 0,
 'relations_removed': 0,
 'weights_aggregated': False,
 'api_calls': 0,
 'embeddings_generated': 0,
 'output_directory': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/dados/processados/entidades/resolucao/338d4698428ef9f95162'),
 'package': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/entidades/Resumo_Integracao_Entidades_338d4698428e_b7438a.zip'),
 'snapshot_sha256': '338d4698428ef9f951623f04a6ca3b468fa227455bb1c904ac6283aaf438c879'}

## 28. Construção da rede e proveniência documental

As entidades resolvidas são conectadas por relações dirigidas e tipadas. A construção exclui identidade, autolaços e grupos das fontes do corpus, preservando a auditoria. A agregação conserva o número de registros como peso e conta separadamente as notícias distintas que sustentam cada conexão. Tabelas de apoio mantêm os vínculos com títulos, links, datas, fontes e categorias jornalísticas.

In [1]:
from pathlib import Path
from redes_construcao import construir_rede

PROJETO_REDE = Path.cwd()
ASSINATURA_ENTIDADES_ESPERADA = (
    '338d4698428ef9f951623f04a6ca3b468fa227455bb1c904ac6283aaf438c879'
)

resumo_rede = construir_rede(
    PROJETO_REDE,
    assinatura_entidades_esperada=ASSINATURA_ENTIDADES_ESPERADA,
)
resumo_rede

[REDE] {"input_relations": 418824, "used_relations": 181348, "excluded_relations": 237476, "exclusive_exclusion_reasons": {"IDENTITY": 223877, "CORPUS_SOURCE": 13372, "SELF_LOOP": 227}, "nodes": 40627, "edges": 83188, "weight_sum": 181348, "support_news": 9113, "selected_news": 23657, "excluded_source_groups": 3, "directed": true, "multirelational": true, "self_loops": 0, "identity_edges": 0, "api_calls": 0, "inferred_relations_added": 0}
[REDE] Saídas: C:\Users\pedro\Documents\MO438\PROJECT\dados\processados\redes\instancias\3920a1985093835e545a
[REDE] Pacote: C:\Users\pedro\Documents\MO438\PROJECT\resultados\redes\Resumo_Construcao_Rede_3920a1985093_6f1d33.zip


{'input_relations': 418824,
 'used_relations': 181348,
 'excluded_relations': 237476,
 'exclusive_exclusion_reasons': {'IDENTITY': 223877,
  'CORPUS_SOURCE': 13372,
  'SELF_LOOP': 227},
 'nodes': 40627,
 'edges': 83188,
 'weight_sum': 181348,
 'support_news': 9113,
 'selected_news': 23657,
 'excluded_source_groups': 3,
 'directed': True,
 'multirelational': True,
 'self_loops': 0,
 'identity_edges': 0,
 'api_calls': 0,
 'inferred_relations_added': 0,
 'output_directory': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/dados/processados/redes/instancias/3920a1985093835e545a'),
 'package': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/redes/Resumo_Construcao_Rede_3920a1985093_6f1d33.zip'),
 'snapshot_sha256': '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'}

## 29. Verificação do ambiente Neo4j

A conferência registra a conexão, as versões, as redes e restrições já existentes e as capacidades GDS disponíveis. A rede de entrada é validada por sua assinatura e integridade. Nesta etapa não são alterados dados ou projeções do banco.

In [2]:
from pathlib import Path
from neo4j_integracao import verificar_ambiente_neo4j

PROJETO_NEO4J = Path.cwd()
ASSINATURA_REDE_ESPERADA = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)

resumo_neo4j = verificar_ambiente_neo4j(
    PROJETO_NEO4J,
    assinatura_rede_esperada=ASSINATURA_REDE_ESPERADA,
)
resumo_neo4j

Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `NewsAIEntity` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=29, offset=38>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 38, 'line': 2, 'column': 29}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "MATCH (n)\n        WHERE n:Entity OR n:NewsAIEntity\n        RETURN coalesce(toString(n.graph_tag), '__SEM_GRAPH_TAG__') AS graph_tag,\n               labels(n) AS labels, count(n) AS nodes\n        ORDER BY nodes DESC LIMIT 100"
Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_descri

[NEO4J] {"connection": "OK", "database": "neo4j", "target_graph_tag": "newsai_3920a1985093835e545a", "inspection_complete": true, "target_has_existing_data": false, "gds_capabilities": {"version_query_ok": true, "projection": true, "pagerank": true, "wcc": true, "louvain": true, "leiden": true}, "database_writes": 0, "load_executed": false}
[NEO4J] Pacote: C:\Users\pedro\Documents\MO438\PROJECT\resultados\neo4j\Resumo_Ambiente_Neo4j_20261119_190032_5ce111.zip


{'connection': 'OK',
 'database': 'neo4j',
 'target_graph_tag': 'newsai_3920a1985093835e545a',
 'inspection_complete': True,
 'target_has_existing_data': False,
 'gds_capabilities': {'version_query_ok': True,
  'projection': True,
  'pagerank': True,
  'wcc': True,
  'louvain': True,
  'leiden': True},
 'database_writes': 0,
 'load_executed': False,
 'package': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/neo4j/Resumo_Ambiente_Neo4j_20261119_190032_5ce111.zip'),
 'report': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/neo4j/Conferencia_Neo4j_20261119_190032_5ce111.json')}

### 29.1. Carga versionada da rede

A rede é importada em lotes, com identificação da edição e restrições de unicidade próprias. O armazenamento utiliza NewsAIEntity e NEWSAI_RELATES_TO para coexistir com outros dados. A conferência final compara IDs, atributos e extremos com as tabelas de entrada. Notícias de exemplo acompanham os elementos; a proveniência documental completa permanece nas tabelas de apoio.

In [6]:
from pathlib import Path
from neo4j_carga import carregar_rede_neo4j

PROJETO_NEO4J = Path.cwd()
ASSINATURA_REDE_ESPERADA = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)
EXECUTAR_CARGA_NEO4J = True
LOTE_CARGA_NEO4J = 1000

In [7]:
resumo_carga_neo4j = carregar_rede_neo4j(
    PROJETO_NEO4J,
    executar_carga=EXECUTAR_CARGA_NEO4J,
    assinatura_rede_esperada=ASSINATURA_REDE_ESPERADA,
    batch_size=LOTE_CARGA_NEO4J,
)
resumo_carga_neo4j

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `graph_tag` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=25, offset=24>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 24, 'line': 1, 'column': 25}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (n:NewsAINetwork {graph_tag:$tag}) RETURN properties(n) AS properties'
Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `NewsAINetwork` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<

[NEO4J] nodes: 1,000/40,627
[NEO4J] nodes: 2,000/40,627
[NEO4J] nodes: 3,000/40,627
[NEO4J] nodes: 4,000/40,627
[NEO4J] nodes: 5,000/40,627
[NEO4J] nodes: 6,000/40,627
[NEO4J] nodes: 7,000/40,627
[NEO4J] nodes: 8,000/40,627
[NEO4J] nodes: 9,000/40,627
[NEO4J] nodes: 10,000/40,627
[NEO4J] nodes: 11,000/40,627
[NEO4J] nodes: 12,000/40,627
[NEO4J] nodes: 13,000/40,627
[NEO4J] nodes: 14,000/40,627
[NEO4J] nodes: 15,000/40,627
[NEO4J] nodes: 16,000/40,627
[NEO4J] nodes: 17,000/40,627
[NEO4J] nodes: 18,000/40,627
[NEO4J] nodes: 19,000/40,627
[NEO4J] nodes: 20,000/40,627
[NEO4J] nodes: 21,000/40,627
[NEO4J] nodes: 22,000/40,627
[NEO4J] nodes: 23,000/40,627
[NEO4J] nodes: 24,000/40,627
[NEO4J] nodes: 25,000/40,627
[NEO4J] nodes: 26,000/40,627
[NEO4J] nodes: 27,000/40,627
[NEO4J] nodes: 28,000/40,627
[NEO4J] nodes: 29,000/40,627
[NEO4J] nodes: 30,000/40,627
[NEO4J] nodes: 31,000/40,627
[NEO4J] nodes: 32,000/40,627
[NEO4J] nodes: 33,000/40,627
[NEO4J] nodes: 34,000/40,627
[NEO4J] nodes: 35,000/4

{'schema': 'neo4j-network-load-1',
 'version': '1.0.0',
 'database': 'neo4j',
 'graph_tag': 'newsai_3920a1985093835e545a',
 'snapshot_sha256': '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a',
 'payload_sha256': 'a5b4842ccf17b8355a4c0a2304bb2e611258ea26f197b44f00ec8f41185f3958',
 'node_label': 'NewsAIEntity',
 'relationship_type': 'NEWSAI_RELATES_TO',
 'created_at_utc': '2026-11-19T19:27:06.517305+00:00',
 'nodes': 40627,
 'edges': 83188,
 'weight_sum': 181348,
 'all_properties_verified': True,
 'self_loops': 0,
 'identity_edges': 0,
 'reused_completed_load': False,
 'batches_written': 125,
 'legacy_data_deleted': False,
 'legacy_constraints_removed': False,
 'gds_projections_created': 0,
 'api_calls_ner_re': 0,
 'package': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/neo4j/Resumo_Carga_Neo4j_20261119_192706_47dd4f.zip')}

## 30. Projeções para análise de redes

A representação dirigida conserva as arestas relacionais e seus pesos. A projeção estrutural reúne cada par não ordenado em uma conexão unitária. A projeção de comunidades reúne o mesmo par e soma os pesos dos registros. A rede original permanece preservada, e tabelas de correspondência permitem recuperar os tipos de relação e as notícias de suporte. As projeções são conferidas por nós, extremos, multiplicidades e propriedades.

In [10]:
from pathlib import Path
from redes_projecoes import criar_projecoes_gds

PROJETO_GDS = Path.cwd()
ASSINATURA_REDE_ESPERADA = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)
EXECUTAR_PROJECOES_GDS = True

In [11]:
resumo_projecoes = criar_projecoes_gds(
    PROJETO_GDS,
    executar_gds=EXECUTAR_PROJECOES_GDS,
    assinatura_rede_esperada=ASSINATURA_REDE_ESPERADA,
)
resumo_projecoes

[GDS] directed {"graphName": "newsai_3920a1985093835e545a_directed_c31d0dcb", "nodeCount": 40627, "relationshipCount": 83188, "database": "neo4j", "memoryUsage": "173 KiB", "weight_sum": 181348.0, "all_nodes_checked": true, "all_relationship_endpoints_and_properties_checked": true, "reused_existing_projection": false}
[GDS] structural {"graphName": "newsai_3920a1985093835e545a_structural_c31d0dcb", "nodeCount": 40627, "relationshipCount": 115072, "database": "neo4j", "memoryUsage": "173 KiB", "weight_sum": 115072.0, "all_nodes_checked": true, "all_relationship_endpoints_and_properties_checked": true, "reused_existing_projection": false}
[GDS] community {"graphName": "newsai_3920a1985093835e545a_community_c31d0dcb", "nodeCount": 40627, "relationshipCount": 115072, "database": "neo4j", "memoryUsage": "173 KiB", "weight_sum": 362696.0, "all_nodes_checked": true, "all_relationship_endpoints_and_properties_checked": true, "reused_existing_projection": false}
[GDS] Pacote: C:\Users\pedro\Doc

{'nodes': 40627,
 'typed_directed_edges': 83188,
 'distinct_directed_pairs': 79647,
 'undirected_pairs': 57536,
 'record_weight_sum': 181348,
 'expected_gds_relationships': {'directed': 83188,
  'structural': 115072,
  'community': 115072},
 'expected_gds_weight_sums': {'directed': 181348,
  'structural': 115072,
  'community': 362696},
 'graph_names': {'directed': 'newsai_3920a1985093835e545a_directed_c31d0dcb',
  'structural': 'newsai_3920a1985093835e545a_structural_c31d0dcb',
  'community': 'newsai_3920a1985093835e545a_community_c31d0dcb'},
 'projections': {'directed': {'graphName': 'newsai_3920a1985093835e545a_directed_c31d0dcb',
   'nodeCount': 40627,
   'relationshipCount': 83188,
   'database': 'neo4j',
   'memoryUsage': '173 KiB',
   'weight_sum': 181348.0,
   'all_nodes_checked': True,
   'all_relationship_endpoints_and_properties_checked': True,
   'reused_existing_projection': False},
  'structural': {'graphName': 'newsai_3920a1985093835e545a_structural_c31d0dcb',
   'nodeCo

## 31. Caracterização estrutural da rede

A análise utiliza a projeção simples, não dirigida e não ponderada. São calculados graus, densidade, componentes, triângulos e coeficientes de clusterização. O grau corresponde ao número de vizinhos distintos. A média principal de clusterização considera nós com grau ≥ 2; a média sobre todos os nós e a transitividade são reportadas separadamente. Os resultados GDS são conferidos por uma implementação independente sobre as arestas locais.

In [14]:
from pathlib import Path
from redes_estrutural import caracterizar_rede

PROJETO_ESTRUTURA = Path.cwd()
ASSINATURA_REDE_ESPERADA = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)
EXECUTAR_ESTRUTURA_GDS = True

In [15]:
resumo_estrutura = caracterizar_rede(
    PROJETO_ESTRUTURA,
    executar_gds=EXECUTAR_ESTRUTURA_GDS,
    assinatura_rede_esperada=ASSINATURA_REDE_ESPERADA,
)
resumo_estrutura

[ESTRUTURA] degree: 40,627 nós; 2.42s.
[ESTRUTURA] triangles: 40,627 nós; 2.43s.
[ESTRUTURA] components: 40,627 nós; 2.48s.
[ESTRUTURA] {"nodes": 40627, "simple_undirected_edges": 57536, "degree_sum": 115072, "mean_degree": 2.832402097127526, "density": 6.971895084742594e-05, "min_degree": 1, "max_degree": 1796, "median_degree": 1.0, "isolated_nodes": 0, "degree_one_nodes": 23080, "degree_le_two_nodes": 31925, "nodes_degree_ge_two": 17547, "triangles": 17125, "connected_triples": 5181743, "mean_local_clustering_degree_ge_two": 0.243565178343415, "mean_local_clustering_all_nodes": 0.10519699176389848, "transitivity": 0.009914617533135085, "zero_clustering_nodes": 32458, "connected_components": 3259, "largest_component_nodes": 32575, "largest_component_share": 0.8018066802865089, "distance_metrics_computed": false, "community_detection_computed": false, "independent_topology_verified": true}
[ESTRUTURA] Pacote: C:\Users\pedro\Documents\MO438\PROJECT\resultados\estrutural\Resumo_Estrutura

{'nodes': 40627,
 'simple_undirected_edges': 57536,
 'degree_sum': 115072,
 'mean_degree': 2.832402097127526,
 'density': 6.971895084742594e-05,
 'min_degree': 1,
 'max_degree': 1796,
 'median_degree': 1.0,
 'isolated_nodes': 0,
 'degree_one_nodes': 23080,
 'degree_le_two_nodes': 31925,
 'nodes_degree_ge_two': 17547,
 'triangles': 17125,
 'connected_triples': 5181743,
 'mean_local_clustering_degree_ge_two': 0.243565178343415,
 'mean_local_clustering_all_nodes': 0.10519699176389848,
 'transitivity': 0.009914617533135085,
 'zero_clustering_nodes': 32458,
 'connected_components': 3259,
 'largest_component_nodes': 32575,
 'largest_component_share': 0.8018066802865089,
 'distance_metrics_computed': False,
 'community_detection_computed': False,
 'independent_topology_verified': True,
 'output_directory': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/estrutural/4d84ef2792a01fccd0a1'),
 'package': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/estrutural/Resum

## 32. Distâncias e diâmetro na maior componente

As distâncias são medidas pelo número de arestas na projeção simples, não dirigida e não ponderada. Calculamos a distância média entre vértices distintos, as excentricidades, o diâmetro e o raio da maior componente conexa. O processamento exato utiliza todas as fontes, em blocos com salvamento para retomada. Resultados parciais ficam identificados e não são tratados como métricas finais.

In [20]:
from pathlib import Path
from redes_distancias import calcular_distancias

PROJETO_DISTANCIAS = Path.cwd()
ASSINATURA_REDE_DISTANCIAS = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)
EXECUTAR_DISTANCIAS = True
MAX_FONTES_DISTANCIAS = None # None = execução completa
TAMANHO_BLOCO_DISTANCIAS = 32

In [21]:
resumo_distancias = calcular_distancias(
    PROJETO_DISTANCIAS,
    executar=EXECUTAR_DISTANCIAS,
    max_fontes=MAX_FONTES_DISTANCIAS,
    tamanho_bloco=TAMANHO_BLOCO_DISTANCIAS,
    assinatura_rede_esperada=ASSINATURA_REDE_DISTANCIAS,
)
resumo_distancias

[DISTÂNCIAS] Retomada: 16/32,575 fontes; até 32,559 novas fontes nesta execução.
[DISTÂNCIAS] 48/32,575 fontes salvas; bloco: 0.31s; tempo restante indicativo: 5.4 min.
[DISTÂNCIAS] 80/32,575 fontes salvas; bloco: 0.26s; tempo restante indicativo: 5.0 min.
[DISTÂNCIAS] 112/32,575 fontes salvas; bloco: 0.29s; tempo restante indicativo: 5.0 min.
[DISTÂNCIAS] 144/32,575 fontes salvas; bloco: 0.29s; tempo restante indicativo: 5.0 min.
[DISTÂNCIAS] 176/32,575 fontes salvas; bloco: 0.29s; tempo restante indicativo: 4.9 min.
[DISTÂNCIAS] 208/32,575 fontes salvas; bloco: 0.30s; tempo restante indicativo: 4.9 min.
[DISTÂNCIAS] 240/32,575 fontes salvas; bloco: 0.30s; tempo restante indicativo: 5.0 min.
[DISTÂNCIAS] 272/32,575 fontes salvas; bloco: 0.27s; tempo restante indicativo: 4.9 min.
[DISTÂNCIAS] 304/32,575 fontes salvas; bloco: 0.31s; tempo restante indicativo: 4.9 min.
[DISTÂNCIAS] 336/32,575 fontes salvas; bloco: 0.34s; tempo restante indicativo: 5.0 min.
[DISTÂNCIAS] 368/32,575 fontes 

{'status': 'COMPLETE',
 'gcc_nodes': 32575,
 'gcc_edges': 52593,
 'sources_completed': 32575,
 'ordered_pairs_evaluated': 1061098050,
 'expected_ordered_pairs': 1061098050,
 'distance_unit': 'number_of_edges',
 'average_distance': 4.859551407148472,
 'diameter': 20,
 'radius': 10,
 'diameter_lower_bound': 20,
 'diameter_upper_bound': 20,
 'radius_lower_bound': 10,
 'radius_upper_bound': 10,
 'computation_seconds': 348.9537694000064,
 'exact_all_sources': True,
 'api_calls': 0,
 'neo4j_connections': 0,
 'independent_bfs_sources_checked': 3,
 'estimated_remaining_compute_seconds': 0.0,
 'output_directory': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/distancias/instancias/ccd0f7e8a03250ff3135'),
 'checkpoint_directory': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/distancias/execucoes/ccd0f7e8a03250ff3135/blocos'),
 'package': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/distancias/Resumo_Distancias_ccd0f7e8a032_330850.zip'),
 'snapsh

## 33. Conectividade, força e PageRank das entidades

Analisamos o protagonismo estrutural dos grupos de entidades por diversidade de vizinhos, número de arestas tipadas, frequência relacional e PageRank ponderado dirigido. Os resultados preservam os IDs e os atributos de cobertura e qualidade. As métricas não demonstram influência social ou causalidade. O PageRank calculado no GDS é confrontado, em todos os nós, com uma implementação local independente.

In [24]:
from pathlib import Path
from redes_centralidades import analisar_centralidades

PROJETO_CENTRALIDADES = Path.cwd()
ASSINATURA_REDE_CENTRALIDADES = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)
EXECUTAR_CENTRALIDADES_GDS = True

In [25]:
resumo_centralidades = analisar_centralidades(
    PROJETO_CENTRALIDADES,
    executar_gds=EXECUTAR_CENTRALIDADES_GDS,
    assinatura_rede_esperada=ASSINATURA_REDE_CENTRALIDADES,
)
resumo_centralidades

[CENTRALIDADES] PageRank: 40,627 nós; 3.92s.
[CENTRALIDADES] {"nodes": 40627, "typed_directed_edges": 83188, "undirected_pairs": 57536, "distinct_directed_pairs": 79647, "record_weight_sum": 181348, "in_typed_degree_sum": 83188, "out_typed_degree_sum": 83188, "typed_degree_sum": 166376, "neighbor_degree_sum": 115072, "in_strength_sum": 181348, "out_strength_sum": 181348, "weighted_strength_sum": 362696, "zero_out_strength_nodes": 6334, "pagerank_sum": 29974.708552195316, "pagerank_share_sum": 1.0, "pagerank_max": 971.9886862412459, "all_pagerank_nodes_verified": true, "pagerank_gds_seconds": 3.9216455000023416, "betweenness_computed": false, "closeness_computed": false, "community_detection_computed": false, "api_calls": 0, "database_writes": 0}
[CENTRALIDADES] Pacote: C:\Users\pedro\Documents\MO438\PROJECT\resultados\centralidades\Resumo_Centralidades_7c10fb53a917_cecc65.zip


{'nodes': 40627,
 'typed_directed_edges': 83188,
 'undirected_pairs': 57536,
 'distinct_directed_pairs': 79647,
 'record_weight_sum': 181348,
 'in_typed_degree_sum': 83188,
 'out_typed_degree_sum': 83188,
 'typed_degree_sum': 166376,
 'neighbor_degree_sum': 115072,
 'in_strength_sum': 181348,
 'out_strength_sum': 181348,
 'weighted_strength_sum': 362696,
 'zero_out_strength_nodes': 6334,
 'pagerank_sum': 29974.708552195316,
 'pagerank_share_sum': 1.0,
 'pagerank_max': 971.9886862412459,
 'all_pagerank_nodes_verified': True,
 'pagerank_gds_seconds': 3.9216455000023416,
 'betweenness_computed': False,
 'closeness_computed': False,
 'community_detection_computed': False,
 'api_calls': 0,
 'database_writes': 0,
 'output_directory': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/centralidades/7c10fb53a9179e7d8e35'),
 'package': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/centralidades/Resumo_Centralidades_7c10fb53a917_cecc65.zip'),
 'snapshot_sha256': '7c1

## 34. Detecção e avaliação de comunidades

Utilizamos a projeção não dirigida ponderada para identificar comunidades com Louvain e comparar os resultados com Leiden. Louvain é a referência; três sementes Leiden permitem examinar a sensibilidade inicial das partições. Avaliamos modularidade, tamanhos, composição por tipo de entidade, conectividade interna e concordância por ARI e NMI. Nenhuma dessas medidas, isoladamente, confirma coerência temática ou validade científica das comunidades.

In [28]:
from pathlib import Path
from redes_comunidades import detectar_comunidades

PROJETO_COMUNIDADES = Path.cwd()
ASSINATURA_REDE_COMUNIDADES = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)
EXECUTAR_COMUNIDADES_GDS = True
SEMENTES_LEIDEN = [42, 43, 44]

In [29]:
resumo_comunidades = detectar_comunidades(
    PROJETO_COMUNIDADES,
    executar_gds=EXECUTAR_COMUNIDADES_GDS,
    sementes_leiden=SEMENTES_LEIDEN,
    assinatura_rede_esperada=ASSINATURA_REDE_COMUNIDADES,
)
resumo_comunidades

[COMUNIDADES] louvain: 40,627 nós; 3,407 comunidades; 4.19s; partição salva.
[COMUNIDADES] leiden_seed_42: 40,627 nós; 3,517 comunidades; 2.63s; partição salva.
[COMUNIDADES] leiden_seed_43: 40,627 nós; 3,539 comunidades; 2.27s; partição salva.
[COMUNIDADES] leiden_seed_44: 40,627 nós; 3,541 comunidades; 1.97s; partição salva.
[COMUNIDADES] {"nodes": 40627, "undirected_pairs": 57536, "run_count": 4, "reference_partition": "louvain", "reference_leiden": "leiden_seed_42", "partitions": [{"run_id": "louvain", "nodes": 40627, "communities": 3407, "weighted_modularity": 0.7979018315456122, "largest_community_nodes": 3480, "singleton_communities": 0, "disconnected_communities": 12, "communities_spanning_original_components": 0, "total_unmirrored_weight": 181348.0, "gcc_communities": 149, "algorithm_seconds": 4.185445499999332}, {"run_id": "leiden_seed_42", "nodes": 40627, "communities": 3517, "weighted_modularity": 0.7913986198658564, "largest_community_nodes": 5032, "singleton_communities":

{'nodes': 40627,
 'undirected_pairs': 57536,
 'run_count': 4,
 'reference_partition': 'louvain',
 'reference_leiden': 'leiden_seed_42',
 'partitions': [{'run_id': 'louvain',
   'nodes': 40627,
   'communities': 3407,
   'weighted_modularity': 0.7979018315456122,
   'largest_community_nodes': 3480,
   'singleton_communities': 0,
   'disconnected_communities': 12,
   'communities_spanning_original_components': 0,
   'total_unmirrored_weight': 181348.0,
   'gcc_communities': 149,
   'algorithm_seconds': 4.185445499999332},
  {'run_id': 'leiden_seed_42',
   'nodes': 40627,
   'communities': 3517,
   'weighted_modularity': 0.7913986198658564,
   'largest_community_nodes': 5032,
   'singleton_communities': 0,
   'disconnected_communities': 0,
   'communities_spanning_original_components': 0,
   'total_unmirrored_weight': 181348.0,
   'gcc_communities': 259,
   'algorithm_seconds': 2.6338699999978417},
  {'run_id': 'leiden_seed_43',
   'nodes': 40627,
   'communities': 3539,
   'weighted_modu

## 35. Caracterização editorial e visualização das comunidades

A visualização principal utiliza a mesma projeção não dirigida e ponderada empregada na detecção de comunidades. Nas vistas globais, as cores indicam as comunidades; nos subgrafos de uma comunidade, indicam inicialmente os tipos de entidade. Uma opção complementar permite inspecionar as direções semânticas originais dos mesmos pares. As notícias de suporte são acessíveis por exemplos, listas completas e exportação, com contagem por união de IDs. A apresentação não modifica as partições ou os resultados científicos anteriores.

In [40]:
from pathlib import Path
from comunidades_visualizacao import gerar_visualizacoes

PROJETO_VISUALIZACOES = Path.cwd()
ASSINATURA_REDE_VISUALIZACOES = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)
EXECUTAR_VISUALIZACOES = True
COMUNIDADES_POR_PARTICAO = 3
MAX_NOS_POR_VISTA = 100
MAX_PARES_NAO_DIRIGIDOS_POR_VISTA = 250
MANTER_APENAS_VISUALIZACAO_ATUAL = True

In [41]:
resumo_visualizacoes = gerar_visualizacoes(
    PROJETO_VISUALIZACOES,
    executar=EXECUTAR_VISUALIZACOES,
    comunidades_por_particao=COMUNIDADES_POR_PARTICAO,
    max_nos=MAX_NOS_POR_VISTA,
    max_arestas=MAX_PARES_NAO_DIRIGIDOS_POR_VISTA,
    assinatura_rede_esperada=ASSINATURA_REDE_VISUALIZACOES,
    manter_apenas_atual=MANTER_APENAS_VISUALIZACAO_ATUAL,
)
resumo_visualizacoes

[VISUALIZAÇÕES] grafo_louvain_global 100 nós; 250 pares não dirigidos; 474 direções para inspeção; arquivos salvos.
[VISUALIZAÇÕES] grafo_louvain_C19 100 nós; 250 pares não dirigidos; 441 direções para inspeção; arquivos salvos.
[VISUALIZAÇÕES] grafo_louvain_C91 100 nós; 250 pares não dirigidos; 467 direções para inspeção; arquivos salvos.
[VISUALIZAÇÕES] grafo_louvain_C2 100 nós; 250 pares não dirigidos; 461 direções para inspeção; arquivos salvos.
[VISUALIZAÇÕES] grafo_leiden_seed_42_global 100 nós; 250 pares não dirigidos; 474 direções para inspeção; arquivos salvos.
[VISUALIZAÇÕES] grafo_leiden_seed_42_C2 100 nós; 250 pares não dirigidos; 469 direções para inspeção; arquivos salvos.
[VISUALIZAÇÕES] grafo_leiden_seed_42_C19 100 nós; 250 pares não dirigidos; 446 direções para inspeção; arquivos salvos.
[VISUALIZAÇÕES] grafo_leiden_seed_42_C12 100 nós; 192 pares não dirigidos; 331 direções para inspeção; arquivos salvos.
[VISUALIZAÇÕES] LIMPEZA — {"removed": ["f5d8a792601ec72a6005", "

{'partitions': ['louvain', 'leiden_seed_42'],
 'community_rows': 6924,
 'support_news': 9113,
 'community_news_links': 63227,
 'graph_views': 8,
 'html_files': 9,
 'graph_figure_files': 24,
 'editorial_figure_files': 6,
 'complete_news_lists': True,
 'external_assets_required': False,
 'default_representation': 'UNDIRECTED_WEIGHTED',
 'directed_semantic_inspection': True,
 'api_calls': 0,
 'neo4j_connections': 0,
 'database_writes': 0,
 'partitions_modified': False,
 'output_directory': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/visualizacoes/bb9d444ccfa56bf35025'),
 'index_html': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/visualizacoes/bb9d444ccfa56bf35025/indice_comunidades.html'),
 'package': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/visualizacoes/Resumo_Visualizacoes_bb9d444ccfa5_04b568.zip'),
 'snapshot_sha256': 'bb9d444ccfa56bf3502510540a75d9bdf6fbee7be60c87891da130c1a70ddb9b',
 'cleanup': {'removed': ['f5d8a792601ec72a

## 36. Cobertura documental para comparação entre fontes

A comparação entre fontes exige distinguir a quantidade de notícias do corpus, a conclusão da extração de relações e o suporte documental à rede final. Esta seção caracteriza essas quantidades por fonte e ano, preserva os estados individuais de processamento e identifica anos com suporte nas três fontes. As proporções utilizam o corpus selecionado como denominador; não representam a totalidade das publicações dos veículos. O diagnóstico fundamenta a escolha de um período comum e a interpretação das limitações de cobertura.

In [45]:
from pathlib import Path
from redes_cobertura import gerar_cobertura

PROJETO_COBERTURA = Path.cwd()
EXECUTAR_COBERTURA = True
ASSINATURA_REDE_COBERTURA = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)

In [46]:
resumo_cobertura = gerar_cobertura(
    PROJETO_COBERTURA,
    executar=EXECUTAR_COBERTURA,
    assinatura_rede_esperada=ASSINATURA_REDE_COBERTURA,
)
resumo_cobertura

[COBERTURA] {"corpus_news": 23657, "support_news": 9113, "retained_occurrences": 181348, "re_completed_strict": 9800, "re_partial": 106, "common_years_with_network_support": [2022, 2023, 2024, 2025], "automatic_period_selection": false, "api_calls": 0, "neo4j_connections": 0, "database_writes": 0, "network_modified": false}
[COBERTURA] Pacote: C:\Users\pedro\Documents\MO438\PROJECT\resultados\comparacao_fontes\cobertura\Resumo_Cobertura_Fontes_15e5f7a1e871_1d6212.zip


{'corpus_news': 23657,
 'support_news': 9113,
 'retained_occurrences': 181348,
 're_completed_strict': 9800,
 're_partial': 106,
 'common_years_with_network_support': [2022, 2023, 2024, 2025],
 'automatic_period_selection': False,
 'api_calls': 0,
 'neo4j_connections': 0,
 'database_writes': 0,
 'network_modified': False,
 'output_directory': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/comparacao_fontes/cobertura/15e5f7a1e87197c8f31e'),
 'package': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/comparacao_fontes/cobertura/Resumo_Cobertura_Fontes_15e5f7a1e871_1d6212.zip'),
 'snapshot_sha256': '15e5f7a1e87197c8f31e12971f6a05fdf7de3f14def50c8c54e487cbe9099fa6'}

## 37. Construção das redes por fonte no período comum

São construídas redes independentes para CNN Brasil, Folha de S.Paulo e G1, no período de 1º de janeiro a 31 de dezembro de 2025. Os identificadores das entidades e dos tipos de relação são compartilhados com a rede global. Pesos, atributos de frequência e suporte documental são recalculados exclusivamente a partir das notícias de cada fonte no período. As redes preservam a direção e os rótulos semânticos; projeções simples e ponderadas não dirigidas são exportadas para a comparação estrutural e a análise de comunidades. As diferenças de cobertura da extração são documentadas e devem acompanhar a interpretação das comparações.

In [49]:
from pathlib import Path
from redes_fontes import construir_redes_fontes

PROJETO_FONTES = Path.cwd()
EXECUTAR_REDES_FONTES = True
DATA_INICIAL_FONTES = '2025-01-01'
DATA_FINAL_FONTES = '2025-12-31'
ASSINATURA_REDE_FONTES = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)

In [50]:
resumo_redes_fontes = construir_redes_fontes(
    PROJETO_FONTES,
    executar=EXECUTAR_REDES_FONTES,
    data_inicial=DATA_INICIAL_FONTES,
    data_final=DATA_FINAL_FONTES,
    assinatura_rede_esperada=ASSINATURA_REDE_FONTES,
)
resumo_redes_fontes

[FONTES] {"source_key": "cnn", "source": "CNN Brasil", "selected_news": 2087, "re_completed_strict": 1885, "re_partial": 0, "support_news": 1750, "nodes": 6517, "typed_directed_edges": 11839, "distinct_directed_pairs": 11414, "undirected_pairs": 8203, "weight_sum": 27163, "self_loops": 0, "identity_edges": 0, "observed_min_date": "2025-01-01", "observed_max_date": "2025-12-30"}
[FONTES] {"source_key": "folha", "source": "Folha de S.Paulo", "selected_news": 3413, "re_completed_strict": 3104, "re_partial": 0, "support_news": 2887, "nodes": 15567, "typed_directed_edges": 29323, "distinct_directed_pairs": 28138, "undirected_pairs": 20601, "weight_sum": 61082, "self_loops": 0, "identity_edges": 0, "observed_min_date": "2025-01-01", "observed_max_date": "2025-12-31"}
[FONTES] {"source_key": "g1", "source": "G1", "selected_news": 2115, "re_completed_strict": 1990, "re_partial": 0, "support_news": 1874, "nodes": 11578, "typed_directed_edges": 23137, "distinct_directed_pairs": 22325, "undirecte

{'networks': [{'source_key': 'cnn',
   'source': 'CNN Brasil',
   'selected_news': 2087,
   're_completed_strict': 1885,
   're_partial': 0,
   'support_news': 1750,
   'nodes': 6517,
   'typed_directed_edges': 11839,
   'distinct_directed_pairs': 11414,
   'undirected_pairs': 8203,
   'weight_sum': 27163,
   'self_loops': 0,
   'identity_edges': 0,
   'observed_min_date': '2025-01-01',
   'observed_max_date': '2025-12-30'},
  {'source_key': 'folha',
   'source': 'Folha de S.Paulo',
   'selected_news': 3413,
   're_completed_strict': 3104,
   're_partial': 0,
   'support_news': 2887,
   'nodes': 15567,
   'typed_directed_edges': 29323,
   'distinct_directed_pairs': 28138,
   'undirected_pairs': 20601,
   'weight_sum': 61082,
   'self_loops': 0,
   'identity_edges': 0,
   'observed_min_date': '2025-01-01',
   'observed_max_date': '2025-12-31'},
  {'source_key': 'g1',
   'source': 'G1',
   'selected_news': 2115,
   're_completed_strict': 1990,
   're_partial': 0,
   'support_news': 1874,

## 38. Comparação estrutural das redes por fonte

As redes de 2025 são comparadas por graus, densidade, componentes, triângulos, clusterização, reciprocidade e sobreposição de entidades e conexões. Graus, componentes e triângulos são calculados no Neo4j/GDS sobre projeções simples não dirigidas e conferidos integralmente por uma implementação independente. A reciprocidade considera pares dirigidos distintos. A sobreposição utiliza os identificadores compartilhados das entidades e das arestas. As diferenças de tamanho, composição e cobertura da extração permanecem limitações da comparação.

In [53]:
from pathlib import Path
from fontes_estrutural import comparar_estrutura_fontes

PROJETO_COMPARACAO = Path.cwd()
EXECUTAR_COMPARACAO_GDS = True
ASSINATURA_REDE_COMPARACAO = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)
ASSINATURA_FONTES_COMPARACAO = (
    'e5c458298e07becc955509750fdd3b43be2e72b3df621c782cb1d330205080d6'
)

In [54]:
resumo_comparacao = comparar_estrutura_fontes(
    PROJETO_COMPARACAO,
    executar_gds=EXECUTAR_COMPARACAO_GDS,
    assinatura_rede_esperada=ASSINATURA_REDE_COMPARACAO,
    assinatura_fontes_esperada=ASSINATURA_FONTES_COMPARACAO,
)
resumo_comparacao

[ESTRUTURA] degree: 6,517 nós; 0.42s.
[ESTRUTURA] triangles: 6,517 nós; 0.42s.
[ESTRUTURA] components: 6,517 nós; 0.46s.
[COMPARAÇÃO] {"source_key": "cnn", "source": "CNN Brasil", "selected_news": 2087, "re_completed_strict": 1885, "re_partial": 0, "support_news": 1750, "nodes": 6517, "typed_directed_edges": 11839, "distinct_directed_pairs": 11414, "undirected_pairs": 8203, "weight_sum": 27163, "self_loops": 0, "identity_edges": 0, "observed_min_date": "2025-01-01", "observed_max_date": "2025-12-30", "simple_undirected_edges": 8203, "degree_sum": 16406, "mean_degree": 2.5174159889519716, "density": 0.00038634376748802516, "min_degree": 1, "max_degree": 376, "median_degree": 1.0, "isolated_nodes": 0, "degree_one_nodes": 3783, "degree_le_two_nodes": 5183, "nodes_degree_ge_two": 2734, "triangles": 1523, "connected_triples": 227098, "mean_local_clustering_degree_ge_two": 0.21753682886871392, "mean_local_clustering_all_nodes": 0.09126065522894952, "transitivity": 0.02011906753912408, "zero_

{'networks': [{'source_key': 'cnn',
   'source': 'CNN Brasil',
   'selected_news': 2087,
   're_completed_strict': 1885,
   're_partial': 0,
   'support_news': 1750,
   'nodes': 6517,
   'typed_directed_edges': 11839,
   'distinct_directed_pairs': 11414,
   'undirected_pairs': 8203,
   'weight_sum': 27163,
   'self_loops': 0,
   'identity_edges': 0,
   'observed_min_date': '2025-01-01',
   'observed_max_date': '2025-12-30',
   'simple_undirected_edges': 8203,
   'degree_sum': 16406,
   'mean_degree': 2.5174159889519716,
   'density': 0.00038634376748802516,
   'min_degree': 1,
   'max_degree': 376,
   'median_degree': 1.0,
   'isolated_nodes': 0,
   'degree_one_nodes': 3783,
   'degree_le_two_nodes': 5183,
   'nodes_degree_ge_two': 2734,
   'triangles': 1523,
   'connected_triples': 227098,
   'mean_local_clustering_degree_ge_two': 0.21753682886871392,
   'mean_local_clustering_all_nodes': 0.09126065522894952,
   'transitivity': 0.02011906753912408,
   'zero_clustering_nodes': 5386,
  

## 39. Protagonismo estrutural das entidades por fonte

São comparados o grau de vizinhança, a força das relações e o PageRank ponderado das entidades nas redes de 2025. O grau mede vizinhos distintos na projeção simples; a força soma as ocorrências relacionais de entrada e saída; o PageRank utiliza a direção e os pesos das relações. Os rankings são apresentados globalmente e por tipo de entidade, com identificadores compartilhados entre fontes e acesso às notícias de suporte. Todos os escores PageRank calculados no GDS são confrontados com uma referência independente. As medidas caracterizam protagonismo estrutural na representação extraída, sem equivaler diretamente a influência social ou importância jornalística.

In [63]:
from pathlib import Path
from fontes_centralidades import analisar_centralidades_fontes

PROJETO_PROTAGONISMO = Path.cwd()
EXECUTAR_PROTAGONISMO_GDS = True
ASSINATURA_REDE_PROTAGONISMO = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)
ASSINATURA_FONTES_PROTAGONISMO = (
    'e5c458298e07becc955509750fdd3b43be2e72b3df621c782cb1d330205080d6'
)

In [64]:
resumo_protagonismo = analisar_centralidades_fontes(
    PROJETO_PROTAGONISMO,
    executar_gds=EXECUTAR_PROTAGONISMO_GDS,
    assinatura_rede_esperada=ASSINATURA_REDE_PROTAGONISMO,
    assinatura_fontes_esperada=ASSINATURA_FONTES_PROTAGONISMO,
)
resumo_protagonismo

[PROTAGONISMO] PROJEÇÃO — CNN Brasil ; leitura com 1 trabalhador; liberação após PageRank.
[CENTRALIDADES] PageRank: 6,517 nós; 1.01s.
[PROTAGONISMO] PROJEÇÃO LIBERADA — newsai_fontes_e5c458298e07_cnn_directed_423745f7
[PROTAGONISMO] {"source_key": "cnn", "source": "CNN Brasil", "nodes": 6517, "typed_directed_edges": 11839, "undirected_pairs": 8203, "distinct_directed_pairs": 11414, "record_weight_sum": 27163, "in_typed_degree_sum": 11839, "out_typed_degree_sum": 11839, "typed_degree_sum": 23678, "neighbor_degree_sum": 16406, "in_strength_sum": 27163, "out_strength_sum": 27163, "weighted_strength_sum": 54326, "zero_out_strength_nodes": 1151, "pagerank_sum": 4566.399888476602, "pagerank_share_sum": 1.0, "pagerank_max": 183.9345899918545, "all_pagerank_nodes_verified": true, "support_links": 15064, "support_news": 1750}
[PROTAGONISMO] PROJEÇÃO — Folha de S.Paulo ; leitura com 1 trabalhador; liberação após PageRank.
[CENTRALIDADES] PageRank: 15,567 nós; 1.30s.
[PROTAGONISMO] PROJEÇÃO LIBE

{'networks': [{'source_key': 'cnn',
   'source': 'CNN Brasil',
   'nodes': 6517,
   'typed_directed_edges': 11839,
   'undirected_pairs': 8203,
   'distinct_directed_pairs': 11414,
   'record_weight_sum': 27163,
   'in_typed_degree_sum': 11839,
   'out_typed_degree_sum': 11839,
   'typed_degree_sum': 23678,
   'neighbor_degree_sum': 16406,
   'in_strength_sum': 27163,
   'out_strength_sum': 27163,
   'weighted_strength_sum': 54326,
   'zero_out_strength_nodes': 1151,
   'pagerank_sum': 4566.399888476602,
   'pagerank_share_sum': 1.0,
   'pagerank_max': 183.9345899918545,
   'all_pagerank_nodes_verified': True,
   'support_links': 15064,
   'support_news': 1750},
  {'source_key': 'folha',
   'source': 'Folha de S.Paulo',
   'nodes': 15567,
   'typed_directed_edges': 29323,
   'undirected_pairs': 20601,
   'distinct_directed_pairs': 28138,
   'record_weight_sum': 61082,
   'in_typed_degree_sum': 29323,
   'out_typed_degree_sum': 29323,
   'typed_degree_sum': 58646,
   'neighbor_degree_su

## 40. Detecção e comparação de comunidades por fonte

As redes das três fontes no período comum de 2025 são analisadas com Louvain e Leiden em projeções ponderadas não dirigidas, agregando rótulos e ambas as direções. Leiden é executado com três sementes para examinar estabilidade. São avaliados modularidade, conectividade interna, composição por tipo de entidade e entidades de destaque. As partições são comparadas dentro de cada fonte e, entre fontes, somente sobre as entidades compartilhadas. Modularidade ou concordância elevadas não garantem validade temática, que requer inspeção das entidades, relações e notícias de suporte.

In [67]:
from pathlib import Path
from fontes_comunidades import detectar_comunidades_fontes

PROJETO_COMUNIDADES_FONTES = Path.cwd()
EXECUTAR_COMUNIDADES_FONTES_GDS = True
SEMENTES_LEIDEN_FONTES = (42, 43, 44)
ASSINATURA_REDE_COMUNIDADES_FONTES = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)
ASSINATURA_FONTES_COMUNIDADES = (
    'e5c458298e07becc955509750fdd3b43be2e72b3df621c782cb1d330205080d6'
)

In [68]:
resumo_comunidades_fontes = detectar_comunidades_fontes(
    PROJETO_COMUNIDADES_FONTES,
    executar_gds=EXECUTAR_COMUNIDADES_FONTES_GDS,
    sementes_leiden=SEMENTES_LEIDEN_FONTES,
    assinatura_rede_esperada=ASSINATURA_REDE_COMUNIDADES_FONTES,
    assinatura_fontes_esperada=ASSINATURA_FONTES_COMUNIDADES,
)
resumo_comunidades_fontes

[COMUNIDADES FONTES] CNN Brasil louvain — 791 comunidades; partição salva.
[COMUNIDADES FONTES] CNN Brasil leiden_seed_42 — 801 comunidades; partição salva.
[COMUNIDADES FONTES] CNN Brasil leiden_seed_43 — 805 comunidades; partição salva.
[COMUNIDADES FONTES] CNN Brasil leiden_seed_44 — 802 comunidades; partição salva.
[COMUNIDADES FONTES] PROJEÇÃO LIBERADA — CNN Brasil
[COMUNIDADES FONTES] Folha de S.Paulo louvain — 1673 comunidades; partição salva.
[COMUNIDADES FONTES] Folha de S.Paulo leiden_seed_42 — 1689 comunidades; partição salva.
[COMUNIDADES FONTES] Folha de S.Paulo leiden_seed_43 — 1689 comunidades; partição salva.
[COMUNIDADES FONTES] Folha de S.Paulo leiden_seed_44 — 1687 comunidades; partição salva.
[COMUNIDADES FONTES] PROJEÇÃO LIBERADA — Folha de S.Paulo
[COMUNIDADES FONTES] G1 louvain — 1081 comunidades; partição salva.
[COMUNIDADES FONTES] G1 leiden_seed_42 — 1095 comunidades; partição salva.
[COMUNIDADES FONTES] G1 leiden_seed_43 — 1100 comunidades; partição salva.
[C

{'networks': [{'source_key': 'cnn',
   'source': 'CNN Brasil',
   'nodes': 6517,
   'undirected_pairs': 8203,
   'component_partition_modularity': 0.17685738596007114,
   'partitions': [{'source_key': 'cnn',
     'source': 'CNN Brasil',
     'run_id': 'louvain',
     'nodes': 6517,
     'communities': 791,
     'weighted_modularity': 0.823537989079954,
     'largest_community_nodes': 622,
     'singleton_communities': 0,
     'disconnected_communities': 1,
     'communities_spanning_original_components': 0,
     'total_unmirrored_weight': 27163.0,
     'gcc_communities': 60,
     'algorithm_seconds': 1.0112584000016795},
    {'source_key': 'cnn',
     'source': 'CNN Brasil',
     'run_id': 'leiden_seed_42',
     'nodes': 6517,
     'communities': 801,
     'weighted_modularity': 0.8240064827308144,
     'largest_community_nodes': 734,
     'singleton_communities': 0,
     'disconnected_communities': 0,
     'communities_spanning_original_components': 0,
     'total_unmirrored_weight': 

## 41. Visualização e suporte documental das comunidades por fonte

Painéis interativos permitem explorar todas as comunidades de Louvain e Leiden de referência de cada fonte, consultar exemplos e listas completas de notícias e exportar essas listas. As vistas de grafos usam a projeção não dirigida ponderada, com inspeção dirigida opcional das relações originais. São desenhadas vistas da rede de cada fonte, de suas maiores comunidades e de todas as comunidades desconectadas das duas partições de referência. A conectividade dessas comunidades é conferida no subgrafo completo e seus membros são exportados por parte conexa. As visualizações não alteram as partições nem atribuem automaticamente temas aos grupos.

In [71]:
from pathlib import Path
from fontes_visualizacao import gerar_visualizacoes_fontes

PROJETO_VISTAS_FONTES = Path.cwd()
EXECUTAR_VISTAS_FONTES = True
COMUNIDADES_POR_PARTICAO_FONTES = 2
MAX_NOS_VISTA_FONTES = 100
MAX_PARES_VISTA_FONTES = 250
MAX_NOS_INSPECAO_FONTES = 200
ASSINATURA_REDE_VISTAS_FONTES = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)
ASSINATURA_FONTES_VISTAS = (
    'e5c458298e07becc955509750fdd3b43be2e72b3df621c782cb1d330205080d6'
)

In [72]:
resumo_vistas_fontes = gerar_visualizacoes_fontes(
    PROJETO_VISTAS_FONTES,
    executar=EXECUTAR_VISTAS_FONTES,
    comunidades_por_particao=COMUNIDADES_POR_PARTICAO_FONTES,
    max_nos=MAX_NOS_VISTA_FONTES,
    max_arestas=MAX_PARES_VISTA_FONTES,
    max_nos_inspecao=MAX_NOS_INSPECAO_FONTES,
    assinatura_rede_esperada=ASSINATURA_REDE_VISTAS_FONTES,
    assinatura_fontes_esperada=ASSINATURA_FONTES_VISTAS,
)
resumo_vistas_fontes

[VISTAS FONTES] cnn_grafo_louvain_rede — 100 nós; 250 pares.
[VISTAS FONTES] cnn_grafo_louvain_C85 — 100 nós; 168 pares.
[VISTAS FONTES] cnn_grafo_louvain_C19 — 100 nós; 186 pares.
[VISTAS FONTES] cnn_grafo_louvain_C30338 — 19 nós; 18 pares.
[VISTAS FONTES] cnn_grafo_leiden_seed_42_rede — 100 nós; 250 pares.
[VISTAS FONTES] cnn_grafo_leiden_seed_42_C85 — 100 nós; 189 pares.
[VISTAS FONTES] cnn_grafo_leiden_seed_42_C19 — 100 nós; 183 pares.
[VISTAS FONTES] folha_grafo_louvain_rede — 100 nós; 250 pares.
[VISTAS FONTES] folha_grafo_louvain_C2 — 100 nós; 245 pares.
[VISTAS FONTES] folha_grafo_louvain_C89 — 100 nós; 235 pares.
[VISTAS FONTES] folha_grafo_louvain_C2416 — 139 nós; 165 pares.
[VISTAS FONTES] folha_grafo_louvain_C24388 — 14 nós; 11 pares.
[VISTAS FONTES] folha_grafo_louvain_C20331 — 11 nós; 10 pares.
[VISTAS FONTES] folha_grafo_louvain_C26902 — 9 nós; 8 pares.
[VISTAS FONTES] folha_grafo_leiden_seed_42_rede — 100 nós; 250 pares.
[VISTAS FONTES] folha_grafo_leiden_seed_42_C2 — 1

{'sources': [{'source_key': 'cnn',
   'source': 'CNN Brasil',
   'community_rows': 1592,
   'support_news': 1750,
   'community_news_links': 9377,
   'graph_views': 7,
   'flagged_communities': 1,
   'index_html': 'cnn_indice_comunidades.html'},
  {'source_key': 'folha',
   'source': 'Folha de S.Paulo',
   'community_rows': 3362,
   'support_news': 2887,
   'community_news_links': 20839,
   'graph_views': 12,
   'flagged_communities': 6,
   'index_html': 'folha_indice_comunidades.html'},
  {'source_key': 'g1',
   'source': 'G1',
   'community_rows': 2176,
   'support_news': 1874,
   'community_news_links': 12290,
   'graph_views': 8,
   'flagged_communities': 2,
   'index_html': 'g1_indice_comunidades.html'}],
 'partitions': ['louvain', 'leiden_seed_42'],
 'graph_views': 27,
 'html_files': 31,
 'graph_figure_files': 81,
 'flagged_communities': 9,
 'complete_news_lists': True,
 'external_assets_required': False,
 'api_calls': 0,
 'neo4j_connections': 0,
 'database_writes': 0,
 'partitio

## 42. Cobertura e planejamento das redes temporais acumuladas

A análise temporal da Folha de S.Paulo utiliza 32 redes acumuladas anuais,
de 1994 a 2025. Cada corte incorpora todas as notícias e relações retidas
desde 1994 até o ano limite. Os 19 marcos de referência são preservados
como seleção da mesma série. A cobertura documental e de extração é
informada anualmente e no acumulado, sem hierarquizar os períodos.

In [79]:
from pathlib import Path
import importlib
import temporal_planejamento
importlib.reload(temporal_planejamento)

PROJETO_TEMPORAL = Path.cwd()
EXECUTAR_PLANEJAMENTO_TEMPORAL = True
MARCOS_TEMPORAIS = list(range(1994, 2026))
ASSINATURA_REDE_TEMPORAL = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)

In [80]:
resumo_planejamento_temporal = temporal_planejamento.planejar_evolucao_temporal(
    PROJETO_TEMPORAL,
    executar=EXECUTAR_PLANEJAMENTO_TEMPORAL,
    marcos=MARCOS_TEMPORAIS,
    assinatura_rede_esperada=ASSINATURA_REDE_TEMPORAL,
)
resumo_planejamento_temporal

[TEMPORAL] {"source": "Folha de S.Paulo", "corpus_news_1994_2025": 10717, "support_news": 5250, "retained_occurrences": 103966, "snapshot_count": 32, "cutoff_years": [1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], "annual_alternative_count": 32, "years_without_completed_re": [2021], "api_calls": 0, "neo4j_connections": 0, "networks_constructed": false, "global_network_modified": false}
[TEMPORAL] Pacote: C:\Users\pedro\Documents\MO438\PROJECT\resultados\temporal\planejamento\Resumo_Planejamento_Temporal_4dcec7767acd_e9cadd.zip


{'source': 'Folha de S.Paulo',
 'corpus_news_1994_2025': 10717,
 'support_news': 5250,
 'retained_occurrences': 103966,
 'snapshot_count': 32,
 'cutoff_years': [1994,
  1995,
  1996,
  1997,
  1998,
  1999,
  2000,
  2001,
  2002,
  2003,
  2004,
  2005,
  2006,
  2007,
  2008,
  2009,
  2010,
  2011,
  2012,
  2013,
  2014,
  2015,
  2016,
  2017,
  2018,
  2019,
  2020,
  2021,
  2022,
  2023,
  2024,
  2025],
 'annual_alternative_count': 32,
 'years_without_completed_re': [2021],
 'api_calls': 0,
 'neo4j_connections': 0,
 'networks_constructed': False,
 'global_network_modified': False,
 'output_directory': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/temporal/planejamento/4dcec7767acdeedca3bf'),
 'package': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/temporal/planejamento/Resumo_Planejamento_Temporal_4dcec7767acd_e9cadd.zip'),
 'snapshot_sha256': '4dcec7767acdeedca3bfe1898eb5be82436ba52073ef70a3291efb1311f29844'}

## 43. Construção e validação das redes acumuladas anuais

São materializados os 32 cortes da Folha de S.Paulo, de 1994 a 2025,
preservando os identificadores das entidades e as relações semânticas
dirigidas e tipadas. Pesos, atributos e suporte documental são recalculados
em cada período. As transições verificam preservação de nós e arestas,
atributos canônicos e não redução dos pesos. Os 19 marcos de referência
são uma seleção da mesma série. As projeções persistidas serão utilizadas
nas análises estruturais posteriores.

In [83]:
from pathlib import Path
import importlib
import temporal_redes
importlib.reload(temporal_redes)

PROJETO_REDES_TEMPORAIS = Path.cwd()
EXECUTAR_REDES_TEMPORAIS = True
ASSINATURA_REDE_REDES_TEMPORAIS = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)

In [84]:
resumo_redes_temporais = temporal_redes.construir_redes_temporais(
    PROJETO_REDES_TEMPORAIS,
    executar=EXECUTAR_REDES_TEMPORAIS,
    assinatura_rede_esperada=ASSINATURA_REDE_REDES_TEMPORAIS,
)
resumo_redes_temporais

[REDES TEMPORAIS] 1994: 101 nós; 128 arestas tipadas; 200 ocorrências; corte salvo e conferido.
[REDES TEMPORAIS] 1995: 178 nós; 241 arestas tipadas; 340 ocorrências; corte salvo e conferido.
[REDES TEMPORAIS] 1996: 305 nós; 411 arestas tipadas; 527 ocorrências; corte salvo e conferido.
[REDES TEMPORAIS] 1997: 431 nós; 562 arestas tipadas; 688 ocorrências; corte salvo e conferido.
[REDES TEMPORAIS] 1998: 562 nós; 738 arestas tipadas; 983 ocorrências; corte salvo e conferido.
[REDES TEMPORAIS] 1999: 658 nós; 864 arestas tipadas; 1,213 ocorrências; corte salvo e conferido.
[REDES TEMPORAIS] 2000: 824 nós; 1,119 arestas tipadas; 2,260 ocorrências; corte salvo e conferido.
[REDES TEMPORAIS] 2001: 1,059 nós; 1,478 arestas tipadas; 2,801 ocorrências; corte salvo e conferido.
[REDES TEMPORAIS] 2002: 1,209 nós; 1,691 arestas tipadas; 3,104 ocorrências; corte salvo e conferido.
[REDES TEMPORAIS] 2003: 1,329 nós; 1,844 arestas tipadas; 3,356 ocorrências; corte salvo e conferido.
[REDES TEMPORAIS

{'source': 'Folha de S.Paulo',
 'snapshot_count': 32,
 'reference_snapshot_count': 19,
 'cutoff_years': [1994,
  1995,
  1996,
  1997,
  1998,
  1999,
  2000,
  2001,
  2002,
  2003,
  2004,
  2005,
  2006,
  2007,
  2008,
  2009,
  2010,
  2011,
  2012,
  2013,
  2014,
  2015,
  2016,
  2017,
  2018,
  2019,
  2020,
  2021,
  2022,
  2023,
  2024,
  2025],
 'last_snapshot': {'cutoff_year': 2025,
  'cumulative_start_year': 1994,
  'is_reference_cutoff': True,
  'source': 'Folha de S.Paulo',
  'selected_news': 10717,
  're_completed_strict': 5670,
  're_partial': 106,
  'support_news': 5250,
  'nodes': 27642,
  'typed_directed_edges': 52116,
  'distinct_directed_pairs': 50059,
  'undirected_pairs': 37099,
  'weight_sum': 103966,
  'self_loops': 0,
  'identity_edges': 0,
  'observed_min_date': '1994-02-02',
  'observed_max_date': '2025-12-31'},
 'all_transitions_verified': True,
 'all_cutoffs_match_independent_coverage': True,
 'api_calls': 0,
 'neo4j_connections': 0,
 'database_writes':

## 44. Evolução estrutural das redes acumuladas anuais

As 32 redes acumuladas da Folha de S.Paulo são caracterizadas na projeção
simples não dirigida e não ponderada. Calculam-se graus, componentes,
triângulos, densidade e clusterização, com conferência independente de
todos os nós. Os incrementos anuais são examinados junto do volume
documental e da cobertura de extração. Os 19 marcos de referência
permanecem disponíveis como seleção da mesma série.

In [88]:
from pathlib import Path
import importlib
import temporal_estrutural
importlib.reload(temporal_estrutural)

PROJETO_ESTRUTURA_TEMPORAL = Path.cwd()
EXECUTAR_ESTRUTURA_TEMPORAL_GDS = True

ASSINATURA_REDE_ESTRUTURA_TEMPORAL = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)
ASSINATURA_SERIE_ESTRUTURA_TEMPORAL = (
    'c0880039218fbbcc0175a7670472ccc760a9efff0e7d5f64208eaddbde977426'
)

In [89]:
resumo_estrutura_temporal = temporal_estrutural.analisar_estrutura_temporal(
    PROJETO_ESTRUTURA_TEMPORAL,
    executar_gds=EXECUTAR_ESTRUTURA_TEMPORAL_GDS,
    assinatura_rede_esperada=ASSINATURA_REDE_ESTRUTURA_TEMPORAL,
    assinatura_temporal_esperada=ASSINATURA_SERIE_ESTRUTURA_TEMPORAL,
)
resumo_estrutura_temporal

[ESTRUTURA TEMPORAL] 1994: projeção com leitura sequencial.
[ESTRUTURA] degree: 101 nós; 0.14s.
[ESTRUTURA] triangles: 101 nós; 0.14s.
[ESTRUTURA] components: 101 nós; 0.12s.
[ESTRUTURA TEMPORAL] PROJEÇÃO LIBERADA — newsai_temporal_6c5d2135a3_1994_a168b9ce
[ESTRUTURA TEMPORAL] 1994: 28 componentes; maior: 17.82%; resultado salvo e conferido.
[ESTRUTURA TEMPORAL] 1995: projeção com leitura sequencial.
[ESTRUTURA] degree: 178 nós; 0.02s.
[ESTRUTURA] triangles: 178 nós; 0.03s.
[ESTRUTURA] components: 178 nós; 0.02s.
[ESTRUTURA TEMPORAL] PROJEÇÃO LIBERADA — newsai_temporal_6c5d2135a3_1995_39bdd026
[ESTRUTURA TEMPORAL] 1995: 45 componentes; maior: 20.79%; resultado salvo e conferido.
[ESTRUTURA TEMPORAL] 1996: projeção com leitura sequencial.
[ESTRUTURA] degree: 305 nós; 0.02s.
[ESTRUTURA] triangles: 305 nós; 0.04s.
[ESTRUTURA] components: 305 nós; 0.04s.
[ESTRUTURA TEMPORAL] PROJEÇÃO LIBERADA — newsai_temporal_6c5d2135a3_1996_b17db3e8
[ESTRUTURA TEMPORAL] 1996: 58 componentes; maior: 44.26

{'snapshot_count': 32,
 'reference_snapshot_count': 19,
 'all_node_metrics_independently_verified': True,
 'figure_files': 15,
 'projections_released': True,
 'last_snapshot': {'cutoff_year': 2025,
  'is_reference_cutoff': True,
  'source': 'Folha de S.Paulo',
  'selected_news': 10717,
  're_completed_strict': 5670,
  're_partial': 106,
  'support_news': 5250,
  'nodes': 27642,
  'typed_directed_edges': 52116,
  'distinct_directed_pairs': 50059,
  'undirected_pairs': 37099,
  'weight_sum': 103966,
  'self_loops': 0,
  'identity_edges': 0,
  'observed_min_date': '1994-02-02',
  'observed_max_date': '2025-12-31',
  'simple_undirected_edges': 37099,
  'degree_sum': 74198,
  'mean_degree': 2.6842486071919542,
  'density': 9.711112503860042e-05,
  'min_degree': 1,
  'max_degree': 1339,
  'median_degree': 1.0,
  'isolated_nodes': 0,
  'degree_one_nodes': 16207,
  'degree_le_two_nodes': 22045,
  'nodes_degree_ge_two': 11435,
  'triangles': 9303,
  'connected_triples': 2757318,
  'mean_local_c

## 45. Formação e reforço de conexões

As transições anuais distinguem pares conectados pela primeira vez entre
duas entidades novas, entre uma nova e outra já observada e entre duas
entidades já observadas. O reforço de pares anteriores é contabilizado
separadamente. Os incrementos são confrontados com o suporte documental
de cada par. A classificação entre os 19 marcos de referência é
recalculada nas próprias fronteiras, sem somar classes anuais.

In [92]:
from pathlib import Path
import importlib
import temporal_conexoes
importlib.reload(temporal_conexoes)

PROJETO_CONEXOES = Path.cwd()
EXECUTAR_CONEXOES = True
ASSINATURA_REDE_CONEXOES = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)
ASSINATURA_TEMPORAL_CONEXOES = (
    'c0880039218fbbcc0175a7670472ccc760a9efff0e7d5f64208eaddbde977426'
)
ASSINATURA_ESTRUTURA_CONEXOES = (
    '6c5d2135a3033580cf5f0a06b6fa043ce8740a79d8f30434530f9ccfcfb873de'
)

In [93]:
resumo_conexoes = temporal_conexoes.analisar_formacao_conexoes(
    PROJETO_CONEXOES,
    executar=EXECUTAR_CONEXOES,
    assinatura_rede_esperada=ASSINATURA_REDE_CONEXOES,
    assinatura_temporal_esperada=ASSINATURA_TEMPORAL_CONEXOES,
    assinatura_estrutura_esperada=ASSINATURA_ESTRUTURA_CONEXOES,
)
resumo_conexoes

[CONEXÕES] 1994→1995: 72 pares novos; 0 pares anteriores reforçados; conferido.
[CONEXÕES] 1995→1996: 136 pares novos; 2 pares anteriores reforçados; conferido.
[CONEXÕES] 1996→1997: 111 pares novos; 4 pares anteriores reforçados; conferido.
[CONEXÕES] 1997→1998: 140 pares novos; 9 pares anteriores reforçados; conferido.
[CONEXÕES] 1998→1999: 94 pares novos; 3 pares anteriores reforçados; conferido.
[CONEXÕES] 1999→2000: 183 pares novos; 3 pares anteriores reforçados; conferido.
[CONEXÕES] 2000→2001: 253 pares novos; 13 pares anteriores reforçados; conferido.
[CONEXÕES] 2001→2002: 154 pares novos; 7 pares anteriores reforçados; conferido.
[CONEXÕES] 2002→2003: 113 pares novos; 11 pares anteriores reforçados; conferido.
[CONEXÕES] 2003→2004: 125 pares novos; 21 pares anteriores reforçados; conferido.
[CONEXÕES] 2004→2005: 127 pares novos; 10 pares anteriores reforçados; conferido.
[CONEXÕES] 2005→2006: 46 pares novos; 3 pares anteriores reforçados; conferido.
[CONEXÕES] 2006→2007: 196 p

{'annual_transitions': 31,
 'reference_transitions': 18,
 'baseline': {'year': 1994, 'nodes': 101, 'pairs': 85, 'occurrences': 200},
 'new_pairs_after_baseline': 37014,
 'new_pair_categories': {'NEW_NEW': 13700, 'OLD_NEW': 18611, 'OLD_OLD': 4703},
 'reinforced_pair_intervals': 3035,
 'occurrences_new_pairs': 80218,
 'occurrences_existing_pairs': 23548,
 'all_pair_increments_verified_against_news': True,
 'figure_files': 12,
 'api_calls': 0,
 'neo4j_connections': 0,
 'database_writes': 0,
 'network_modified': False,
 'cumulative_advantage_tested': False,
 'elapsed_seconds_this_run': 21.18388640000194,
 'snapshot_sha256': '7d088d358aed2958eb6f7ebe13170b6f04e10f59ae577612b876f32c6860aacf',
 'output_directory': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/temporal/conexoes/7d088d358aed2958eb6f'),
 'package': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/temporal/conexoes/Resumo_Formacao_Conexoes_7d088d358aed_815ee6.zip')}

## 46. Protagonismo e persistência temporal das entidades

Os graus já verificados dos 32 cortes são associados à força relacional,
à exposição documental e aos ganhos de vizinhos. Rankings preservam
empates; a persistência é apresentada tanto para um grupo de tamanho
fixo quanto incluindo empates na posição limite. Os casos configurados
permanecem separados por ID, com seus tipos canônicos e acesso às
notícias. Esta análise descritiva prepara a investigação posterior de
padrões compatíveis com vantagem cumulativa, sem testar causalidade.

In [96]:
from pathlib import Path
import importlib
import temporal_protagonismo
importlib.reload(temporal_protagonismo)

PROJETO_PROTAGONISMO_TEMPORAL = Path.cwd()
EXECUTAR_PROTAGONISMO_TEMPORAL = True
TOP_K_TEMPORAL = 20
TOP_POR_TIPO_TEMPORAL = 5
CASOS_TEMPORAIS = [5276, 32443]

ASSINATURA_REDE_PROTAGONISMO_TEMPORAL = (
    '3920a1985093835e545aa0f7e083570cc239ac4dfe3aac42493c397f7db9a80a'
)
ASSINATURA_SERIE_PROTAGONISMO_TEMPORAL = (
    'c0880039218fbbcc0175a7670472ccc760a9efff0e7d5f64208eaddbde977426'
)
ASSINATURA_ESTRUTURA_PROTAGONISMO_TEMPORAL = (
    '6c5d2135a3033580cf5f0a06b6fa043ce8740a79d8f30434530f9ccfcfb873de'
)
ASSINATURA_CONEXOES_PROTAGONISMO_TEMPORAL = (
    '7d088d358aed2958eb6f7ebe13170b6f04e10f59ae577612b876f32c6860aacf'
)

In [97]:
resumo_protagonismo_temporal = temporal_protagonismo.analisar_protagonismo_temporal(
    PROJETO_PROTAGONISMO_TEMPORAL,
    executar=EXECUTAR_PROTAGONISMO_TEMPORAL,
    top_k=TOP_K_TEMPORAL,
    top_por_tipo=TOP_POR_TIPO_TEMPORAL,
    casos=CASOS_TEMPORAIS,
    assinatura_rede_esperada=ASSINATURA_REDE_PROTAGONISMO_TEMPORAL,
    assinatura_temporal_esperada=ASSINATURA_SERIE_PROTAGONISMO_TEMPORAL,
    assinatura_estrutura_esperada=ASSINATURA_ESTRUTURA_PROTAGONISMO_TEMPORAL,
    assinatura_conexoes_esperada=ASSINATURA_CONEXOES_PROTAGONISMO_TEMPORAL,
)
resumo_protagonismo_temporal

[PROTAGONISMO TEMPORAL] 1994: 101 entidades; rankings e ganhos conferidos.
[PROTAGONISMO TEMPORAL] 1995: 178 entidades; rankings e ganhos conferidos.
[PROTAGONISMO TEMPORAL] 1996: 305 entidades; rankings e ganhos conferidos.
[PROTAGONISMO TEMPORAL] 1997: 431 entidades; rankings e ganhos conferidos.
[PROTAGONISMO TEMPORAL] 1998: 562 entidades; rankings e ganhos conferidos.
[PROTAGONISMO TEMPORAL] 1999: 658 entidades; rankings e ganhos conferidos.
[PROTAGONISMO TEMPORAL] 2000: 824 entidades; rankings e ganhos conferidos.
[PROTAGONISMO TEMPORAL] 2001: 1,059 entidades; rankings e ganhos conferidos.
[PROTAGONISMO TEMPORAL] 2002: 1,209 entidades; rankings e ganhos conferidos.
[PROTAGONISMO TEMPORAL] 2003: 1,329 entidades; rankings e ganhos conferidos.
[PROTAGONISMO TEMPORAL] 2004: 1,436 entidades; rankings e ganhos conferidos.
[PROTAGONISMO TEMPORAL] 2005: 1,549 entidades; rankings e ganhos conferidos.
[PROTAGONISMO TEMPORAL] 2006: 1,600 entidades; rankings e ganhos conferidos.
[PROTAGONISMO

C:\Users\pedro\Documents\MO438\PROJECT\temporal_protagonismo.py:148: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  if campo=='new_neighbors':v=v.where(~d.is_baseline_stock.fillna(False).astype(bool))


[PROTAGONISMO TEMPORAL] {"snapshot_count": 32, "top_k": 20, "selected_entities": 16, "configured_case_ids": [5276, 32443], "entity_cutoff_rows": 160901, "unique_entities": 27642, "all_degrees_match_saved_structure": true, "all_neighbor_gains_match_pair_events": true, "rank_ties_preserved": true, "figure_files": 12, "pagerank_computed": false, "cumulative_advantage_tested": false, "api_calls": 0, "neo4j_connections": 0, "database_writes": 0, "entity_groups_merged": 0, "canonical_types_changed": 0, "network_modified": false, "elapsed_seconds_this_run": 26.30839669999841, "snapshot_sha256": "eabe8076fc0c5c7cb10bb5e0c50fa90e5478e74a3522fe8d1e89ba30f7c24491"}
[PROTAGONISMO TEMPORAL] Pacote: C:\Users\pedro\Documents\MO438\PROJECT\resultados\temporal\protagonismo\Resumo_Protagonismo_Temporal_eabe8076fc0c_304a8e.zip


{'snapshot_count': 32,
 'top_k': 20,
 'selected_entities': 16,
 'configured_case_ids': [5276, 32443],
 'entity_cutoff_rows': 160901,
 'unique_entities': 27642,
 'all_degrees_match_saved_structure': True,
 'all_neighbor_gains_match_pair_events': True,
 'rank_ties_preserved': True,
 'figure_files': 12,
 'pagerank_computed': False,
 'cumulative_advantage_tested': False,
 'api_calls': 0,
 'neo4j_connections': 0,
 'database_writes': 0,
 'entity_groups_merged': 0,
 'canonical_types_changed': 0,
 'network_modified': False,
 'elapsed_seconds_this_run': 26.30839669999841,
 'snapshot_sha256': 'eabe8076fc0c5c7cb10bb5e0c50fa90e5478e74a3522fe8d1e89ba30f7c24491',
 'output_directory': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/temporal/protagonismo/eabe8076fc0c5c7cb10b'),
 'package': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/temporal/protagonismo/Resumo_Protagonismo_Temporal_eabe8076fc0c_304a8e.zip')}

## 47. Cauda da distribuição de graus e concentração complementar

Os graus dos 32 cortes são examinados por CCDF e ajuste discreto de
lei de potência, com xmin selecionado pela distância KS. A cauda é
comparada a lognormal, exponencial e lei de potência com corte exponencial.
No acumulado de 2025 realiza-se um diagnóstico semiparamétrico por
simulação, com reestimação dos parâmetros e progresso persistido.
Gini e participações superiores complementam a caracterização.
Os diagnósticos iid não demonstram que a rede seja livre de escala
nem identificam um mecanismo de vantagem cumulativa.

In [98]:
%pip install powerlaw==1.5


  Attempting uninstall: powerlaw
    Found existing installation: powerlaw 2.0.0
    Uninstalling powerlaw-2.0.0:
      Successfully uninstalled powerlaw-2.0.0


In [101]:
from pathlib import Path
import importlib
import temporal_caudas
importlib.reload(temporal_caudas)

PROJETO_CAUDAS = Path.cwd()
EXECUTAR_CAUDAS = True
REPETICOES_BOOTSTRAP_CAUDAS = 199
SEMENTE_CAUDAS = 42
ASSINATURA_PROTAGONISMO_CAUDAS = (
    'eabe8076fc0c5c7cb10bb5e0c50fa90e5478e74a3522fe8d1e89ba30f7c24491'
)

In [102]:
resumo_caudas = temporal_caudas.analisar_caudas_temporais(
    PROJETO_CAUDAS,
    executar=EXECUTAR_CAUDAS,
    repeticoes_bootstrap=REPETICOES_BOOTSTRAP_CAUDAS,
    seed=SEMENTE_CAUDAS,
    assinatura_protagonismo_esperada=ASSINATURA_PROTAGONISMO_CAUDAS,
)
resumo_caudas

[CAUDAS] 1994: xmin=2; α=3.207; 40 nós na cauda; ajuste salvo em preparação.
[CAUDAS] 1995: xmin=4; α=4.005; 16 nós na cauda; ajuste salvo em preparação.
[CAUDAS] 1996: xmin=3; α=3.214; 61 nós na cauda; ajuste salvo em preparação.
[CAUDAS] 1997: xmin=3; α=3.053; 76 nós na cauda; ajuste salvo em preparação.
[CAUDAS] 1998: xmin=2; α=2.848; 235 nós na cauda; ajuste salvo em preparação.
[CAUDAS] 1999: xmin=2; α=2.807; 267 nós na cauda; ajuste salvo em preparação.
[CAUDAS] 2000: xmin=2; α=2.732; 325 nós na cauda; ajuste salvo em preparação.
[CAUDAS] 2001: xmin=2; α=2.718; 424 nós na cauda; ajuste salvo em preparação.
[CAUDAS] 2002: xmin=2; α=2.730; 486 nós na cauda; ajuste salvo em preparação.
[CAUDAS] 2003: xmin=2; α=2.737; 525 nós na cauda; ajuste salvo em preparação.
[CAUDAS] 2004: xmin=2; α=2.741; 574 nós na cauda; ajuste salvo em preparação.
[CAUDAS] 2005: xmin=2; α=2.726; 617 nós na cauda; ajuste salvo em preparação.
[CAUDAS] 2006: xmin=2; α=2.731; 630 nós na cauda; ajuste salvo em pr

{'snapshot_count': 32,
 'comparisons': 96,
 'comparisons_requiring_review': 1,
 'bootstrap_2025': {'repetitions_requested': 199,
  'valid_repetitions': 199,
  'status': 'COMPLETE',
  'observed_ks': 0.004921240721109554,
  'p_iid_semiparametric': np.float64(0.2),
  'monte_carlo_standard_error': 0.028355248200333437,
  'plus_one_correction': True,
  'seed': 42,
  'network_dependence_accounted_for': False},
 'figure_files': 15,
 'gini_role': 'complementary',
 'scale_free_network_concluded': False,
 'cumulative_advantage_tested': False,
 'api_calls': 0,
 'neo4j_connections': 0,
 'network_modified': False,
 'output_directory': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/temporal/caudas/0d063aa9743e774fa189'),
 'package': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/temporal/caudas/Resumo_Caudas_Temporais_0d063aa9743e_50e8e7.zip'),
 'snapshot_sha256': '0d063aa9743e774fa189e4a4c48413cb5fc35875af6dc92f41fe6ef72d3d3930'}

## 48. Grau anterior e aquisição de novos vizinhos

Investigamos se entidades já presentes com maior grau em um corte acumulado
adquirem mais vizinhos no ano seguinte. A análise considera 31 transições
anuais da Folha de S.Paulo, de 1994→1995 até 2024→2025. O estoque de 1994
não é contado como ganho; entrantes são auditados separadamente. Todos os
nós anteriores são incluídos, mesmo com ganho zero.

As associações são descritivas, com empates preservados no cálculo dos
postos. A análise distingue novos vizinhos de reforço de pares existentes,
compara faixas de grau e contextualiza tipo de entidade, idade na rede,
exposição documental e cobertura de extração. As referências uniforme e
proporcional ao grau são alocações algébricas, sem simulação de redes.
Os estudos dos casos mantêm IDs e tipos canônicos e permitem consultar as
notícias de origem. Padrões positivos são compatíveis com vantagem
cumulativa observada, sem demonstrar causalidade ou Efeito Mateus.

In [3]:
from pathlib import Path
import importlib
import temporal_vantagem
importlib.reload(temporal_vantagem)

PROJETO_VANTAGEM = Path.cwd()
EXECUTAR_VANTAGEM = True
ASSINATURA_PROTAGONISMO_VANTAGEM = (
    'eabe8076fc0c5c7cb10bb5e0c50fa90e5478e74a3522fe8d1e89ba30f7c24491'
)

In [4]:
resumo_vantagem = temporal_vantagem.analisar_vantagem_cumulativa(
    PROJETO_VANTAGEM,
    executar=EXECUTAR_VANTAGEM,
    assinatura_protagonismo_esperada=ASSINATURA_PROTAGONISMO_VANTAGEM,
)
resumo_vantagem

[VANTAGEM] 1994→1995: 101 entidades anteriores; 27 ganhos; zeros preservados.
[VANTAGEM] 1995→1996: 178 entidades anteriores; 59 ganhos; zeros preservados.
[VANTAGEM] 1996→1997: 305 entidades anteriores; 40 ganhos; zeros preservados.
[VANTAGEM] 1997→1998: 431 entidades anteriores; 61 ganhos; zeros preservados.
[VANTAGEM] 1998→1999: 562 entidades anteriores; 36 ganhos; zeros preservados.
[VANTAGEM] 1999→2000: 658 entidades anteriores; 77 ganhos; zeros preservados.
[VANTAGEM] 2000→2001: 824 entidades anteriores; 143 ganhos; zeros preservados.
[VANTAGEM] 2001→2002: 1,059 entidades anteriores; 77 ganhos; zeros preservados.
[VANTAGEM] 2002→2003: 1,209 entidades anteriores; 59 ganhos; zeros preservados.
[VANTAGEM] 2003→2004: 1,329 entidades anteriores; 70 ganhos; zeros preservados.
[VANTAGEM] 2004→2005: 1,436 entidades anteriores; 87 ganhos; zeros preservados.
[VANTAGEM] 2005→2006: 1,549 entidades anteriores; 31 ganhos; zeros preservados.
[VANTAGEM] 2006→2007: 1,600 entidades anteriores; 127

{'annual_transitions': 31,
 'entity_transition_rows': 133259,
 'zero_gain_rows': 126880,
 'association_rows': 1488,
 'stratum_rows': 840,
 'configured_case_ids': [5276, 32443],
 'figure_files': 12,
 'all_degree_gains_verified': True,
 'all_previous_nodes_included': True,
 'year_without_completed_re': [2021],
 'association_analysis_completed': True,
 'causality_identified': False,
 'preferential_attachment_exponent_fitted': False,
 'iid_p_values_computed': False,
 'api_calls': 0,
 'neo4j_connections': 0,
 'database_writes': 0,
 'entity_groups_merged': 0,
 'network_modified': False,
 'elapsed_seconds_this_run': 19.378737699997146,
 'output_directory': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/temporal/vantagem/0a8e038c994ad2107359'),
 'package': WindowsPath('C:/Users/pedro/Documents/MO438/PROJECT/resultados/temporal/vantagem/Resumo_Vantagem_Cumulativa_0a8e038c994a_2fb4b6.zip'),
 'snapshot_sha256': '0a8e038c994ad210735988dfb397e38845d6efb66b1b04f025f78ec04c84735f'}

## 49. Exploração visual das redes acumuladas

O painel explora os 32 cortes anuais acumulados da Folha de S.Paulo,
de 1994 a 2025. Os destaques são selecionados automaticamente por grau
na rede completa do ano escolhido, incluindo empates na 15ª posição.
O grau conta vizinhos distintos; o critério e os valores aparecem na tela.

A busca cobre todas as entidades da série por nome ou ID. A entidade
selecionada abre uma vista de sua vizinhança; o texto e o ID são mantidos
ao trocar de ano, com indicação de ausência antes de sua entrada na rede.
Borda vermelha e halo identificam a busca. As cores de preenchimento representam
tipos; tamanho representa grau em escala logarítmica, sem proporção direta.
Os nomes estão visíveis e a física ativada por padrão, com opções de
controle. Nenhuma interação modifica a rede ou seus indicadores.

A cobertura das extrações e as notícias de suporte são acessíveis para
contextualizar a leitura. Em 2021 não há RE concluída nesta edição.
Ausência de crescimento observado não significa ausência de relações
no noticiário. Os subgrafos exibidos são seleções visuais limitadas.

In [15]:
from pathlib import Path
import importlib
import temporal_visualizacoes
importlib.reload(temporal_visualizacoes)

PROJETO_VISTAS_TEMPORAIS = Path.cwd()
EXECUTAR_VISTAS_TEMPORAIS = True
MAX_NOS_VISTA_TEMPORAL = 100
MAX_PARES_VISTA_TEMPORAL = 250
ASSINATURA_SERIE_VISTAS = (
    'c0880039218fbbcc0175a7670472ccc760a9efff0e7d5f64208eaddbde977426'
)
ASSINATURA_PROTAGONISMO_VISTAS = (
    'eabe8076fc0c5c7cb10bb5e0c50fa90e5478e74a3522fe8d1e89ba30f7c24491'
)

In [16]:
resumo_vistas_temporais = temporal_visualizacoes.gerar_visualizacoes_temporais(
    PROJETO_VISTAS_TEMPORAIS,
    executar=EXECUTAR_VISTAS_TEMPORAIS,
    max_nos=MAX_NOS_VISTA_TEMPORAL,
    max_pares=MAX_PARES_VISTA_TEMPORAL,
    assinatura_temporal_esperada=ASSINATURA_SERIE_VISTAS,
    assinatura_protagonismo_esperada=ASSINATURA_PROTAGONISMO_VISTAS,
)
resumo_vistas_temporais

[VISTAS TEMPORAIS] 1994: 101 nós pesquisáveis; 16 destaques automáticos; dados conferidos.
[VISTAS TEMPORAIS] 1995: 178 nós pesquisáveis; 16 destaques automáticos; dados conferidos.
[VISTAS TEMPORAIS] 1996: 305 nós pesquisáveis; 19 destaques automáticos; dados conferidos.
[VISTAS TEMPORAIS] 1997: 431 nós pesquisáveis; 16 destaques automáticos; dados conferidos.
[VISTAS TEMPORAIS] 1998: 562 nós pesquisáveis; 25 destaques automáticos; dados conferidos.
[VISTAS TEMPORAIS] 1999: 658 nós pesquisáveis; 21 destaques automáticos; dados conferidos.
[VISTAS TEMPORAIS] 2000: 824 nós pesquisáveis; 18 destaques automáticos; dados conferidos.
[VISTAS TEMPORAIS] 2001: 1,059 nós pesquisáveis; 18 destaques automáticos; dados conferidos.
[VISTAS TEMPORAIS] 2002: 1,209 nós pesquisáveis; 16 destaques automáticos; dados conferidos.
[VISTAS TEMPORAIS] 2003: 1,329 nós pesquisáveis; 17 destaques automáticos; dados conferidos.
[VISTAS TEMPORAIS] 2004: 1,436 nós pesquisáveis; 15 destaques automáticos; dados con

{'snapshot_count': 32,
 'searchable_entities': 27642,
 'automatic_highlights_minimum': 15,
 'include_boundary_ties': True,
 'annual_highlight_counts': {'1994': 16,
  '1995': 16,
  '1996': 19,
  '1997': 16,
  '1998': 25,
  '1999': 21,
  '2000': 18,
  '2001': 18,
  '2002': 16,
  '2003': 17,
  '2004': 15,
  '2005': 15,
  '2006': 15,
  '2007': 15,
  '2008': 16,
  '2009': 15,
  '2010': 15,
  '2011': 17,
  '2012': 20,
  '2013': 15,
  '2014': 16,
  '2015': 15,
  '2016': 15,
  '2017': 16,
  '2018': 15,
  '2019': 15,
  '2020': 15,
  '2021': 15,
  '2022': 15,
  '2023': 15,
  '2024': 15,
  '2025': 15},
 'manual_case_ids': [],
 'html_files': 1,
 'figure_files': 33,
 'support_news': 5250,
 'physics_initially_enabled': True,
 'all_labels_initially_visible': True,
 'persistent_search': True,
 'all_pair_weights_and_document_counts_verified': True,
 'complete_evidence_lists': True,
 'external_assets_required': False,
 'api_calls': 0,
 'neo4j_connections': 0,
 'network_modified': False,
 'entity_groups_

## 50. Revisão das partições, do ajuste de 2012 e das visualizações previstas

A revisão usa apenas os resultados persistidos. Reconfere a conectividade
interna das partições por referências independentes, compara os IDs com
os checkpoints da detecção e reproduz numericamente o ajuste lognormal
salvo para 2012. A causa de eventuais desconexões do GDS não é atribuída
sem reprodução no ambiente local.

Também gera o subgrafo induzido das 50 entidades por força global e
visualizações da maior componente completa, com as partições originais.
Os rótulos de relação e as identidades são preservados. O grupo CNN e
a rede conjunta de 2025 não são revisados nesta etapa.

In [23]:
from pathlib import Path
import importlib
import revisao_pendencias
importlib.reload(revisao_pendencias)

PROJETO_REVISAO = Path.cwd()
EXECUTAR_REVISAO = True

In [24]:
resumo_revisao = revisao_pendencias.executar_revisao(
    PROJETO_REVISAO, executar=EXECUTAR_REVISAO
)
resumo_revisao

[REVISÃO] GLOBAL / leiden_seed_42: 0 comunidades desconexas; checkpoint e duas referências conferidos.
[REVISÃO] GLOBAL / leiden_seed_43: 0 comunidades desconexas; checkpoint e duas referências conferidos.
[REVISÃO] GLOBAL / leiden_seed_44: 1 comunidades desconexas; checkpoint e duas referências conferidos.
[REVISÃO] GLOBAL / louvain: 12 comunidades desconexas; checkpoint e duas referências conferidos.
[REVISÃO] cnn / leiden_seed_42: 0 comunidades desconexas; checkpoint e duas referências conferidos.
[REVISÃO] cnn / leiden_seed_43: 0 comunidades desconexas; checkpoint e duas referências conferidos.
[REVISÃO] cnn / leiden_seed_44: 0 comunidades desconexas; checkpoint e duas referências conferidos.
[REVISÃO] cnn / louvain: 1 comunidades desconexas; checkpoint e duas referências conferidos.
[REVISÃO] folha / leiden_seed_42: 2 comunidades desconexas; checkpoint e duas referências conferidos.
[REVISÃO] folha / leiden_seed_43: 1 comunidades desconexas; checkpoint e duas referências conferido

{'partition_runs': 16,
 'all_saved_connectivity_counts_reproduced': True,
 'all_checkpoints_match': True,
 'disconnected_leiden_community_occurrences': 7,
 'leiden_root_cause': 'NOT_ESTABLISHED_OFFLINE',
 'partitions_modified': False,
 'lognormal_2012': {'year': 2012,
  'xmin': 3.0,
  'tail_nodes': 464,
  'mu_saved': -804.441490016917,
  'sigma_saved': 21.48115390809859,
  'saved_status': 'REVIEW_NUMERICS',
  'saved_warning': 'invalid value encountered in divide',
  'saved_R': -0.0444667924871629,
  'stable_R_same_parameters': -0.04446679248269714,
  'absolute_difference_R': 4.46576109425223e-12,
  'saved_p_iid_diagnostic': 0.8092803047349488,
  'all_final_log_probabilities_finite': True,
  'parameters_reestimated': False,
  'old_output_modified': False,
  'verdict': 'FINAL_POINT_REPRODUCED_PARAMETER_STABILITY_NOT_ESTABLISHED',
  'interpretation': 'O valor final foi reproduzido de forma estável; isso não demonstra um máximo estável ou parâmetros identificáveis. Sem preferência conclusi

## 50.1. Reprodução local do Leiden global no GDS

A revisão já confirmou que a desconexão está na partição salva. Para investigar a causa no seu Neo4j, execute o bloco abaixo. Ele não seleciona automaticamente uma nova partição nem modifica os resultados oficiais. Requer a projeção `community` global já disponível no catálogo GDS; sua topologia e seus pesos são conferidos integralmente antes das execuções.

In [27]:
import diagnostico_leiden_gds
importlib.reload(diagnostico_leiden_gds)

EXECUTAR_DIAGNOSTICO_LEIDEN_GDS = True
resumo_diagnostico_leiden = diagnostico_leiden_gds.diagnosticar_global(
    PROJETO_REVISAO,
    executar_gds=EXECUTAR_DIAGNOSTICO_LEIDEN_GDS,
)
resumo_diagnostico_leiden

[DIAGNÓSTICO GDS] Etapa: VALIDAR_ARQUIVOS_LOCAIS
[DIAGNÓSTICO GDS] Etapa: CONFIGURAR_NEO4J
[DIAGNÓSTICO GDS] Etapa: CONECTAR_NEO4J
[DIAGNÓSTICO GDS] Etapa: INSPECIONAR_CATALOGO_GDS
[DIAGNÓSTICO GDS] Etapa: CRIAR_PROJECAO_TEMPORARIA
[DIAGNÓSTICO GDS] Projeção original ausente; criada somente a projeção temporária deste diagnóstico.
[DIAGNÓSTICO GDS] Etapa: VALIDAR_PROJECAO_COMPLETA
[DIAGNÓSTICO GDS] Etapa: EXECUTAR_seed44_original
[DIAGNÓSTICO GDS] Etapa: AVALIAR_seed44_original
[DIAGNÓSTICO GDS] seed44_original comunidades desconexas: 1
[DIAGNÓSTICO GDS] Etapa: EXECUTAR_seed44_sem_consecutiveIds
[DIAGNÓSTICO GDS] Etapa: AVALIAR_seed44_sem_consecutiveIds
[DIAGNÓSTICO GDS] seed44_sem_consecutiveIds comunidades desconexas: 1
[DIAGNÓSTICO GDS] Etapa: EXECUTAR_seed44_mais_niveis
[DIAGNÓSTICO GDS] Etapa: AVALIAR_seed44_mais_niveis
[DIAGNÓSTICO GDS] seed44_mais_niveis comunidades desconexas: 1
[DIAGNÓSTICO GDS] Etapa: EXECUTAR_seed44_tolerancia_menor
[DIAGNÓSTICO GDS] Etapa: AVALIAR_seed44_to

{'status': 'COMPLETE',
 'package': 'C:\\Users\\pedro\\Documents\\MO438\\PROJECT\\resultados\\revisao_pendencias\\diagnostico_gds\\Resumo_Diagnostico_Leiden_GDS.zip',
 'runs': 4,
 'database_writes': 0,
 'partitions_modified': False,
 'gds_graphs_created': 1,
 'gds_graphs_removed': 1}